# Python for Data Engineering

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-interview-questions/blob/master/notebooks/topics/python.ipynb)

*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

Python is the default glue language of data engineering: orchestrators are written in it, engines expose a first-class API, and most pipeline code is Python even when the compute runs in Spark or a warehouse. The interview bar is not algorithmic trivia; it is whether you can write pipeline code that is memory-safe, idempotent, observable, testable, and reproducibly packaged.

Three themes run through this page: **scale** (processing more than memory allows), **correctness** (idempotent writes, retries, dead-letter handling), and **operability** (structured logs, pinned environments, CI tests).

## Why is Python widely used in data engineering?

Python is popular because it is productive, readable, and has a strong ecosystem for data processing, orchestration, and integration. It is commonly used for glue code, transformations, APIs, and automation, even when the heavy compute runs on distributed engines.

What that looks like in a real stack: **orchestration** (Airflow, Dagster), **transformation** (PySpark, Polars, pandas, SQLAlchemy), **ingestion** (`httpx`, `boto3`, Kafka clients), and **testing/CI** (pytest, Great Expectations). The trade-off to name: Python is slow per CPU-cycle and heavy per process, so the winning pattern is "Python for control flow and small data, native or distributed engines for heavy lifting."

In [ ]:
# Python usually plays the control-plane role: orchestrate, do not crunch
from airflow.sdk import dag, task


@dag(dag_id="orders_elt", schedule="0 3 * * *", start_date=None, tags=["elt"])
def orders_elt():
    @task
    def extract() -> str:
        return extract_to_s3("orders", "2026-10-04")

    @task
    def transform(path: str) -> int:
        return run_spark_job("jobs/transform_orders.py", path)

    transform(extract())


orders_elt()

**Mechanism.** Python is the control plane, not the compute engine. The interpreter decides *what* to run and calls into C, Java, or Rust code for the heavy parts, so a short Python file can drive a cluster-wide job.

**Worked example.** A daily job is 200 lines of Python: list the partitions, submit a Spark job per partition, poll a control table, write a status row. The row math runs inside Spark; Python only moves names and paths.

**Common mistake.** Rewriting the transform as a Python loop over a million rows. It runs, but it is far slower and pulls the data into memory. Push work down to SQL or Spark and keep Python for control flow.

## How do iterators and generators help with large data processing?

Iterators and generators allow streaming data instead of loading everything into memory. They enable processing large datasets row-by-row or batch-by-batch. This reduces memory usage and often improves stability of ETL jobs.

Gotchas: a generator is **single-use**; `yield` defers side effects to consumption; `itertools.islice`/`batched` compose well; and batching avoids per-row overhead.

In [ ]:
# Streaming pipeline: parse -> batch -> write with memory bounded by batch size
import csv
import itertools
from collections.abc import Iterable, Iterator
from typing import Any


def read_rows(path: str) -> Iterator[dict[str, str]]:
    with open(path, newline="", encoding="utf-8") as handle:
        yield from csv.DictReader(handle)


def batched(items: Iterable[Any], size: int) -> Iterator[list[Any]]:
    iterator = iter(items)
    while chunk := list(itertools.islice(iterator, size)):
        yield chunk


def run(path: str, batch_size: int = 10_000) -> int:
    return sum(write_batch(batch) for batch in batched(read_rows(path), batch_size))

Each stage is independently testable with a list, and peak memory tracks `batch_size` rather than the file.

**Mechanism.** The choice follows the bottleneck. A thread waits on I/O while the GIL is released, a process has its own interpreter and its own GIL, and async I/O multiplexes many waits on one thread with an event loop.

**Worked example.** 100 HTTP fetches at 50 ms each take about 5 s one at a time. A 100-worker thread pool overlaps the waits and finishes in roughly 50 ms. The same 100 CPU tasks would not speed up, because the GIL serialises bytecode.

**Common mistake.** Reaching for threads to fix a slow numeric loop, or starting one process per row. Match the pool to the work, cap it to the machine, and measure before you tune.

## What is the difference between threads, multiprocessing, and async IO in Python?

Threads are useful for IO-bound workloads (network, disk) but limited by the GIL for CPU-bound tasks. Multiprocessing runs multiple processes and can use multiple CPU cores for CPU-bound work. Async IO uses an event loop to handle many concurrent IO operations efficiently when tasks spend time waiting.

In [ ]:
# Async IO: thousands of concurrent requests without thousands of threads
import asyncio

import httpx


async def fetch(client: httpx.AsyncClient, url: str) -> dict:
    response = await client.get(url, timeout=30)
    response.raise_for_status()
    return response.json()


async def fetch_many(urls: list[str], concurrency: int = 100) -> list[dict]:
    limits = httpx.Limits(max_connections=concurrency)
    async with httpx.AsyncClient(limits=limits) as client:
        return await asyncio.gather(*(fetch(client, url) for url in urls))

For CPU-bound work swap in `ProcessPoolExecutor`; for blocking IO that is not async-friendly, a `ThreadPoolExecutor` with bounded workers is often the simplest correct choice. Measure whether the workload is IO-bound or CPU-bound first.

## What is the GIL and why does it matter?

The Global Interpreter Lock (GIL) allows only one thread to execute Python bytecode at a time in CPython. This limits CPU-bound parallelism with threads. Many data engineering tasks are IO-bound, so threads can still help, but CPU-bound tasks often require multiprocessing or native libraries.

Recent nuance: CPython 3.13 introduced an **experimental free-threaded build** (PEP 703) where the GIL can be disabled, though it is not the default; the GIL is **released** around blocking IO and inside many C extensions (NumPy, Arrow, compression); and it is a CPython implementation detail, not a language rule.

In [ ]:
# If 4 threads take ~4x one thread, the GIL is serializing pure-Python CPU work
import time
from concurrent.futures import ThreadPoolExecutor


def cpu_task(n: int) -> int:
    return sum(i * i for i in range(n))


start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    list(pool.map(cpu_task, [2_000_000] * 4))
print(f"4 threads took {time.perf_counter() - start:.2f}s")

When the hot path sits in a GIL-releasing native library, threading scales much better; once it does not, move the work to processes or a distributed engine.

**Mechanism.** CPython keeps one lock around its interpreter state, so only one thread runs bytecode at a time. C extensions release that lock during long native calls, which is why NumPy, Arrow, and compression can still run in parallel.

**Worked example.** Four threads each summing two million squares take about as long as one thread doing four times the work. Four processes each take about a quarter, because each process owns an interpreter.

**Common mistake.** Assuming any threaded code is parallel. Check whether the hot loop is pure Python. If it is, use processes or move the loop into a native library.

## How do you read and write Parquet efficiently in Python?

Using `pyarrow` is common because it supports efficient columnar reads, predicate pushdown (engine-dependent), and schema handling. You should select only needed columns, write reasonably sized row groups/files, and avoid creating many tiny files. For large datasets, integrate with a distributed engine rather than pure local processing.

In [ ]:
# Column projection + pushed-down filter, then a typed write
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.parquet as pq

dataset = ds.dataset("s3://acme-curated/orders/", format="parquet", partitioning=["order_date"])
table = dataset.to_table(
    columns=["order_id", "customer_id", "amount"],
    filter=(pc.field("order_date") == "2026-10-04") & (pc.field("amount") > 0),
)

schema = pa.schema([("order_id", pa.string()), ("customer_id", pa.string()),
                    ("amount", pa.decimal128(12, 2))])
with pq.ParquetWriter("orders/ds=2026-10-04/part-00000.parquet", schema,
                      compression="zstd", use_dictionary=True) as writer:
    for batch in table.select(schema.names).to_batches(max_chunksize=100_000):
        writer.write_batch(batch)

Rules: never write one file per row from a local process; size files at 128-512 MB; use `zstd` cold and `snappy` hot; read with an explicit schema in production; use `pyarrow.fs`/`fsspec` for remote paths; and hand off to Spark/Polars when the work needs shuffles or hundreds of GB.

**Mechanism.** Parquet is columnar with per-column chunks and statistics, so a reader can skip row groups and read only the columns a query needs. Predicate pushdown is a property of the reader, not of the file format alone.

**Worked example.** A table with 50 columns is filtered by `order_date = '2026-10-04'`. Reading only `order_id`, `customer_id`, and `amount` for the matching row group touches a small fraction of the bytes on disk.

**Common mistake.** Writing thousands of tiny files, one per record. Every file carries a footer and metadata, so planning cost explodes. Aim for files in the 128-512 MB range.

## How do you process large CSV files without running out of memory?

You can stream the file in chunks (for example using chunked readers), process line-by-line, or use a library that supports streaming and type inference control. You also avoid loading wide columns you do not need, and you write results incrementally. For very large data, consider Parquet and/or distributed processing.

In [ ]:
# pandas chunks: aggregate incrementally, never hold the whole file
import pandas as pd

agg: dict[str, float] = {}
for chunk in pd.read_csv(
    "orders_2026.csv",
    usecols=["order_id", "amount", "country"],
    dtype={"amount": "float64", "country": "category"},
    chunksize=1_000_000,
):
    for country, total in chunk.groupby("country", observed=True)["amount"].sum().items():
        agg[country] = agg.get(country, 0.0) + total

```bash
# One-off scale: SQL over a CSV larger than memory (moves parsing into DuckDB)
duckdb -c "COPY (SELECT country, sum(amount) FROM read_csv_auto('orders_2026.csv') GROUP BY country) TO 'by_country.parquet' (FORMAT PARQUET);"
```

The theme is to aggregate or filter incrementally and emit as you go, so peak memory depends on chunk size and aggregation state rather than file size. Set explicit dtypes, because inference is itself a memory and time cost.

**Mechanism.** A CSV reader is a stream: it yields one row at a time and the memory cost is the row plus the buffer, not the file. Aggregating as you read keeps only the result state, such as one total per country.

**Worked example.** Reading a 40 GB file in one-million-row chunks and adding each chunk's `groupby` total into a dict uses memory proportional to the number of countries, not to the file size.

**Common mistake.** Calling `read_csv` without `chunksize` and then filtering. The full file is parsed into memory before any filter runs. Filter and aggregate inside the chunk loop instead.

## How do you implement retries with exponential backoff?

Retries should include exponential backoff and jitter to avoid thundering herds. You should only retry on transient errors (timeouts, rate limits) and limit total retry time. Always make operations idempotent or use deduplication to prevent duplicates on retries.

In [ ]:
# Capped exponential backoff with full jitter, or use tenacity's
# wait_exponential_jitter + retry_if_exception_type for the declarative version
import random
import time
from collections.abc import Callable
from typing import TypeVar

T = TypeVar("T")


class TransientError(Exception):
    """Raised for retryable failures (timeouts, 429, 503)."""


def retry(fn: Callable[[], T], *, attempts: int = 5, base: float = 0.5, cap: float = 30.0) -> T:
    last: Exception | None = None
    for attempt in range(1, attempts + 1):
        try:
            return fn()
        except TransientError as exc:
            last = exc
            if attempt == attempts:
                break
            time.sleep(random.uniform(0, min(cap, base * 2 ** (attempt - 1))))
    raise RuntimeError(f"retries exhausted after {attempts} attempts") from last

Rules: retry only transient classes; honor `Retry-After` on 429/503; cap total time, not just attempts; use idempotency keys on writes; add jitter; and log every attempt with the attempt number.

**Mechanism.** Transient failures are independent events, so retrying at once can hit the same overloaded server. Backoff spaces attempts out, and jitter spreads clients so they do not synchronise into a thundering herd.

**Worked example.** With base 0.5 s, factor 2, cap 30 s, and full jitter, the wait is `random.uniform(0, min(30, 0.5 * 2 ** attempt))`. A burst of retries spreads across the whole window instead of landing on one instant.

**Common mistake.** Retrying a `ValueError` or an HTTP 400. Retry only timeouts, 429, and 5xx, and only when the write is idempotent or carries an idempotency key.

## What logging practices are important for data pipelines?

Logs should include pipeline identifiers, run ids, input/output counts, and key timestamps. Structured logging (JSON) helps search and correlation. You should log errors with context and avoid logging sensitive data.

In [ ]:
# JSON logging with run-scoped context fields
import json
import logging
from typing import Any

logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger("orders")


class JsonFormatter(logging.Formatter):
    def format(self, record: logging.LogRecord) -> str:
        payload: dict[str, Any] = {"level": record.levelname,
                                   "logger": record.name,
                                   "message": record.getMessage()}
        payload.update(getattr(record, "extra_fields", {}))
        if record.exc_info:
            payload["traceback"] = self.formatException(record.exc_info)
        return json.dumps(payload)


handler = logging.StreamHandler()
handler.setFormatter(JsonFormatter())
log.handlers = [handler]
log.info("stage_complete", extra={"extra_fields": {"run_id": "r-123", "rows_in": 1000, "rows_out": 998}})

Standards that matter: one start/end line per stage with `run_id`, dataset, and partition; counts in and out; `logging` with a JSON formatter rather than `print`; never log secrets or raw PII; `exc_info=True` for exceptions; and a correlation id so logs join to traces and the control table.

**Mechanism.** `print` writes unstructured text to one stream. `logging` gives named loggers, levels, and formatters, so a handler can emit one JSON object per event that a log platform can index field by field.

**Worked example.** One JSON line per stage with `run_id`, `dataset`, `partition`, `rows_in`, `rows_out`, and `duration_ms`. Finding a failed run becomes a filter on `run_id`, not a regex over prose.

**Common mistake.** Logging secrets or raw PII, and logging inside a tight row loop. Log per batch or per stage, and never log a credential.

## How do you structure a Python project for data pipelines?

A common structure separates reusable library code from job entrypoints. You keep configuration separate, use dependency injection for external services, and standardize logging and error handling. Small CLI entrypoints improve local runs and CI tests.

A common layout is `src/<pkg>/` with `config.py`, `io/`, `transforms/` (pure functions), and `jobs/` (thin CLI entrypoints), plus `tests/unit`, `tests/integration`, and `dags/` where Airflow files import the package.

Principles: business logic lives in importable modules while `jobs/` is a thin shell; config is parsed into a typed object once (`Settings.from_env()` returning a frozen dataclass); clients are injected so tests can substitute fakes; there are **no import-time side effects** (DAG parsing depends on this); and the package is installed (`uv sync`) so jobs and tests import the same code.

## How do you manage dependencies and reproducible environments?

Use pinned dependencies and a lockfile (or equivalent) and isolate environments (virtualenv/containers). Reproducibility also requires pinning Python versions and building deterministic images. CI should use the same environment definition as production.

```bash
# Reproducible workflow: lock once, install exactly what CI and prod use
uv lock                         # writes uv.lock with hashes
uv sync --frozen                # install exactly the lockfile
uv run pytest                   # tests run in the pinned environment
```

```yaml
# pyproject.toml (excerpt, shown as YAML for readability)
project:
  name: acme-data
  requires-python: ">=3.12"
  dependencies: ["pyarrow>=17,<18", "polars>=1.0,<2", "httpx>=0.27,<1"]
  optional-dependencies:
    spark: ["pyspark==3.5.1"]
    dev: ["pytest>=8", "ruff", "mypy"]
```

Anti-patterns: unpinned transitive dependencies (behavior changes silently on rebuild); installing packages at task runtime; different environments for CI and production; and hand-maintaining hundreds of `==` pins instead of using a lockfile tool. Container images should pin the base Python by digest, `COPY uv.lock`, run `uv sync --frozen`, and then copy `src` - the same lock the tests use.

## How do you test data transformations in Python?

Use small fixture datasets and assert expected outputs (golden tests). Add property tests for invariants (non-negative amounts, uniqueness). Integration tests validate real connectors and schemas, but they should be isolated and stable.

Keep transformations **pure** so the fast unit layer covers most logic, then add one integration test per connector behind a marker.

In [ ]:
# tests/unit/test_orders.py
import pandas as pd
import pytest

from acme_data.transforms.orders import add_tax


@pytest.fixture
def orders() -> pd.DataFrame:
    return pd.DataFrame({"order_id": ["o1", "o2"], "amount_cents": [1000, 2000],
                         "country": ["DE", "US"]})


def test_add_tax(orders):
    out = add_tax(orders, rates={"DE": 0.19, "US": 0.0})
    assert out["amount_cents"].tolist() == [1190, 2000]


# Property test: for any list of non-negative amounts the tax never decreases them
# @given(st.lists(st.integers(min_value=0), min_size=1))
# def test_tax_never_decreases(amounts): ...

```bash
uv run pytest tests/unit -q                       # always, in CI
uv run pytest -m integration --maxfail=1          # opt-in, needs a test warehouse
```

Keep fixtures small enough to verify by eye, and assert business rules rather than snapshotting current output - change-detector tests pass forever and catch nothing.

**Mechanism.** A transformation test is pure: fixed input, call the function, compare the output to a literal. Keeping I/O outside the function means the test needs no cluster, no network, and no fixtures beyond plain data.

**Worked example.** A `dedupe(rows)` function keeps the last row per key. `assert dedupe([{"id": 1, "v": 1}, {"id": 1, "v": 2}]) == [{"id": 1, "v": 2}]` runs in microseconds and names the exact behaviour that broke.

**Common mistake.** Testing through a full pipeline run and asserting on a dashboard. It is slow, flaky, and does not localise the bug. Test the small function, then test the wiring separately.

## How do you profile and optimize slow Python code?

Profile first to find hotspots (CPU, IO, memory). Common fixes include vectorizing operations, reducing data copies, using efficient data structures, and offloading heavy compute to native libraries or distributed engines. Optimization should be guided by measurable impact.

Pick the tool by question: `cProfile` + `pstats` for wall-clock hotspots, `line_profiler` for a specific line, `tracemalloc`/`memray` for peak or leaking memory, and `py-spy dump` to see whether a running process is IO- or CPU-bound.

In [ ]:
# cProfile + pstats: sort by cumulative time to find the real hotspot
import cProfile
import pstats


def run_pipeline(ds: str) -> int:
    """Stand-in for the real pipeline so the profiler demo is runnable."""
    return sum(i * i for i in range(20_000))


def main() -> None:
    run_pipeline("2026-10-04")


profiler = cProfile.Profile()
profiler.enable()
main()
profiler.disable()
pstats.Stats(profiler).sort_stats("cumulative").print_stats(20)

Common wins in order of impact: vectorize row-wise `.apply`/loops (`(df["amount"] * (1 + df["tax_rate"])).sum()` instead of `iterrows`); avoid copies (`copy=False`, reuse buffers, prefer Arrow over `to_list()`); choose the right container; reduce IO by batching and streaming; and move compute to SQL/Spark when the data never needed to be in Python.

## How do you write memory-safe Python ETL code?

Memory-safe ETL code avoids loading unbounded data into lists or DataFrames. Prefer iterators, chunked reads, streaming writes, and explicit column selection. This is especially important for scripts that run in containers with fixed memory limits.

Example:

In [ ]:
import csv

def active_users(path):
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["status"] == "active":
                yield {
                    "user_id": int(row["user_id"]),
                    "email": row["email"].lower(),
                }

for user in active_users("users.csv"):
    send_to_api(user)

This processes one row at a time instead of building a full in-memory collection. In interviews, call out the trade-off: row-by-row code is memory efficient, while vectorized/batch processing can be faster when data fits in memory.

Checklist: bound every buffer (chunk, batch, and queue sizes); select columns in the reader rather than dropping later; use `category` dtype for low-cardinality strings and downcast numerics; never `concat` inside a loop (collect and concat once, or stream to disk); and cap concurrency so parallel fetches cannot exceed the container limit. Test with a larger-than-expected input, because a pipeline that only works on the sample partition is an outage waiting to happen.

## How do you design Python code so a data pipeline is restartable?

A restartable pipeline can safely resume after failure without corrupting downstream data. Use idempotent writes, deterministic output paths, checkpoints/control tables, and stable run identifiers. Avoid hidden global state and make external side effects explicit.

Example pattern:

In [ ]:
def run_partition(ds: str) -> None:
    source = f"s3://raw/events/ds={ds}/"
    staging = f"s3://warehouse/events_staging/run_ds={ds}/"
    final = f"s3://warehouse/events/ds={ds}/"

    records = transform(read_json(source))
    write_parquet(records, staging)
    validate_partition(staging)
    replace_partition(staging, final)
    mark_success("events", ds)

The important idea is to write to staging first, validate, then atomically publish or replace the final partition. If the job retries, it produces the same partition instead of appending duplicates. A control table (`platform.partition_state` with dataset, partition, status, attempt, rows_written, finished_at) tracks per-partition state so the loop can skip completed work.

Rules that make restart safe: deterministic paths (same logical date, same output); atomic publish via rename, partition overwrite, or MERGE; a stable run id passed through every stage; explicit enumeration of external side effects; and no mutable module-level state that assumes one run per process.

## How do you handle bad records in Python pipelines?

Do not let one malformed row either crash an entire batch blindly or disappear silently. A strong pipeline separates valid records from rejected records, records the reason for rejection, and emits metrics so data quality issues are visible.

In [ ]:
# Dead-letter records carry enough context to replay, plus a metric
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from typing import Any


@dataclass
class DeadLetter:
    source_file: str
    line_number: int
    error_message: str
    payload: str
    detected_at: str


def parse_with_dlq(row: dict[str, str], source_file: str, line_number: int) -> dict[str, Any] | None:
    try:
        return {"order_id": row["order_id"], "amount": float(row["amount"])}
    except (KeyError, ValueError) as exc:
        dead_letter.write(asdict(DeadLetter(
            source_file=source_file, line_number=line_number, error_message=str(exc),
            payload=json.dumps(row)[:4000], detected_at=datetime.now(timezone.utc).isoformat(),
        )))
        REJECTED.labels(pipeline="orders").inc()
        return None

In production, rejected records go to a dead-letter path/table with the source file, line number, pipeline version, and error message, so teams can fix upstream data or replay rejects after correction. Enforce a reject-rate threshold so a flood of malformed records blocks rather than silently publishing. **Failure mode:** swallowing bad rows with `try/except: pass` - the job goes green, the metric declines, and the records cannot be replayed.

**Mechanism.** A pipeline that raises on the first bad row stops a whole partition. A quarantine path records the bad record with its error and offset, and the good rows keep flowing.

**Worked example.** A per-row `try`/`except` appends `(raw_line, error)` to a dead-letter file and increments a counter. The job finishes, reports `bad_records=3`, and the file can be replayed later.

**Common mistake.** Silently dropping bad rows, or letting one malformed row fail the run. Count and store rejects, and fail only when the reject rate crosses a threshold you chose.

## When should you use pandas, Polars, PyArrow, or Spark from Python?

Use pandas for small to medium in-memory analysis and simple transformations. Use Polars or PyArrow when you need faster columnar processing, lazy execution, or efficient Parquet/Arrow interoperability on a single machine. Use Spark/PySpark when data volume, shuffle requirements, or production architecture require distributed processing.

Example decision rule:

+ pandas: one CSV or Parquet extract that fits comfortably in memory
+ PyArrow: efficient file/schema operations, Parquet datasets, Arrow interchange
+ Polars: fast local transformations and lazy query optimization
+ PySpark: joins, aggregations, and pipelines over data too large for one machine

The interview answer should emphasize workload shape, not tool popularity. A 5 GB file may be fine in Polars on a large instance, while a 500 GB join with skewed keys usually belongs in Spark or a warehouse.

In [ ]:
# Polars: lazy scan with projection and predicate pushdown on a large local dataset
import polars as pl

result = (
    pl.scan_parquet("s3://acme-curated/orders/*/*.parquet")
    .filter(pl.col("order_date") == "2026-10-04")
    .group_by("country")
    .agg(pl.col("amount").sum().alias("revenue"))
    .collect(streaming=True)
)

Rule of thumb: **stay single-node until the data genuinely exceeds memory or the shuffle does not fit on one machine.** Polars/DuckDB push that boundary higher than pandas, and moving to Spark prematurely costs cluster operations and JVM serialization you did not need to pay.

## Real world use cases for Python for Data Engineering

### 1. E-commerce: API ingestion with retries and a dead-letter queue

A retailer pulls orders from a partner REST API that rate-limits aggressively. Roughly 4M orders/day arrive in 5-minute windows, then land in S3 and load into the warehouse.

In [ ]:
# Bounded-concurrency fetcher with retry and backoff; exhausted pages go to a DLQ
import asyncio

import httpx
from tenacity import retry, retry_if_exception_type, stop_after_attempt, wait_exponential_jitter


class RateLimited(Exception):
    pass


@retry(retry=retry_if_exception_type((RateLimited, httpx.TransportError)),
       wait=wait_exponential_jitter(initial=1, max=60), stop=stop_after_attempt(5), reraise=True)
async def fetch_page(client: httpx.AsyncClient, cursor: str | None) -> dict:
    response = await client.get("/orders", params={"cursor": cursor, "limit": 500})
    if response.status_code == 429:
        raise RateLimited(response.headers.get("Retry-After", "unknown"))
    response.raise_for_status()
    return response.json()

A page that exhausts retries is written to a quarantine prefix with its cursor, so the pull resumes from the last good cursor instead of re-reading the whole day.

**Why this design:** the API is the bottleneck, so async IO plus bounded concurrency maximizes throughput, retries with jitter absorb 429s, and a durable cursor makes a failure cost one page rather than one day.

### 2. Fintech: reconciliation of two systems with Polars

A payments team reconciles ~120M transactions/day between a core ledger and the warehouse, classifying differences before the close.

In [ ]:
# Vectorized reconciliation: a set join, not a loop over 120M rows
import polars as pl

ledger = pl.scan_parquet("s3://acme-ledger/txn/dt=2026-10-04/*.parquet").select("txn_id", "amount_cents")
warehouse = pl.scan_parquet("s3://acme-wh/txn/dt=2026-10-04/*.parquet").select("txn_id", "amount_cents")

recon = (
    ledger.join(warehouse, on="txn_id", how="full", suffix="_wh")
    .with_columns(
        pl.when(pl.col("amount_cents").is_null()).then(pl.lit("missing_in_warehouse"))
        .when(pl.col("amount_cents_wh").is_null()).then(pl.lit("missing_in_ledger"))
        .when(pl.col("amount_cents") != pl.col("amount_cents_wh")).then(pl.lit("amount_mismatch"))
        .otherwise(pl.lit("match")).alias("recon_status")
    )
    .group_by("recon_status").agg(pl.len().alias("count"))
    .collect(streaming=True)
)

**Why this design:** reconciliation is a set operation, so a vectorized join handles 120M rows on one node in minutes, and the output classifies every difference instead of returning a binary gate the finance team cannot act on.

### 3. Multi-tenant SaaS: memory-bounded export per tenant

A B2B platform lets customers download their data. Some tenants have billions of events; the export service runs in a container with a 2 GB limit and must never OOM.

In [ ]:
# Server-side cursor plus fetchmany keeps memory proportional to the batch
BATCH = 100_000


def stream_tenant_events(conn, tenant_id: str, out_path: str) -> int:
    written = 0
    with gzip.open(out_path, "wt", encoding="utf-8") as handle, conn.cursor(name="export_cursor") as cur:
        cur.itersize = BATCH
        cur.execute("SELECT event_id, event_ts FROM events WHERE tenant_id = %s", (tenant_id,))
        while rows := cur.fetchmany(BATCH):
            for row in rows:
                handle.write(json.dumps(row) + "\n")
            written += len(rows)
    return written

**Why this design:** the container limit is a hard constraint, so memory is bounded by the batch rather than the tenant, and compressed NDJSON streams bytes to disk instead of buffering them.

### 4. ML feature platform: a reproducible training snapshot

An ML team builds daily features for 40M users. Reproducibility matters because model comparisons must attribute changes to the model, not to a re-run of feature code.

In [ ]:
# Installable, versioned feature job with a content hash for the snapshot
import hashlib

import polars as pl


def build_features(input_glob: str, code_version: str) -> tuple[pl.DataFrame, str]:
    frame = (
        pl.scan_parquet(input_glob)
        .group_by("user_id")
        .agg(pl.col("event_type").eq("view").sum().alias("views_1d"),
             pl.col("amount").sum().alias("spend_1d"))
        .collect(streaming=True)
    )
    digest = hashlib.sha256(f"{code_version}{frame.shape}{frame.sum().to_dict()}".encode())
    return frame, digest.hexdigest()[:16]

**Why this design:** the job is a normal installable package with a lockfile, so CI and production run the same code and dependency versions, and a content hash identifies the exact snapshot an experiment trained on.

---

# Practice notebook: Python basics

The cells below are from `notebooks/01_python_basics.ipynb` and are merged here so this Colab notebook is self-contained. Run them top to bottom; assertions confirm the expected behavior.

# Python Basics for Data Engineering Interviews

This notebook is a hands-on tour of the Python language features that come up in data-engineering interviews.
It is deliberately granular: every idea gets its own explanation and its own small, runnable example, so you can
stop, predict the output, and only then run the cell.

**How to run it**

1. Run the cells top to bottom with `Shift+Enter`, or use *Run All*.
2. Outputs are saved in the notebook, so it reads like a document even without a kernel.
3. Most clusters of examples end with a **Check yourself** cell full of `assert`s; if it runs silently, your
   mental model is correct.

Companion notebooks: `02_python_coding_problems.ipynb` (timed algorithms), `03_sql_interview_questions.ipynb`
(SQL against SQLite), `04_pyspark_concepts.ipynb` (Spark), and `05_interview_questions.ipynb` (mixed bank).

#### A two-line sanity check

Everything here uses the standard library only: no pandas, no numpy, no Spark. Run this cell first; it proves
the kernel is alive and shows which Python you are on. interviewers expect you to know the version you target,
because syntax such as `str | None` and the `zoneinfo` module need Python 3.10+.

In [ ]:
import sys

print("python", sys.version.split()[0])
print("ready")

## 1. Values, dynamic typing and numbers

### Names are bound to objects

In Python a variable is a *name* bound to an object, and the object carries the type. There is no declaration,
no compile-time type check, and a name can be rebound to an object of a different type at any moment. Data
engineers care because a column that is "usually an int" can arrive as `None` or a string from a messy source,
and the failure shows up at runtime, deep inside a pipeline.

### `type()`, `isinstance()` and numbers

`type(obj)` tells you what an object is, and `isinstance(obj, T)` is the safe runtime check: it respects
subclassing, whereas `type(x) is T` is exact and brittle. Data engineers care because a column that is "usually
an int" can arrive as `None` or a string from a messy source.

`int` is arbitrary precision, so it never overflows. `float` is a 64-bit IEEE-754 double: fine for measurements
and averages, hopeless for exact money. Note that `bool` is a subclass of `int`, which is why `True + True == 2`.

In [ ]:
value = 42
print(type(value).__name__)
value = "42"
print(type(value).__name__)
print(isinstance(True, int), type(True) is int)

rows = 10 ** 18 + 7
print(rows, 3 / 4, 1_000_000)       # ints never overflow; _ is cosmetic

### True division, floor division, modulo and rounding

`/` always returns a float; `//` floors toward negative infinity; `%` is the remainder and its sign follows the
divisor. Batching code that assumes `/` is integer division, or that splits negative offsets the "obvious" way,
is a classic source of off-by-one bugs when paging through partitions.

`round()` uses banker's rounding (half-to-even) and its second argument is *digits*, not a step, while
`math.floor`/`math.ceil` return ints and are the right tools for bucket boundaries. Interviewers like asking why
`round(2.675, 2)` is not `2.68`.

In [ ]:
import math

print(7 / 2, 7 // 2, 7 % 2)
print(-7 // 2, -7 % 2)               # floors toward -inf
print(divmod(7, 2))                  # quotient and remainder
print(round(2.5), round(3.5), round(2.675, 2))
print(math.floor(-2.1), math.ceil(2.1), (95 + 9) // 10)

### The float precision trap

Binary floating point cannot represent most decimal fractions exactly, so `0.1 + 0.2 != 0.3`. Never compare
floats with `==` in validation code; use `math.isclose` with an explicit tolerance, or don't use floats for
values that must balance.

In [ ]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)
print(math.isclose(0.1 + 0.2, 0.3, rel_tol=1e-9))
print(1.1 + 2.2)

### `Decimal` for money

`decimal.Decimal` is exact decimal arithmetic with configurable rounding, and it is the standard answer to
"how do you sum revenue without losing cents?". Build it from *strings*, not floats -- `Decimal(0.1)` inherits
the binary error, while `Decimal("0.1")` is exactly one tenth.

In [ ]:
from decimal import Decimal

print(Decimal("0.1") + Decimal("0.2"))     # exact
print(Decimal(0.1))                        # float error carried in

total = sum(Decimal(amount) for amount in ["19.99", "4.01", "0.99"])
print("total:", total)

**Check yourself.** These asserts cover types, division, rounding and money arithmetic. Predict each
value before you run the cell; the asserts are the feedback loop that replaces a code reviewer.

In [ ]:
import math
from decimal import Decimal

assert isinstance(7 // 2, int) and isinstance(7 / 2, float)
assert divmod(7, 2) == (3, 1)
assert math.ceil(95 / 10) == 10
assert Decimal("0.1") + Decimal("0.2") == Decimal("0.3")
assert not math.isclose(0.1, 0.2) and math.isclose(0.1 + 0.2, 0.3, rel_tol=1e-9)
print("section 1 checks passed")

## 2. Strings

### Literals, indexing and immutability

A `str` is an immutable sequence of Unicode code points. Immutability means every "modification" builds a new
string, which is why repeated `+=` in a loop is quadratic and worth flagging in an interview. Indexing is
zero-based and negative indices count from the end.

In [ ]:
text = "data"
print(text[0], text[-1], len(text))
print(text[0:2], text[:2], text[2:])

upper = text.upper()
print(upper, text)          # original is untouched: strings are immutable

### f-strings

f-strings are the modern, readable way to build messages, and they support format specs, conversions and even
inline expressions. In logging and exception messages they are strongly preferred over `%` and `.format`;
the only caveat is to keep expressions short and side-effect free.

In [ ]:
table, rows, rate = "orders", 1234, 0.98765

print(f"{table}: {rows:,} rows")
print(f"success rate {rate:.1%}")
print(f"{table!r} padded -> |{table:>10}|")
print(f"{rows * 2 = }")               # debug form, Python 3.8+

### Common methods

`strip`, `lower`, `startswith`, `replace` and friends are the workhorses of cleaning raw text. They all return
new strings, so chaining is idiomatic and safe. Remember `strip()` removes whitespace from both ends, and
`strip("ab")` removes any of the characters `a`/`b`, not the substring `"ab"`.

In [ ]:
raw = "  Data Engineering  "
print("[" + raw.strip() + "]")
print(raw.strip().lower())
print(raw.strip().replace(" ", "_"))
print("orders_2026.csv".startswith("orders"), end=" ")
print("orders_2026.csv".endswith((".csv", ".json")))

### `split` and `join` versus `+` in a loop

`split` turns text into a list, `join` turns an iterable of strings back into one string, and `join` is the
only efficient way to concatenate many pieces. `"".join(parts)` allocates once; `total += part` in a loop
allocates a new string every iteration. This is a favourite performance question.

In [ ]:
line = "o1,10.5,USD"
fields = line.split(",")
print(fields, len(fields))

row = ["2026-10-04", "o1", "10.5"]
print(",".join(row))
print(" / ".join(["a", "b", "c"]))

**Check yourself.** String basics: slicing, immutability, formatting, cleaning and joining.

In [ ]:
assert "engineering"[0:3] == "eng"
assert "  x  ".strip() == "x"
assert "-".join(["a", "b"]) == "a-b"
assert "a,b,c".split(",") == ["a", "b", "c"]
assert f"{1234:,}" == "1,234"
assert "Data".lower() == "data" and "Data"[0] == "D"
print("section 2 checks passed")

## 3. Lists

### Literal, indexing and slicing

A `list` is an ordered, mutable sequence of arbitrary objects. Indexing is O(1); slicing copies a range into a
new list. Slicing with a step is how you sample every n-th row, and `[::-1]` reverses. Note that negative step
slices behave differently from `reversed()` for very large lists because they copy.

In [ ]:
numbers = [3, 1, 4, 1, 5, 9, 2, 6]
print(numbers, numbers[0], numbers[-1])
print(numbers[2:5], numbers[:3], numbers[-3:])
print(numbers[::2], numbers[::-1])
print(len(numbers), 5 in numbers)

### Mutation

`append`, `insert`, `pop`, `remove` and item assignment change the list in place and return `None` -- a common
interview trap is `rows = rows.append(x)`, which sets `rows` to `None`. Mutation is visible through every name
that references the same list, a theme we return to in the mutability section.

In [ ]:
rows = ["r1", "r2"]
rows.append("r3")
rows[0] = "r0"
rows.insert(1, "r1.5")
print(rows)

removed = rows.pop()
print(rows, "removed:", removed)
rows.remove("r1.5")
print(rows)

### `append` vs `extend` vs `+`

`append(x)` adds one element (even if `x` is a list); `extend(iterable)` adds each element; `a + b` builds a
brand-new list and leaves both inputs alone. Getting this wrong produces the infamous list-of-lists bug in
batch collectors, where `append` nests batches instead of flattening them.

In [ ]:
batch = [1, 2]
batch.append([3, 4])       # one element: a nested list
print("append:", batch)

batch = [1, 2]
batch.extend([3, 4])       # each element
print("extend:", batch)

combined = [1, 2] + [3, 4]
print("plus  :", combined)

### Sorting with `key=`

`sorted()` returns a new list and `list.sort()` sorts in place; both accept `key=` so you can sort by a derived
value without touching the data. A key can return a tuple, which is how you implement a deterministic
multi-field sort: primary field ascending, secondary field descending. Ties matter in data pipelines because
non-deterministic ordering breaks reproducible tests.

In [ ]:
counts = {"b": 2, "a": 2, "c": 5}
ranked = sorted(counts, key=lambda key: (-counts[key], key))
print(ranked)

records = [{"id": "o2", "amount": 5}, {"id": "o1", "amount": 9}]
by_amount = sorted(records, key=lambda row: row["amount"])
print([row["id"] for row in by_amount])

**Check yourself.** Lists: slicing, in-place mutation, `append` vs `extend`, and deterministic sorting.

In [ ]:
assert [1, 2, 3][::-1] == [3, 2, 1]
assert [1, 2, 3][1:] == [2, 3]

data = [3, 1, 2]
data.sort()
assert data == [1, 2, 3]

nested = [1]
nested.append([2])
assert nested == [1, [2]]
flat = [1]
flat.extend([2])
assert flat == [1, 2]
assert [2, 1] + [3] == [2, 1, 3]
print("section 3 checks passed")

## 4. Tuples

### Immutability is the feature

A tuple is an ordered, immutable sequence. Because it cannot change, a tuple of immutables is hashable and can
be a dict key or a set member -- exactly what you want for a composite key such as `(customer_id, ds)`. The
classic ML mistake of using a list where a hashable key is required is an easy interview warm-up.

In [ ]:
record = ("order-1", 42.5)
print(record, record[0], len(record))

try:
    record[0] = "order-2"
except TypeError as exc:
    print("TypeError:", exc)

lookup = {("cust-1", "2026-10-04"): 12}
print(lookup[("cust-1", "2026-10-04")])

### Unpacking and swapping

Tuple unpacking binds several names in one statement and works with any iterable. It is the idiomatic Python
swap, the idiomatic way to receive a function's multiple return values, and the reason `for key, value in
items` is so readable. Use `*rest` to collect the remainder.

In [ ]:
order_id, amount = ("o1", 10.5)
print(order_id, amount)

left, right = 1, 2
left, right = right, left
print(left, right)

first, *rest = [1, 2, 3, 4]
print(first, rest)
head, *middle, tail = (1, 2, 3, 4)
print(head, middle, tail)

### Fixed-shape records

The convention is: tuples for heterogeneous fixed-shape records (a row, a key, a return value), lists for
homogeneous variable-length sequences (a batch of rows). A tuple also documents arity -- `(id, amount)` says
exactly two fields will always be there, which is why query results and coordinates are usually tuples.

In [ ]:
Point = tuple  # for illustration only
point = (3, 4)
x, y = point
print("manhattan:", abs(x) + abs(y))

def min_max(values):
    return min(values), max(values)

low, high = min_max([4, 1, 9])
print(low, high)

**Check yourself.** Tuples: immutability, hashability, unpacking and swapping.

In [ ]:
assert (1, 2) + (3,) == (1, 2, 3)
assert (1, 2)[0] == 1
a, b = 1, 2
a, b = b, a
assert (a, b) == (2, 1)
assert {("k", 1): "v"}[("k", 1)] == "v"
assert min_max([2, 8]) == (2, 8)
print("section 4 checks passed")

## 5. Sets

### Deduplication and membership

A `set` stores unique, hashable elements with average O(1) membership tests, versus O(n) for a list. That one
fact answers a large share of "how would you speed this up" questions: build a set of seen IDs once, then test
membership in a loop. Sets are unordered, so never rely on their iteration order.

In [ ]:
events = ["click", "view", "click", "buy", "view"]
print("list:", events)
print("set :", set(events))
print("unique count:", len(set(events)))

seen = {"a", "b"}
print("a" in seen, "z" in seen)         # O(1) average

### Set algebra

`|` is union, `&` is intersection, `-` is difference and `^` is symmetric difference; each also has a named
method and an in-place operator (`|=`, `&=`, ...). Set algebra is the clearest way to express reconciliation
logic: rows only in yesterday's file, rows in both, rows that appeared or disappeared.

In [ ]:
yesterday = {"o1", "o2", "o3"}
today = {"o2", "o3", "o4"}

print("union       :", sorted(yesterday | today))
print("intersection:", sorted(yesterday & today))
print("only today  :", sorted(today - yesterday))
print("symmetric   :", sorted(yesterday ^ today))

**Check yourself.** Sets: dedupe, membership and the four set operations.

In [ ]:
assert len(set([1, 1, 2, 3])) == 3
assert set("mississippi") == set("mississippi")
assert {1, 2} | {3} == {1, 2, 3}
assert {1, 2, 3} & {2, 3, 4} == {2, 3}
assert {1, 2, 3} - {2} == {1, 3}
assert {1, 2} ^ {2, 3} == {1, 3}
assert 3 in {1, 2, 3}
print("section 5 checks passed")

**Mechanism.** A set is a hash table, so `in` is an average O(1) probe instead of a scan. Elements must be hashable and equal values collapse into one, which is why a set keeps neither duplicates nor order.

**Worked example.** Deduplicating a million order ids with `set(ids)` is linear. Testing each id against a list would be quadratic. `len(ids) - len(set(ids))` counts the duplicates in one pass.

**Common mistake.** Relying on set iteration order, or putting a list inside a set. Use `sorted(seen)` when output order matters, and use tuples as hashable keys.

## 6. Dictionaries

### Literal access versus `get`

A dict maps hashable keys to values with average O(1) lookup, and since Python 3.7 it preserves insertion order.
`d[key]` raises `KeyError` when the key is missing; `d.get(key, default)` returns a fallback. In pipeline code
`get` is usually what you want for optional columns, while `[]` is the right choice when a missing key really is
a bug that should fail loudly.

In [ ]:
config = {"mode": "append", "compression": "zstd"}
print(config["mode"])
print(config.get("partition_by", "none"))

try:
    config["missing"]
except KeyError as exc:
    print("KeyError:", exc)

### Grouping and counting

`d.setdefault(key, [])` returns the existing value or inserts the default and returns it, which turns a
group-by into a two-line loop. `collections.defaultdict` does the same with less noise, and `collections.Counter`
is a purpose-built tally with helpers such as `most_common`. Both are standard library and both come up in
"solve this without pandas" exercises.

In [ ]:
from collections import Counter, defaultdict

orders = [("us", 10), ("eu", 5), ("us", 7), ("apac", 2)]
by_region = {}
for region, amount in orders:
    by_region.setdefault(region, []).append(amount)
print(by_region, {region: sum(v) for region, v in by_region.items()})

counts = defaultdict(int)
for word in "the cat the hat".split():
    counts[word] += 1
print(dict(counts), Counter("mississippi").most_common(2))

### Insertion order and iterating items

Iteration yields keys by default; use `.items()` for pairs, `.values()` for values, and `.keys()` (or the dict
itself) for keys. Because order is insertion order, `list(d.items())` is deterministic -- but if you need a
guaranteed order independent of insertion, sort explicitly.

In [ ]:
pipeline = {"extract": "s3", "transform": "spark", "load": "snowflake"}
for stage, tool in pipeline.items():
    print(f"{stage:>10} -> {tool}")

print(list(pipeline))
print(sorted(pipeline.items()))

### Dict comprehensions

A dict comprehension builds a mapping in one expression and reads like the set-builder notation it is named
after. Keep them to a single `for` and at most one `if`; beyond that a loop is clearer, which is itself a
reasonable thing to say in an interview.

In [ ]:
squares = {n: n * n for n in range(5)}
print(squares)

raw = {"a": "1", "b": "2", "skip": ""}
clean = {key: int(value) for key, value in raw.items() if value}
print(clean)

**Check yourself.** Dicts: access, defaults, grouping, tallies, ordering and comprehensions.

In [ ]:
from collections import Counter, defaultdict

assert {"a": 1}.get("b", 0) == 0
grouped = {}
for key, value in [("a", 1), ("a", 2)]:
    grouped.setdefault(key, []).append(value)
assert grouped == {"a": [1, 2]}

assert dict(defaultdict(int, {"a": 1})) == {"a": 1}
assert Counter("aab").most_common(1) == [("a", 2)]
assert list({"x": 1, "y": 2}.items()) == [("x", 1), ("y", 2)]
assert {n: n for n in range(3)} == {0: 0, 1: 1, 2: 2}
print("section 6 checks passed")

## 7. Mutability, references and copies

### Names reference objects

Assignment never copies: it binds another name to the *same* object, and `id()` exposes that identity. This is
why passing a list into a function and mutating it is visible to the caller, and why two "separate" config
dicts can secretly be one object. Almost every surprising "my data changed" bug traces back to aliasing.

In [ ]:
a = [1, 2, 3]
b = a                      # alias, not a copy
b.append(4)
print(a, b, a is b, id(a) == id(b))

c = list(a)                # real copy
c.append(99)
print(a, c, a is c)

### Shallow versus deep copy

`copy.copy` creates a new outer container but shares the inner objects; `copy.deepcopy` recursively copies
everything. Shallow copies are cheap and usually enough when you only replace top-level entries; deep copies
protect you when nested structures are mutated, at the cost of memory and time.

In [ ]:
import copy

nested = [[1], [2]]
shallow = copy.copy(nested)
deep = copy.deepcopy(nested)

nested[0].append("boom")
print("shallow:", shallow)     # inner list shared
print("deep   :", deep)        # fully independent

### Mutation through function arguments

Python is call-by-object-sharing: the function receives the same object, so mutating it affects the caller,
but rebinding the parameter inside the function does not. Interviewers ask this to see whether you can explain
why a helper that "resets" a list argument silently fails.

In [ ]:
def add_row_bad(rows):
    rows = rows + ["new"]      # rebinds locally: caller never sees it

def add_row_good(rows):
    rows.append("new")         # mutates the shared object

rows = ["r1"]
add_row_bad(rows)
print("after bad :", rows)
add_row_good(rows)
print("after good:", rows)

**Check yourself.** Aliasing, copies and mutation through arguments.

In [ ]:
original = [1, 2, 3]
alias = original
assert alias is original  # two names, one object

clone = original[:]
clone.append(4)
assert original == [1, 2, 3] and clone == [1, 2, 3, 4]

In [ ]:
import copy

nested = [[1]]
shallow = copy.copy(nested)
deep = copy.deepcopy(nested)
nested[0].append(2)
assert shallow[0] == [1, 2]   # shallow shares the inner list
assert deep[0] == [1]         # deep is fully independent

In [ ]:
def mutate(items):
    items.append("x")

bucket = []
mutate(bucket)
assert bucket == ["x"]        # the caller's list changed
print("section 7 checks passed")

## 8. Control flow

### `if` / `elif` / `else`

Conditions are expressions; there is no `switch` requirement and no braces, just indentation. In data code the
common shape is a small validation ladder: reject bad input early, classify the row, otherwise continue. Keep
branches shallow and return early rather than nesting -- a code-review point that also reduces cyclomatic
complexity.

In [ ]:
def classify(amount):
    if amount < 0:
        return "refund"
    elif amount == 0:
        return "empty"
    else:
        return "sale"

print([classify(value) for value in (-5, 0, 12)])

### Truthiness

Every object has a boolean value: empty containers, `0`, `0.0`, `""`, `None` and `False` are falsy, everything
else is truthy. `if rows:` is idiomatic and safe, but it conflates "empty" with "missing" -- if that distinction
matters, write `if rows is None` explicitly. This is a favourite interview nuance.

In [ ]:
for value in [0, 0.0, "", [], {}, None, False, 1, "x", [0]]:
    print(repr(value), "->", bool(value))

### Loop control: `for`/`else`, `while`, `break`, `continue`

The `else` on a loop runs only if the loop finished without `break`; it is the clean way to express "search, and
report if not found" without a flag variable. `while` repeats until a condition changes, which suits retry and
cursor loops, and `break`/`continue` exit or skip the current iteration. Always make sure the loop variable
actually changes, or you have written an infinite loop.

In [ ]:
for row in [1, 2, 3]:
    if row == 2:
        print("found 2")
        break
else:
    print("no 2")

for row in [1, 3]:
    if row == 2:
        break
else:
    print("no 2 in second loop")

attempt = 0
while True:
    attempt += 1
    if attempt < 3:
        continue
    print("succeeded on attempt", attempt)
    break

### `match` (structural pattern matching, 3.10+)

`match` compares a subject against patterns and can destructure sequences, mappings and classes. It is not a
faster `if`; it shines when routing on shapes, such as dispatching on a config dict or an event type. A `case _
` wildcard is required if you want a default.

In [ ]:
def route(event):
    event_type = event.get("type")
    if event_type == "insert" and "table" in event:
        return f"insert into {event['table']}"
    if event_type == "delete":
        return "delete"
    return "unknown"

print(route({"type": "insert", "table": "orders"}))
print(route({"type": "delete"}))
print(route({"type": "other"}))


**Check yourself.** Branching, truthiness, loop `else`, `break`/`continue` and `match`.

In [ ]:
assert bool([]) is False and bool([0]) is True
assert bool(0.0) is False and bool("0") is True

In [ ]:
found = False
for row in [1, 2, 3]:
    if row == 3:
        found = True
        break
assert found

total = 0
for row in [1, 2, 3, 4]:
    if row % 2:
        continue
    total += row
assert total == 6

In [ ]:
def route(event):
    event_type = event.get("type")
    if event_type == "insert" and "table" in event:
        return f"insert into {event['table']}"
    return "unknown"

assert route({"type": "insert", "table": "t"}) == "insert into t"
assert route({}) == "unknown"
print("section 8 checks passed")


## 9. Comprehensions and generator expressions

### List comprehensions

A list comprehension expresses "build a list by transforming/filtering an iterable" in one readable expression,
usually faster than an equivalent `for` + `append`. Read it as: output expression, then the loops, then the
conditions. Prefer it when the logic is a single transformation.

In [ ]:
numbers = [3, 1, 4, 1, 5, 9, 2, 6]
evens = [n for n in numbers if n % 2 == 0]
squares = [n * n for n in numbers]
print("evens  :", evens)
print("squares:", squares)

### Dict and set comprehensions

The same syntax builds dicts (with `key: value`) and sets (deduplicating automatically). A set comprehension
is the fastest way to collect distinct partition keys while transforming them at the same time.

In [ ]:
words = ["Orders", "orders", "CUSTOMERS", "customers"]
print({w.lower() for w in words})              # set comprehension
print({w.lower(): len(w) for w in words})      # dict comprehension

### Generator expressions -- and when *not* to use a comprehension

Parentheses instead of brackets create a generator expression: lazy, single-use and O(1) memory, which makes it
the right argument to `sum`, `any`, `all`, `max` and `join`. Comprehensions are for mapping and filtering, not
for side effects, deep branching or multi-level state. If the expression needs more than one `for` and one `if`,
write a plain loop -- "I would keep a loop for clarity" is a strong interview answer.

In [ ]:
total = sum(n * n for n in range(1_000_000))     # no million-element list
print(total, any(n > 999_998 for n in range(1_000_000)))
print(", ".join(str(n) for n in range(4)))

pairs = []
for left in range(3):
    for right in range(3):
        if left != right:
            pairs.append((left, right))
print(pairs)

**Check yourself.** Comprehensions and generator expressions, plus laziness.

In [ ]:
assert [n for n in range(5) if n % 2 == 0] == [0, 2, 4]
assert {n % 2 for n in range(5)} == {0, 1}
assert {n: n * n for n in range(3)} == {0: 0, 1: 1, 2: 4}
assert sum(n for n in range(5)) == 10

lazy = (n for n in range(3))
assert next(lazy) == 0
assert list(lazy) == [1, 2]
print("section 9 checks passed")

## 10. Functions

### `def`, `return` and signatures

Functions are the unit of reuse and testing in a pipeline: a pure function that turns rows into rows is trivial
to unit test, while a function that reads S3, transforms and writes is not. State the signature, return a value,
and avoid printing as a side effect in library code.

In [ ]:
def add_tax(amount, rate=0.2):
    return round(amount * (1 + rate), 2)

print(add_tax(100))
print(add_tax(100, rate=0.1))
print(add_tax.__name__, add_tax.__defaults__)

### Default arguments

Defaults are evaluated **once**, at function definition time. That is fine for immutables and dangerous for
containers, which is why `None` sentinels are the standard pattern. The pitfall is subtle enough that it appears
in almost every Python interview and in plenty of real code review.

In [ ]:
def append_bad(item, bucket=[]):
    bucket.append(item)
    return bucket

print(append_bad("a"))
print(append_bad("b"))          # surprising: the default is shared

def append_good(item, bucket=None):
    bucket = [] if bucket is None else bucket
    bucket.append(item)
    return bucket

print(append_good("a"), append_good("b"))

### `*args` and `**kwargs`

`*args` collects extra positional arguments into a tuple and `**kwargs` collects extra keyword arguments into a
dict; together, `def wrapper(*args, **kwargs)` is the skeleton of every decorator you will write. `*args` keeps
logging and batch helpers generic, and `**kwargs` lets a wrapper forward options it does not understand to the
function it wraps. Note that parameters written after `*args` are keyword-only.

In [ ]:
def total(*values):
    return sum(values)

def load(table, **options):
    return {"table": table, **options}

print(total(), total(1, 2), total(*[1, 2, 3]))
print(load("orders", mode="overwrite", compression="zstd"))
print(load("events", **{"mode": "append"}))

### Keyword-only arguments

A bare `*` in the signature marks everything after it as keyword-only, so `save(path, mode="append")` cannot be
called positionally by mistake. This is a cheap API-design win for functions with several boolean-ish options,
and it makes call sites self-documenting.

In [ ]:
def write_partition(table, *, mode="append", dry_run=False):
    return f"{table} mode={mode} dry_run={dry_run}"

print(write_partition("orders"))
try:
    write_partition("orders", "overwrite")
except TypeError as exc:
    print("TypeError:", exc)

### Multiple returns and docstrings

Returning a tuple is the Pythonic way to return several values; the caller unpacks it. Docstrings document the
contract and are what `help()` and your IDE show -- write them for anything that is not self-evident, and
remember that tuple returns are exactly why tuples are called "fixed-shape records".

In [ ]:
def summarize(rows):
    """Return (count, total) for an iterable of numeric rows."""
    values = list(rows)
    return len(values), sum(values)

count, total = summarize([1, 2, 3])
print(count, total)
print(summarize.__doc__)

**Check yourself.** Defaults, `*args`, `**kwargs`, keyword-only parameters and tuple returns.

In [ ]:
def append_good(item, bucket=None):
    bucket = [] if bucket is None else bucket
    bucket.append(item)
    return bucket

assert append_good("a") == ["a"] and append_good("b") == ["b"]

def total(*values):
    return sum(values)

assert total(1, 2, 3) == 6 and total() == 0

def configured(*, mode="append", **options):
    return {"mode": mode, **options}

assert configured(mode="overwrite", x=1) == {"mode": "overwrite", "x": 1}
assert summarize([2, 4]) == (2, 6)
print("section 10 checks passed")

## 11. Scope and closures

### The LEGB rule

Name lookup goes Local, then Enclosing, then Global, then Built-in. Reading an enclosing variable works without
any keyword; *rebinding* it requires `nonlocal` (enclosing) or `global` (module). Understanding LEGB explains
both closure behaviour and the nasty class of bug where a loop variable leaks into a later function call.

In [ ]:
greeting = "global"

def outer():
    greeting = "enclosing"
    def inner():
        return greeting          # reads the enclosing binding
    return inner()

print(outer(), greeting)

### Rebinding outer names: `global` and `nonlocal`

Reading an enclosing variable needs no keyword, but *rebinding* it does: `nonlocal` for the nearest enclosing
function scope and `global` for a module-level name. `global` is occasionally needed for module state such as a
counter, but module-level mutable state breaks test isolation and parallel runs. `nonlocal` is the mechanism
behind closures that keep state, such as factory-produced counters.

In [ ]:
counter = 0

def bump():
    global counter
    counter += 1

def make_counter():
    count = 0
    def increment():
        nonlocal count
        count += 1
        return count
    return increment

bump()
bump()
first, second = make_counter(), make_counter()
print(counter, [first(), first(), second(), first()])

### Late binding and the default-argument escape hatch

A lambda or inner function that references a loop variable looks up that variable when it is *called*, not when
it is defined -- so a list of lambdas built in a loop all see the final value. Bind the current value with a
default argument (`lambda i=i: i`) or `functools.partial` to freeze it. This is the same "defaults are
evaluated once" rule from the functions section, used deliberately.

In [ ]:
callbacks = [lambda: i for i in range(3)]
print([callback() for callback in callbacks])       # all 2

fixed = [lambda i=i: i for i in range(3)]
print([callback() for callback in fixed])           # 0, 1, 2

**Check yourself.** Scope, `nonlocal` and late binding.

In [ ]:
def make_counter():
    count = 0
    def increment():
        nonlocal count
        count += 1
        return count
    return increment

counter = make_counter()
assert [counter(), counter(), counter()] == [1, 2, 3]

callbacks = [lambda i=i: i for i in range(3)]
assert [callback() for callback in callbacks] == [0, 1, 2]

hits = []
def record(value):
    hits.append(value)
record("x")
assert hits == ["x"]
print("section 11 checks passed")

## 12. Classes

### `__init__` and `self`

A class bundles state and behaviour; an instance is one object with its own state. `self` is simply the
instance passed in as the first argument, and `__init__` is where you attach per-instance attributes. Data
engineers model connections, configs, clients and domain records this way, so fluency here is table stakes.

In [ ]:
class Account:
    def __init__(self, owner, balance=0.0):
        self.owner = owner
        self.balance = balance

account = Account("ada", 100.0)
print(account.owner, account.balance)
print(vars(account))

### Instance versus class attributes

Attributes defined in the class body are shared by all instances; attributes set on `self` are per instance.
Python resolves `instance.attr` by checking the instance dict first, then the class. Rebinding a class attribute
therefore affects every instance that has not shadowed it -- handy for defaults, dangerous for mutable state.

In [ ]:
class Account:
    currency = "USD"                 # class attribute: shared

    def __init__(self, owner, balance=0.0):
        self.owner = owner           # instance attributes
        self.balance = balance

first, second = Account("ada"), Account("grace", 50)
print(first.currency, second.currency, Account.currency)
print("currency" in vars(first))     # read through to the class, not stored

### Instance methods

An instance method takes `self` and reads or mutates that object. `deposit` is the canonical example: it
validates input, changes state, records history and returns the new balance. Keeping validation inside the
method means every caller gets the same rules -- the object protects its own invariants.

In [ ]:
class Account:
    def __init__(self, owner, balance=0.0):
        self.owner = owner
        self.balance = balance
        self.history = []

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self.balance += amount
        self.history.append(("deposit", amount))
        return self.balance

account = Account("ada", 100)
print(account.deposit(50), account.balance, account.history)

### Class methods

A `@classmethod` receives the class (`cls`), not the instance, and is the idiomatic *alternative constructor*:
`Account.from_statement(...)` reads better than a bare `Account(...)` plus setup, and it works correctly with
subclasses because `cls` is the subclass that was called.

In [ ]:
class Account:
    def __init__(self, owner, balance=0.0):
        self.owner = owner
        self.balance = balance

    @classmethod
    def from_statement(cls, owner, amounts):
        return cls(owner, sum(amounts))

account = Account.from_statement("ada", [100, 50])
print(account.balance, type(account).__name__)

### Static methods

A `@staticmethod` takes neither `self` nor `cls`; it is a plain function namespaced under the class because it
is conceptually part of it. `fee_for(amount)` is pure arithmetic, so it needs no instance state, and calling it
on the class documents that.

In [ ]:
class Account:
    def __init__(self, owner):
        self.owner = owner

    @staticmethod
    def fee_for(amount):
        return round(amount * 0.01, 2)

print(Account.fee_for(200))
print(Account("ada").fee_for(200))     # reachable via an instance too

**Check yourself.** `__init__`, instance attributes, an instance method, a classmethod constructor and a
static helper.

In [ ]:
class Account:
    currency = "USD"                      # class attribute, shared

    def __init__(self, owner, balance=0.0):
        self.owner = owner                # instance attributes
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount
        return self.balance

    @classmethod
    def from_statement(cls, owner, amounts):
        return cls(owner, sum(amounts))

    @staticmethod
    def fee_for(amount):
        return round(amount * 0.01, 2)

In [ ]:
account = Account.from_statement("ada", [100, 50])
assert account.balance == 150 and account.currency == "USD"
assert account.deposit(50) == 200
assert Account.fee_for(200) == 2.0
assert isinstance(account, Account)
print("section 12 checks passed")

## 13. Inheritance

### `super()`, overriding and the MRO

A subclass inherits attributes and can override methods; `super()` calls the next implementation in the method
resolution order, which is how you extend parent behaviour instead of replacing it. `__init__` is not called
automatically, so you must call `super().__init__(...)` yourself. The MRO is the linearised list of classes
Python searches for an attribute -- `__mro__` shows it, C3 linearisation guarantees a consistent order for
diamonds, and every class ends at `object`. "Which `save` wins?" is answered by reading the MRO left to right.

In [ ]:
class Storage:
    def __init__(self, path):
        self.path = path

    def save(self, rows):
        return f"wrote {len(rows)} rows to {self.path}"

class ParquetStorage(Storage):
    def save(self, rows):
        return super().save(rows) + " as Parquet"

print([cls.__name__ for cls in ParquetStorage.__mro__])
print(ParquetStorage("s3://lake").save([1, 2, 3]))

### Polymorphism

Calling the same method on different subclasses and getting type-appropriate behaviour is polymorphism. It is
the reason a pipeline can iterate over a list of storage backends and call `save` without an `if/elif` chain.
Duck typing means you do not even need a shared base class, only a compatible method.

In [ ]:
class Storage:
    def save(self, rows):
        return f"wrote {len(rows)} rows"

class ParquetStorage(Storage):
    def save(self, rows):
        return super().save(rows) + " as Parquet"

class DeltaStorage(Storage):
    def save(self, rows):
        return f"merged {len(rows)} rows (Delta)"

for backend in (ParquetStorage(), DeltaStorage()):
    print(backend.save([1, 2, 3]))

**Check yourself.** Inheritance, `super()`, MRO order and polymorphic dispatch.

In [ ]:
class Storage:
    def save(self, rows):
        return f"wrote {len(rows)} rows"

class ParquetStorage(Storage):
    def save(self, rows):
        return super().save(rows) + " as Parquet"

assert ParquetStorage().save([1]) == "wrote 1 rows as Parquet"
assert [c.__name__ for c in ParquetStorage.__mro__] == ["ParquetStorage", "Storage", "object"]
assert isinstance(ParquetStorage(), Storage)

backends = [Storage(), ParquetStorage()]
assert [b.save([1, 2]) for b in backends] == ["wrote 2 rows", "wrote 2 rows as Parquet"]
print("section 13 checks passed")

## 14. Dunder methods

### `__repr__` versus `__str__`

`__repr__` is the unambiguous, developer-facing representation (ideally `eval`-able) used in the REPL, in error
messages and inside containers; `__str__` is the human-friendly form used by `print` and f-strings. If you only
implement one, implement `__repr__`, because `str` falls back to it. Interviewers ask this because debuggable
objects save hours.

In [ ]:
class Metric:
    def __init__(self, name, value):
        self.name = name
        self.value = value

    def __repr__(self):
        return f"Metric({self.name!r}, {self.value!r})"

    def __str__(self):
        return f"{self.name}={self.value}"

metric = Metric("rows", 12)
print(metric)             # __str__
print(repr(metric))       # __repr__
print([metric])           # containers always use __repr__

### `__eq__`

Defining `__eq__` makes `==` compare your object field by field instead of by identity. Return `NotImplemented`
(not `False`) for unsupported types so Python can try the reflected comparison; that is the difference between a
correct operator and one that silently misbehaves.

In [ ]:
class Money:
    def __init__(self, cents, currency="USD"):
        self.cents, self.currency = int(cents), currency

    def __repr__(self):
        return f"Money({self.cents}, {self.currency!r})"

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return (self.cents, self.currency) == (other.cents, other.currency)

print(Money(100) == Money(100))
print(Money(100) == Money(101))
print(Money(100) == "100")

### `__hash__`

Objects used as dict keys or set members must be hashable, and equal objects must hash equally. Defining
`__eq__` sets `__hash__` to `None` unless you define it too -- so a class that was hashable silently becomes
unhashable. The fix is to hash the same fields you compared.

In [ ]:
class Unhashable:
    def __eq__(self, other):
        return True

try:
    hash(Unhashable())
except TypeError as exc:
    print("TypeError:", exc)

class Money:
    def __init__(self, cents):
        self.cents = cents

    def __eq__(self, other):
        return isinstance(other, Money) and self.cents == other.cents

    def __hash__(self):
        return hash(self.cents)

print(len({Money(1), Money(1), Money(2)}))

### `__add__`

Operator overloading lets domain values behave like numbers: `Money + Money` reads better than
`add_money(a, b)` and keeps currency validation in one place. Raise on incompatible operands, and remember that
`__radd__` (not shown) covers `sum([...])` and reversed operands.

In [ ]:
class Money:
    def __init__(self, cents, currency="USD"):
        self.cents, self.currency = int(cents), currency

    def __add__(self, other):
        if self.currency != other.currency:
            raise ValueError("cannot add different currencies")
        return Money(self.cents + other.cents, self.currency)

    def __repr__(self):
        return f"Money({self.cents}, {self.currency!r})"

print(Money(1250) + Money(100))

### Making a container: `__len__`, `__bool__`, `__iter__`, `__getitem__`

`__len__` makes `len(obj)` work and, when `__bool__` is absent, also drives truthiness; defining both is the
clearest contract. `__iter__` makes your object usable in `for` loops and in `list(...)`, `sum(...)` and
`any(...)`, while `__getitem__` enables indexing, slicing and (as a fallback) iteration. Implementing them is
how a `Batch` wrapper around rows starts behaving like the rows themselves.

In [ ]:
class Batch:
    def __init__(self, rows):
        self.rows = list(rows)

    def __len__(self):
        return len(self.rows)

    def __bool__(self):
        return bool(self.rows)

    def __iter__(self):
        return iter(self.rows)

    def __getitem__(self, index):
        return self.rows[index]

batch = Batch(["r1", "r2", "r3"])
print(len(batch), bool(Batch([])))
print(list(batch), batch[0], batch[-1], batch[1:])

### Also worth knowing: `__enter__`, `__exit__`, `__call__`

`__enter__`/`__exit__` make `with` work (the next sections use them), and `__call__` makes an instance callable
like a function, which is how you write stateful "functions" or tiny middleware. You do not need to memorise
every dunder; know these plus `__init__`, `__repr__`, `__eq__`, `__hash__`, `__len__` and `__iter__`.

In [ ]:
class Doubler:
    def __call__(self, value):
        return value * 2

def apply(fn, values):
    return [fn(value) for value in values]

print(apply(Doubler(), [1, 2, 3]))

**Check yourself.** `__repr__`/`__str__`, `__eq__`/`__hash__`, `__add__`, `__len__`, `__iter__` and
`__call__`.

In [ ]:
class Money:
    def __init__(self, cents):
        self.cents = int(cents)

    def __repr__(self):
        return f"Money({self.cents})"

    def __str__(self):
        return f"${self.cents / 100:.2f}"

    def __eq__(self, other):
        return isinstance(other, Money) and self.cents == other.cents

    def __hash__(self):
        return hash(self.cents)

    def __add__(self, other):
        return Money(self.cents + other.cents)

In [ ]:
assert repr(Money(1250)) == "Money(1250)"
assert str(Money(1250)) == "$12.50"
assert Money(1) + Money(2) == Money(3)
assert len({Money(1), Money(1), Money(2)}) == 2   # __eq__ + __hash__ agree
print("section 14 checks passed")

## 15. Properties and encapsulation

### Computed attributes with `@property`

`@property` lets a method be accessed like an attribute, so `temperature.fahrenheit` needs no parentheses and
callers never learn whether the value is stored or derived. Data engineers use this for derived fields
(percentage, duration, partition date) that must stay consistent with the stored state.

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self._celsius = float(celsius)

    @property
    def celsius(self):
        return self._celsius

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32

temperature = Temperature(25)
print(temperature.celsius, temperature.fahrenheit)

### Validation in a setter

A setter runs on every assignment, which is the natural place to enforce invariants: no negative timestamps,
no amounts above a limit, no unknown statuses. Callers keep using plain attribute syntax, so validation is
added without breaking the public API -- that is encapsulation in practice.

In [ ]:
class Temperature:
    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._celsius = float(value)

temperature = Temperature()
temperature.celsius = -40
print(temperature.celsius)

try:
    temperature.celsius = -300
except ValueError as exc:
    print("ValueError:", exc)

**Check yourself.** `@property` getters, computed attributes and validating setters.

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._celsius = float(value)

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32

temperature = Temperature(25)
assert temperature.fahrenheit == 77.0
temperature.celsius = -40
assert temperature.fahrenheit == -40.0
print("section 15 checks passed")

**Mechanism.** `@property` installs a descriptor on the class. Attribute lookup finds the descriptor before the instance dictionary, so reading `obj.x` calls the getter even though the syntax looks like a plain attribute.

**Worked example.** A `duration` property computes `end - start` from two stored fields. Callers write `row.duration`, and the value can never drift from the fields it is derived from.

**Common mistake.** Putting real work or I/O in a getter, so a cheap-looking attribute access hides a query. Also, a property with no setter is read-only, and assigning to it raises `AttributeError`.

## 16. Dataclasses, `NamedTuple` and `Enum`

### `@dataclass`

`@dataclass` generates `__init__`, `__repr__` and `__eq__` from annotated fields, turning a five-line
boilerplate class into three lines. Use it for records, configs and internal DTOs -- anywhere you would
otherwise hand-write a constructor and an equality method.

In [ ]:
from dataclasses import dataclass

@dataclass
class Order:
    order_id: str
    amount: float

order = Order("o-1", 42.5)
print(order)
print(order == Order("o-1", 42.5), order == Order("o-2", 42.5))

### `field(default_factory=...)` and `frozen=True`

Mutable defaults are forbidden in dataclasses for the same reason they are dangerous in functions: one list would
be shared by every instance, so `field(default_factory=list)` asks the constructor to call `list()` per instance.
`frozen=True` makes instances immutable and therefore hashable, so they can be dict keys or set members -- the
dataclass equivalent of `NamedTuple`, and the safe default for value objects such as keys and partition
descriptors.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Cart:
    items: list = field(default_factory=list)

@dataclass(frozen=True)
class Partition:
    table: str
    day: str

first, second = Cart(), Cart()
first.items.append("a")
partition = Partition("orders", "2026-10-04")
print(first.items, second.items)                 # no shared list
print(len({partition, Partition("orders", "2026-10-04")}))

### `NamedTuple` and `Enum`

`typing.NamedTuple` gives you a tuple that is also a class: it unpacks positionally like a row, has named fields
for readability, and is immutable and hashable -- choose it when the value genuinely is a tuple with names.
`Enum` replaces magic strings with a closed set of named members, so `status is Status.SHIPPED` is typo-proof for
tooling, and iteration over the enum gives you a canonical list for validation.

In [ ]:
from enum import Enum
from typing import NamedTuple

class Summary(NamedTuple):
    partition: str
    rows: int

class Status(Enum):
    PLACED = "placed"
    SHIPPED = "shipped"

summary = Summary("ds=2026-10-04", 120)
print(summary.rows, summary[0], summary.partition)
print([status.value for status in Status], Status("shipped") is Status.SHIPPED)

**Check yourself.** Dataclasses, `default_factory`, `frozen=True`, `NamedTuple` and `Enum`. `Status` and
`Summary` are reused from the example above.

In [ ]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class Order:
    order_id: str
    amount: float
    tags: tuple = ()

@dataclass
class Cart:
    items: list = field(default_factory=list)

In [ ]:
order = Order("o-1", 42.5, tags=("priority",))
assert order == Order("o-1", 42.5, tags=("priority",))
assert len({order, Order("o-1", 42.5, tags=("priority",))}) == 1  # frozen => hashable
assert Cart().items == []
assert Cart().items is not Cart().items                          # fresh list per instance
assert Status("placed") is Status.PLACED
assert Summary("ds=1", 5).rows == 5
print("section 16 checks passed")

## 17. Exceptions

### `try` / `except` / `else` / `finally`

`try` runs the risky code, `except` handles a specific failure, `else` runs only when no exception occurred, and
`finally` always runs. Catch the narrowest exception you can actually handle: a bare `except Exception: pass`
hides data bugs. `else` is rare but useful -- it keeps the `try` block to the single operation you are guarding.

In [ ]:
def parse_amount(text):
    return float(text)

for candidate in ["10.5", "not-a-number"]:
    try:
        amount = parse_amount(candidate)
    except ValueError as exc:
        print(f"{candidate!r} -> ValueError: {exc}")
    else:
        print(f"{candidate!r} -> parsed {amount}")
    finally:
        print("  done")

### Custom errors and chaining with `raise ... from`

Define a base exception for your package and subclass it for specific failures: callers can then catch the whole
family or one precise case, and you can attach structured context (the partition name) to the error. `raise New
from original` records the causal chain in `__cause__`, so the traceback says "the above exception was the direct
cause". `raise` with no argument re-raises the active exception. Never swallow the original context; future you
is the one debugging at 3am.

In [ ]:
class DataQualityError(Exception):
    """Base class for pipeline data-quality failures."""

class EmptyPartitionError(DataQualityError):
    def __init__(self, partition):
        super().__init__(f"partition {partition} is empty")
        self.partition = partition

try:
    raise EmptyPartitionError("ds=2026-10-03")
except DataQualityError as exc:
    print(type(exc).__name__, "->", exc, "| partition:", exc.partition)

try:
    raise RuntimeError("config unavailable") from OSError("no such file")
except RuntimeError as exc:
    print(type(exc).__name__, "-> cause:", type(exc.__cause__).__name__)

**Check yourself.** `else`/`finally`, catching a hierarchy, and exception chaining.

In [ ]:
class DataQualityError(Exception):
    pass

class EmptyPartitionError(DataQualityError):
    def __init__(self, partition):
        super().__init__(f"partition {partition} is empty")
        self.partition = partition

assert issubclass(EmptyPartitionError, DataQualityError)

In [ ]:
steps = []
try:
    raise EmptyPartitionError("ds=2026-10-03")
except DataQualityError as exc:
    steps.append(f"caught {exc.partition}")
else:
    steps.append("no error")
finally:
    steps.append("cleanup")

assert steps == ["caught ds=2026-10-03", "cleanup"]

In [ ]:
try:
    raise RuntimeError("outer") from ValueError("inner")
except RuntimeError as exc:
    assert isinstance(exc.__cause__, ValueError)  # chained for the stack trace
print("section 17 checks passed")

## 18. Context managers

### `with` guarantees teardown

A context manager wraps a block with setup and guaranteed cleanup, even when the block raises. Connecting to a
warehouse, opening a transaction, taking a lock and creating a temporary file are all `with` jobs. The guarantee
is the whole point: no leaked handles when an exception unwinds the stack.

In [ ]:
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "part.txt"
    path.write_text("a\nb\n", encoding="utf-8")
    print(path.read_text(encoding="utf-8").splitlines())

print("temp dir removed:", not path.exists())

### Writing one with `__enter__` / `__exit__`

`__enter__` returns the value bound by `as`; `__exit__(exc_type, exc, tb)` runs on the way out and can inspect the
exception. Returning `True` swallows the exception and returning `False` (or `None`) lets it propagate -- for a
transaction manager you want `False` plus a rollback, which is what makes partial writes safe when anything in
the block fails.

In [ ]:
class Transaction:
    def __init__(self, name):
        self.name = name
        self.log = []

    def __enter__(self):
        self.log.append(f"begin {self.name}")
        return self

    def __exit__(self, exc_type, exc, tb):
        self.log.append("rollback" if exc_type else f"commit {self.name}")
        return False

tx = Transaction("load-orders")
with tx:
    tx.log.append("insert rows")

failing = Transaction("bad-load")
try:
    with failing:
        raise RuntimeError("warehouse down")
except RuntimeError as exc:
    print("propagated:", exc)
print(tx.log, failing.log)

### `contextlib.contextmanager`

`@contextmanager` turns a generator into a context manager: everything before `yield` is setup, everything after
is teardown, and a `try/finally` around the yield makes the cleanup exception-safe. It is the fastest way to
build timers, log scopes and temporary-resource helpers.

In [ ]:
from contextlib import contextmanager
import time

@contextmanager
def timer():
    started = time.perf_counter()
    try:
        yield
    finally:
        print(f"elapsed {time.perf_counter() - started:.4f}s")

with timer():
    sum(range(1000))

**Check yourself.** `with`, a class-based manager and a generator-based manager. The `Transaction` class
is reused from the example above, which is safe because notebooks execute top to bottom.

In [ ]:
committed = Transaction("load")
with committed:
    committed.log.append("insert")
assert committed.log == ["begin load", "insert", "commit load"]

In [ ]:
rolled_back = Transaction("load")
try:
    with rolled_back:
        raise ValueError("bad row")
except ValueError:
    pass
assert rolled_back.log[-1] == "rollback"   # __exit__ saw the exception

In [ ]:
from contextlib import contextmanager

@contextmanager
def tagging(tag):
    yield tag.upper()

with tagging("ok") as value:
    assert value == "OK"
print("section 18 checks passed")

## 19. Iterators and generators

### Iterable versus iterator

An *iterable* can produce an iterator (`__iter__`); an *iterator* remembers position and returns the next value
(`__next__`) until it raises `StopIteration`. Lists, dicts, files and sets are iterables; `iter(x)` gives the
iterator. A generator is a compact way to write an iterator: a function containing `yield` suspends its frame at
each value, so local state is preserved and memory stays constant regardless of input size. Knowing the
iterable/iterator distinction is a very common screening question.

In [ ]:
rows = ["r1", "r2"]
row_iterator = iter(rows)
print(type(row_iterator).__name__, next(row_iterator), next(row_iterator))
try:
    next(row_iterator)
except StopIteration:
    print("exhausted")

def countdown(start):
    while start > 0:
        yield start
        start -= 1

generator = countdown(3)
print(generator)
print(next(generator), next(generator), next(generator))

### A lazy pipeline over records

Generators compose: parse yields dicts, filter yields a subset, and batching yields lists. No stage materialises
the whole dataset, so you can process a file larger than memory. This is the shape interviewers want when they
ask you to "stream" a transformation.

In [ ]:
def parse_orders(lines):
    for line in lines:
        order_id, amount = line.split(",")
        yield {"order_id": order_id, "amount": float(amount)}

def only_positive(orders):
    for order in orders:
        if order["amount"] > 0:
            yield order

raw = ["o1,10.5", "o2,-3", "o3,7"]
print(list(only_positive(parse_orders(raw))))

### Generators versus lists for memory

A list comprehension builds every element up front; a generator expression builds them one at a time. For a
million rows the difference is megabytes versus bytes, plus lower peak memory in the pipeline. The cost is that
a generator is single-use and you cannot index it or take `len()`.

In [ ]:
import sys

as_list = [n * n for n in range(10_000)]
as_generator = (n * n for n in range(10_000))

print("list     :", sys.getsizeof(as_list), "bytes")
print("generator:", sys.getsizeof(as_generator), "bytes")
print("sum equal:", sum(as_list) == sum(as_generator))

**Check yourself.** Iterators, `yield`, lazy composition and single-use semantics.

In [ ]:
def countdown(start):
    while start > 0:
        yield start
        start -= 1

assert list(countdown(3)) == [3, 2, 1]
iterator = iter(countdown(2))
assert next(iterator) == 2 and list(iterator) == [1]

def parse_orders(lines):
    for line in lines:
        order_id, amount = line.split(",")
        yield {"order_id": order_id, "amount": float(amount)}

parsed = parse_orders(["o1,1.5"])
assert next(parsed) == {"order_id": "o1", "amount": 1.5}

lazy = (n for n in range(3))
assert next(lazy) == 0
assert list(lazy) == [1, 2]
assert list(lazy) == []            # single use
print("section 19 checks passed")

## 20. Decorators

### A function decorator

A decorator is a callable that takes a function and returns a replacement, so you can wrap cross-cutting
behaviour (timing, logging, retrying, caching) around it. The `*args, **kwargs` wrapper forwards anything, and
`functools.wraps` copies the original metadata so the decorated function still looks like itself.

In [ ]:
import functools

def traced(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return fn(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

@traced
def add(left, right):
    return left + right

add(1, 2)
add(3, 4)
print(add(5, 6), add.calls, add.__name__)

### `functools.lru_cache`

`lru_cache` memoises results keyed by the arguments, turning exponential recursion into linear work and making
repeated lookups free. Requirements: arguments must be hashable, and the function must be pure. It is a great
one-line answer for "how would you avoid recomputing this lookup?".

In [ ]:
import functools

@functools.lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(30))
print(fib.cache_info())

### A retry decorator

Transient failures -- a dropped socket, a throttled API -- are normal in ingestion, so a `retry(times=...)`
decorator that catches one narrow exception and re-raises on the final attempt is standard practice. Keep the
exception type explicit; retrying every exception turns data bugs into slow data bugs.

In [ ]:
import functools

def retry(times=3):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return fn(*args, **kwargs)
                except ConnectionError:
                    wrapper.attempts = attempt
                    if attempt == times:
                        raise
        return wrapper
    return decorator

state = {"fails": 2}

@retry(times=5)
def flaky():
    if state["fails"] > 0:
        state["fails"] -= 1
        raise ConnectionError("temporarily unavailable")
    return "ok"

print(flaky(), flaky.attempts)

**Check yourself.** A wrapping decorator, `functools.wraps`, memoisation and retry semantics. `traced` and
`fib` are reused from the examples above.

In [ ]:
calls = {"n": 0}

@traced
def increment():
    calls["n"] += 1
    return calls["n"]

assert increment() == 1 and increment.calls == 1
assert increment.__name__ == "increment"

@functools.lru_cache(maxsize=None)
def square(n):
    return n * n

assert square(10) == 100
assert square(10) == 100 and square.cache_info().hits == 1
assert fib(20) == 6765
print("section 20 checks passed")

## 21. Type hints

### Annotations document intent

Annotations attach types to parameters, return values and variables: `def average(values: list[float]) ->
float`. Python stores them in `__annotations__` and otherwise ignores them at runtime. They are documentation
that tooling can read, which is why they belong on public functions and data models.

`Optional[T]` (or `T | None`) says "may be missing", `Iterator[T]` says "lazy stream of T", and `Iterable[T]` is
the permissive input you usually want for a parameter. Being explicit about optionality forces callers to handle
the `None` case, which is the most common source of pipeline crashes.

In [ ]:
from typing import Iterator, Optional

def row_count(rows: list[dict]) -> int:
    return len(rows)

def average(values: list[float]) -> Optional[float]:
    return sum(values) / len(values) if values else None

def running_total(values: list[float]) -> Iterator[float]:
    total = 0.0
    for value in values:
        total += value
        yield total

print(row_count([{"id": 1}]), row_count.__annotations__)
print(average([]), average([1, 2, 3]), list(running_total([1, 2, 3])))

### Hints are not enforcement

Nothing stops you from passing a string where an `int` is annotated; static checkers such as mypy catch it, the
interpreter does not. For data crossing a trust boundary, add real runtime validation (`isinstance`, a schema
library, or explicit tests). Saying this out loud shows you understand the difference between hints and
contracts.

In [ ]:
def double(value: int) -> int:
    return value * 2

print(double(21))
print(double("21"))        # no runtime error: hints are metadata only

**Check yourself.** Annotations, optional returns and lazy iterators.

In [ ]:
from typing import Iterator, Optional

def average(values: list[float]) -> Optional[float]:
    return sum(values) / len(values) if values else None

def running_total(values: list[float]) -> Iterator[float]:
    total = 0.0
    for value in values:
        total += value
        yield total

assert average([1, 2, 3]) == 2.0
assert average([]) is None
assert list(running_total([1, 2, 3])) == [1.0, 3.0, 6.0]
assert row_count.__annotations__["return"] is int
print("section 21 checks passed")

## 22. Dates and times

### Durations and timezone-aware datetimes

`datetime` is a point in time, `date` is a calendar day, and `timedelta` is a duration you can add and subtract.
Always compute durations as timedeltas and convert to seconds explicitly; string arithmetic on timestamps is a bug
factory, and pipeline scheduling, SLA checks and retention windows are all built from these two types.

A naive datetime has no timezone and silently means different instants in different places, so store UTC, attach
`timezone.utc`, and convert with `astimezone` only for display -- `zoneinfo` gives you DST-correct rules. Note
that `datetime.utcnow()` is deprecated precisely because it returns a naive value.

In [ ]:
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

started = datetime(2026, 10, 4, 12, 0, tzinfo=timezone.utc)
finished = started + timedelta(hours=2, minutes=30)
print(started.isoformat(), "->", finished.isoformat())
print("duration seconds:", (finished - started).total_seconds())

as_new_york = finished.astimezone(ZoneInfo("America/New_York"))
print("new york:", as_new_york.isoformat(), "| offset:", as_new_york.utcoffset())

### ISO parsing and formatting

`datetime.fromisoformat` parses ISO-8601 strings (the format every API and warehouse returns) and
`.isoformat()` writes them back. Keep the timezone in the string when you serialise; dropping it is how
pipelines accidentally shift timestamps by hours.

In [ ]:
from datetime import datetime, timezone

parsed = datetime.fromisoformat("2026-10-04T12:00:00+00:00")
print(parsed, parsed.tzinfo is not None)
print(parsed == datetime(2026, 10, 4, 12, 0, tzinfo=timezone.utc))
print(parsed.date().isoformat(), parsed.isoformat())

**Check yourself.** Durations, timezone-aware conversion and ISO round-tripping.

In [ ]:
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

started = datetime(2026, 10, 4, 12, 0, tzinfo=timezone.utc)
finished = started + timedelta(hours=2, minutes=30)
assert (finished - started).total_seconds() == 9000

as_new_york = finished.astimezone(ZoneInfo("America/New_York"))
assert as_new_york.hour == 10 and as_new_york.utcoffset() == timedelta(hours=-4)

parsed = datetime.fromisoformat("2026-10-04T12:00:00+00:00")
assert parsed == started and parsed.tzinfo is not None
print("section 22 checks passed")

## 23. Files, `pathlib` and JSON

### `Path` basics

`pathlib.Path` is the modern file API: `/` joins, `.suffix` and `.name` inspect, `.read_text`/`.write_text` read
and write, and it works the same on Windows and POSIX. Data code is full of paths, so fluency here is expected;
`.stem`, `.parent` and `.with_suffix` are the other methods worth knowing.

In [ ]:
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "orders.ndjson"
    path.write_text("line\n", encoding="utf-8")
    print(path.name, path.suffix, path.stem, path.parent == Path(tmp))
    print(path.exists(), path.read_text(encoding="utf-8").strip())

### Writing and reading NDJSON

NDJSON (newline-delimited JSON) is the lingua franca of ingestion: one self-contained JSON object per line,
streamable, append-friendly and impossible to corrupt with a partial last line. Open with an explicit encoding
and write one `json.dumps` per row; read it back by iterating lines and `json.loads`-ing each, ideally combining
that with a generator so the file never has to fit in memory.

In [ ]:
import json
import tempfile
from pathlib import Path

orders = [{"order_id": "o1", "amount": 10.5}, {"order_id": "o2", "amount": 7.0}]

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "orders.ndjson"
    with path.open("w", encoding="utf-8") as handle:
        for order in orders:
            handle.write(json.dumps(order) + "\n")
    loaded = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
    print(path.read_text(encoding="utf-8").strip().replace("\n", " | "))
    print(loaded == orders, "total:", sum(row["amount"] for row in loaded))

### `json.dumps` and `json.loads`

`json.dumps` serialises to a string (`json.dump` writes to a handle), `json.loads` parses a string
(`json.load` reads a handle). Useful flags: `sort_keys=True` for stable diffs, `indent=2` for humans and
`default=str` to avoid crashing on datetimes. JSON has no datetime type -- decide a wire format and stick to
it.

In [ ]:
import json
from datetime import date

print(json.dumps({"b": 1, "a": 2}, sort_keys=True))
print(json.loads('{"n": 3}')["n"])
print(json.dumps({"day": date(2026, 10, 4)}, default=str))

**Check yourself.** `Path` attributes, an NDJSON round-trip and `json` string helpers.

In [ ]:
import json
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "orders.ndjson"
    orders = [{"order_id": "o1", "amount": 10.5}, {"order_id": "o2", "amount": 7.0}]
    with path.open("w", encoding="utf-8") as handle:
        for order in orders:
            handle.write(json.dumps(order) + "\n")

    loaded = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
    assert loaded == orders
    assert path.suffix == ".ndjson" and path.name == "orders.ndjson"
    assert sum(row["amount"] for row in loaded) == 17.5
assert json.loads(json.dumps({"a": 1})) == {"a": 1}
print("section 23 checks passed")

## 24. Concurrency

### The GIL, threads, processes and asyncio

The global interpreter lock lets only one thread execute Python bytecode at a time, so threads speed up
**I/O-bound** work (waiting on network or disk releases the lock) but not CPU-bound work. Use
`ProcessPoolExecutor` to bypass the GIL for CPU-bound Python, and `asyncio` when the whole call chain is async.
Spark or another distributed engine is the answer when the data is genuinely large.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def fetch_partition(partition):
    time.sleep(0.01)                    # pretend this is a network call
    return f"rows for {partition}"

partitions = [f"ds=2026-10-{day:02d}" for day in range(1, 6)]
print(partitions[0], len(partitions))

### A deterministic thread pool

`ThreadPoolExecutor.map` applies a function across an iterable using a bounded pool and returns results in input
order. Five sleeps of 10 ms overlap, so the wall clock is far below the ~50 ms a sequential loop would take.
The demo is deterministic: fixed inputs, no randomness, assertion on the results rather than on timing.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def fetch_partition(partition):
    time.sleep(0.01)
    return f"rows for {partition}"

partitions = [f"ds=2026-10-{day:02d}" for day in range(1, 6)]
started = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as pool:
    results = list(pool.map(fetch_partition, partitions))
elapsed = time.perf_counter() - started

print(results)
print(f"5 mocked fetches in {elapsed * 1000:.1f} ms (sequential would be ~50 ms)")

An `asyncio` version of the same idea, shown to read rather than run because a notebook already owns an event
loop:

```python
import asyncio

async def fetch(partition):
    await asyncio.sleep(0.01)
    return f"rows for {partition}"

async def main():
    return await asyncio.gather(*(fetch(p) for p in partitions))
```

Rule of thumb: threads for blocking I/O libraries that are not async-aware, asyncio when the whole chain is
async, processes for CPU-heavy Python, and a distributed engine for large-scale parallelism.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def fetch_partition(partition):
    time.sleep(0.01)
    return f"rows for {partition}"

partitions = [f"ds=2026-10-{day:02d}" for day in range(1, 6)]
with ThreadPoolExecutor(max_workers=5) as pool:
    results = list(pool.map(fetch_partition, partitions))

assert results[0] == "rows for ds=2026-10-01" and len(results) == 5
assert all(result.startswith("rows for ") for result in results)
print("section 24 checks passed")

## 25. Common pitfalls checklist

### Float equality

`0.1 + 0.2 != 0.3` because binary floats cannot represent those decimals exactly. Compare with `math.isclose`
and an explicit tolerance, and never use `==` on floats in validation or tests unless you control the exact bit
pattern.

In [ ]:
import math

print(0.1 + 0.2 == 0.3)
print(math.isclose(0.1 + 0.2, 0.3, rel_tol=1e-9))
print(math.isclose(1e-12, 0.0, abs_tol=1e-9))

### Building strings in a loop

`total += part` copies the whole accumulated string every iteration, which is quadratic. Collect pieces in a
list and `"".join(...)` once, or use an `io.StringIO` for genuinely large output. Interviewers often ask for
the complexity of both approaches.

In [ ]:
parts = [str(value) for value in range(5)]
joined = "-".join(parts)
print(joined)

buffer = []
for value in range(5):
    buffer.append(str(value))
print("|".join(buffer))

### Identity, equality and shadowed builtins

`==` compares values through `__eq__`, while `is` compares identity (the same object in memory), so use `is` only
for singletons: `None`, `True`, `False` and sentinels. Small integers and interned strings sometimes make `is`
"work" by accident, which is exactly why relying on it is a bug waiting to happen. Similarly, naming a variable
`list`, `dict`, `id`, `sum` or `type` hides the builtin for the rest of the scope -- use `rows`, `mapping`,
`row_id` or `total` instead.

In [ ]:
left, right = [1, 2], [1, 2]
print(left == right, left is right, [] is [])
print(None is None)

rows = [1, 2, 3]          # not `list`
total = sum(rows)         # not `sum`
print(total, max(rows), min(rows), list("ab"), dict(a=1))

### Mutable defaults and late binding

Two classics to recite: default arguments are evaluated once at definition time (so a default list is shared
across calls), and closures capture variables by reference (so lambdas built in a loop all see the last value).
The fixes are the `None` sentinel and binding the value with `lambda i=i: i`.

In [ ]:
def append_bad(item, bucket=[]):
    bucket.append(item)
    return bucket

print(append_bad("a"), append_bad("b"))     # shared default

def append_good(item, bucket=None):
    bucket = [] if bucket is None else bucket
    bucket.append(item)
    return bucket

print(append_good("a"), append_good("b"))
print([callback() for callback in [lambda: i for i in range(3)]])
print([callback() for callback in [lambda i=i: i for i in range(3)]])

**Check yourself.** Float comparison, join over concatenation, identity versus equality, and the two
classic mutation traps.

In [ ]:
import math

assert not math.isclose(0.1 + 0.2, 0.3, rel_tol=0, abs_tol=0)
assert math.isclose(0.1 + 0.2, 0.3, rel_tol=1e-9)
assert "-".join(str(i) for i in range(3)) == "0-1-2"
assert [1, 2] == [1, 2] and [1, 2] is not [1, 2]
assert None is None and True is True

def append_good(item, bucket=None):
    bucket = [] if bucket is None else bucket
    bucket.append(item)
    return bucket

assert append_good("a") == ["a"] and append_good("b") == ["b"]
assert [callback() for callback in [lambda: i for i in range(3)]] == [2, 2, 2]
print("section 25 checks passed")

## Interview questions

Answer each one out loud before reading the model answer. The answers are deliberately short; your job is to add
the concrete example from the sections above.

**Q.** What is the difference between a list and a tuple, and when do you prefer each?

**A.** Lists are mutable, variable-length and used for homogeneous sequences you will grow or change, such as a
batch of rows. Tuples are immutable, so they are hashable and can be dict keys or set members, and they describe
fixed-shape records such as `(customer_id, ds)`. Prefer a tuple when the arity is part of the contract or when
you need a composite key, and a list when you need to append. In a query result, tuples signal "row shape" and
lists signal "collection".

**Q.** Explain the mutable default argument trap.

**A.** Default arguments are evaluated once, when the `def` statement executes, so `def f(item, bucket=[])`
creates one list shared by every call. Calling `f("a")` then `f("b")` returns `["a", "b"]`, which surprises
everyone the first time. The fix is `bucket=None` plus `bucket = [] if bucket is None else bucket` inside the
function. Dataclasses have the analogous rule and require `field(default_factory=list)`.

**Q.** What is the difference between `__str__` and `__repr__`?

**A.** `__repr__` is the unambiguous developer representation, ideally something like `Money(1250, 'USD')` that
could be re-evaluated, and it is what the REPL, debuggers and containers show. `__str__` is the human-readable
form used by `print` and f-strings. `__str__` falls back to `__repr__`, so if you implement only one, implement
`__repr__`. A good `__repr__` is one of the highest-value debugging investments in a codebase.

**Q.** Why must `__eq__` and `__hash__` be consistent?

**A.** Dicts and sets find objects by hash first and then confirm with `==`, so equal objects must produce equal
hashes. If they do not, a set can hold two "equal" elements or a dict lookup can miss a key that is present.
Defining `__eq__` sets `__hash__` to `None`, silently making the class unhashable, so you must define `__hash__`
over the same fields. `@dataclass(frozen=True)` gives you both for free.

**Q.** When would you use `@classmethod` over `@staticmethod`?

**A.** A `@classmethod` receives `cls` and is the idiomatic alternative constructor:
`Account.from_statement(owner, amounts)` reads better than a bare constructor plus setup, and it respects
subclasses because `cls` is the called class. A `@staticmethod` receives neither `self` nor `cls`; it is a
namespaced pure helper such as `Account.fee_for(amount)` that belongs to the class conceptually but needs no
state. If it touches the class, use classmethod; if it touches nothing, staticmethod is fine.

**Q.** How do generators reduce memory usage?

**A.** A generator produces values lazily, keeping only the current item and the suspended frame instead of
materialising the whole sequence. That is how you stream a file larger than RAM: parse one line, yield one
record, let the consumer write it, and repeat. The trade-offs are that a generator is single-use, cannot be
indexed or measured with `len()`, and holds the function frame open until exhausted. Composing several
generators gives you a streaming pipeline with bounded memory.

**Q.** What does a context manager guarantee?

**A.** That the teardown code runs when the block exits, whether it finished normally or raised an exception.
That is why connections, transactions, locks and temporary files use `with`: no leaked handles even when an
error unwinds the stack. `__exit__` receives the exception type and value, and returning a truthy value would
swallow the exception -- transaction managers return `False` so failures propagate after a rollback.
`contextlib.contextmanager` builds one from a generator with a `try/finally` around the `yield`.

**Q.** What does `functools.wraps` do, and why does it matter?

**A.** It copies `__name__`, `__doc__`, `__module__`, `__qualname__`, `__dict__` and `__wrapped__` from the
wrapped function onto the wrapper. Without it, a decorated function reports the wrapper's name, which breaks log
lines, documentation, tracebacks and tools that introspect signatures. With it, debuggers and `inspect.signature`
still see the original. It is one line and should be in every decorator you write.

**Q.** Explain the GIL and its practical consequence.

**A.** The global interpreter lock allows only one thread to execute Python bytecode at a time, so threads do not
give you parallel CPU work in pure Python. They still help I/O-bound code because blocking calls release the
lock while they wait, which is why a `ThreadPoolExecutor` speeds up many API or database calls. For CPU-bound
work use `ProcessPoolExecutor`, a C extension that releases the GIL, or a distributed engine such as Spark.
Asyncio is the single-threaded alternative for high-concurrency network I/O.

**Q.** What is the difference between `is` and `==`?

**A.** `==` compares values by calling `__eq__`, so two distinct lists with the same contents are equal. `is`
compares identity, meaning the very same object in memory, and is only appropriate for singletons such as
`None`, `True`, `False` and sentinels. CPython caches small integers and interns some strings, so `is` sometimes
appears to work on them -- that is an implementation detail and a latent bug. Write `if value is None`, never
`if value == None`.

**Q.** Why are type hints not enforcement?

**A.** Annotations are stored in `__annotations__` and ignored at runtime, so `double("21")` runs happily even
though the parameter is annotated `int`. Their value is static analysis with mypy or pyright, editor completion
and documentation that cannot drift from the code. To actually enforce types at a boundary -- an API payload, a
config file, a message from Kafka -- add runtime validation with `isinstance`, a schema library, or explicit
tests. Hints describe intent; they do not create a contract.

**Q.** How do you make an instance immutable?

**A.** The simplest options are `@dataclass(frozen=True)`, which blocks attribute assignment and generates
`__hash__`, or `typing.NamedTuple`, which is immutable by construction. You can also define `__setattr__` to
raise, or expose read-only `@property` attributes backed by private state. Immutability makes values safely
hashable, shareable across threads and safe as dict keys. Remember that `frozen=True` is shallow: a mutable
field such as a list inside a frozen dataclass can still be mutated.

**Q.** What is the difference between a shallow copy and a deep copy?

**A.** A shallow copy (`list(x)`, `x.copy()`, `copy.copy`) creates a new outer container but the inner objects
are shared, so mutating a nested list is visible through both copies. `copy.deepcopy` recursively copies
everything, producing a fully independent structure at the cost of time and memory. Use deep copies for nested
config or record structures you intend to mutate, and shallow copies when you only replace top-level entries.
This is also why `dict(config)` does not protect a nested `config["credentials"]`.

**Q.** `d[key]` versus `d.get(key)` -- and when do `setdefault` or `defaultdict` help?

**A.** `d[key]` raises `KeyError` on a missing key, which is right when the key is required and absence is a
bug; `d.get(key, default)` returns a fallback, which is right for optional configuration. `setdefault(key, [])`
returns the existing list or inserts and returns a new one, turning a group-by into a two-line loop.
`collections.defaultdict(list)` expresses the same intent more cleanly, and `Counter` handles tallying with
`most_common`. All of them rely on average O(1) dict lookup, which is why grouping is cheap.

**Q.** Explain late binding in closures.

**A.** A closure captures the variable, not its value, so a function or lambda looks it up when called. Building
`[lambda: i for i in range(3)]` gives three callables that all return `2`, because the loop finished before any
of them ran. Bind the current value with a default argument, `lambda i=i: i`, or use `functools.partial`. The
same mechanism powers useful patterns such as counters that use `nonlocal`, so learn both the power and the
trap.

**Q.** What does `for ... else` do?

**A.** The `else` clause on a loop runs only if the loop completed without hitting `break`. It is the clean way
to express "search and report if not found" without a boolean flag, for example scanning partitions and
branching when no match exists. Many engineers have never seen it, so it is worth knowing -- and worth using
sparingly, because an explicit flag is sometimes clearer to a reviewer.

**Q.** Why prefer `Decimal` over `float` for money?

**A.** Binary floating point cannot represent most decimal fractions exactly, so cents drift when you sum
thousands of amounts. `decimal.Decimal` performs exact decimal arithmetic with explicit rounding and is the
standard-library answer for currency and any value that must reconcile to the cent. Construct it from strings
(`Decimal("19.99")`) because `Decimal(19.99)` inherits the binary error. Floats remain correct for rates,
measurements and statistics where small error is acceptable.

**Q.** Why does `@dataclass` need `field(default_factory=list)`?

**A.** A dataclass turns a default into a value shared by the `__init__` signature, which for a mutable default
reproduces the shared-list bug. `field(default_factory=list)` tells the generated constructor to call `list()`
for every new instance, so each object gets its own container. The same rule covers `dict`, `set` and any
mutable default. Use the factory for every mutable field and a plain default only for immutables.

**Q.** What does `raise ... from` do?

**A.** It sets the new exception's `__cause__` to the original, so the traceback shows "the above exception was
the direct cause of the following exception". That preserves the debugging context while letting you raise an
exception that fits your API, such as wrapping an `OSError` in a `RuntimeError` that names the missing config
file. A bare `raise` inside an `except` block re-raises the active exception unchanged. Never discard the
original error silently.

**Q.** How does the LEGB rule work, and when do you need `global` or `nonlocal`?

**A.** Name lookup checks the Local scope, then any Enclosing function scope, then the Global module scope, then
Built-ins. Reading an enclosing variable needs no keyword, but rebinding it does: `nonlocal` for an enclosing
function scope and `global` for a module-level name. Without the keyword, `count += 1` creates a new local and
raises `UnboundLocalError` if it is also read first. Prefer returning values or using a mutable container; module
state breaks test isolation and parallel runs.

## Self-assessment checklist

Tick each item only when you can explain it *and* write a small example without looking things up.

- [ ] I can name three differences between a list and a tuple, and say which one is hashable.
- [ ] I can explain why `0.1 + 0.2 != 0.3` and what `math.isclose` does instead.
- [ ] I can write a dict comprehension that filters and transforms in one line.
- [ ] I can group rows by key with `setdefault` and again with `defaultdict`.
- [ ] I can explain aliasing, and the difference between `copy.copy` and `copy.deepcopy`.
- [ ] I can describe what `__repr__`, `__eq__` and `__hash__` each control.
- [ ] I can write a class with `__init__`, a classmethod constructor and a static helper.
- [ ] I can trace the MRO of a small inheritance hierarchy.
- [ ] I can add a validating `@property` setter without breaking the attribute API.
- [ ] I can choose between `@dataclass`, `NamedTuple` and `Enum` for a given value type.
- [ ] I can build a custom exception hierarchy and chain with `raise ... from`.
- [ ] I can write a context manager both as a class and with `@contextmanager`.
- [ ] I can turn a list-based function into a lazy generator pipeline.
- [ ] I can write a decorator that uses `functools.wraps`, and explain what `lru_cache` stores.
- [ ] I can say what type hints do and do not enforce at runtime.
- [ ] I can convert a UTC timestamp to another zone with `zoneinfo` and explain why UTC is stored.
- [ ] I can round-trip NDJSON with `pathlib` and `json` without loading it all into memory.
- [ ] I can explain the GIL and pick threads, processes or asyncio for a given workload.
- [ ] I can list the mutable-default, late-binding, `is`/`==` and float-equality pitfalls from memory.

If any box is unticked, jump back to that section and re-run its **Check yourself** cell.

## Keep practising

* `02_python_coding_problems.ipynb` -- timed coding problems with self-checking tests.
* `03_sql_interview_questions.ipynb` -- SQL window functions against a SQLite sandbox.
* `04_pyspark_concepts.ipynb` -- PySpark concepts, guarded so it runs without Spark installed.
* `05_interview_questions.ipynb` -- a mixed question bank for mock interviews.

---

# Practice notebook: Python coding problems

The cells below are from `notebooks/02_python_coding_problems.ipynb` and are merged here so this Colab notebook is self-contained. Run them top to bottom; assertions confirm the expected behavior.

# Python Coding Problems for Data Engineers

A hands-on workbook of the Python that actually gets tested in data-engineering interviews: streaming a file bigger than RAM, deduplicating at-least-once deliveries, hash joins, schema drift, sessionization, rolling metrics, and the small utility classes (LRU cache, retry with backoff) that show up in production pipelines.

It covers **14 problems**. Every problem is broken into small cells: an explanation, the solution, then separate tests for the happy path, the edge cases, and the exact stated output.

## How to use this notebook

- **Run cells top to bottom** (`Kernel -> Restart & Run All`). Every problem is: a statement, small solution cells, then several test cells and a `**Check yourself**` cell.
- **Every solution is followed by self-checking assertions.** A cell that finishes without an exception is proof the solution is correct on the documented cases; the printed output shows the real results.
- **Standard library only.** No pandas, no PySpark, no magics and no shell escapes. Everything is deterministic and re-runnable.
- **Try before you look.** Cover the solution cells, attempt the problem, then reveal them and run the tests.
- **Notes cells** give the complexity, the edge cases, and 1-2 interview follow-ups you should be able to answer out loud.

## The problems

1. Deduplicate, count, and group without pandas
2. Multi-key sorting with key functions
3. Top-K and top-N-per-group with `heapq`
4. Rolling windows: max sum, moving average, percentile
5. Two pointers and prefix sums
6. Merge overlapping intervals
7. Binary search over sorted partitions
8. Generator-based streaming of a large file
9. Parse CSV, JSON, and NDJSON
10. Flatten nested JSON into rows
11. In-memory hash join of two datasets
12. Schema drift and late-arriving dedupe
13. Sessionize clickstream events
14. Production utilities: LRU cache and retry with backoff

The notebook closes with `## Interview questions`: 12 likely questions with concise model answers, plus runnable code that verifies several of those answers.

### Kernel sanity check

Everything here uses the standard library only: no pandas, no numpy, no Spark. This first cell proves the kernel is alive and shows which Python you are on; interviewers expect you to know the version you target, because syntax such as `str | None` and `itertools.batched` depends on it.

In [ ]:
import platform
import sys

print(f"Python {sys.version.split()[0]} on {platform.system()}")
print("Standard library only: no pandas / pyspark / shell magics required.")

## Problem 1 - Deduplicate, count, and group without pandas

**Statement.** Implement the workhorses of in-memory aggregation with the standard library only:

1. `dedupe_preserving_order(records, key)` - keep the **first** occurrence of each key, in input order.
2. `frequency_map(items)` - the equivalent of `COUNT(*) GROUP BY item`.
3. `group_by(records, key)` - bucket rows per key, preserving first-seen group order.
4. `aggregate(rows)` - SQL-style `COUNT`, `COUNT(DISTINCT)`, `SUM`, and `AVG` per group.

**Example.**

```text
records = [{"id":"a","v":1}, {"id":"b","v":2}, {"id":"a","v":3}, {"id":"c","v":4}]
dedupe   -> [{"id":"a","v":1}, {"id":"b","v":2}, {"id":"c","v":4}]
frequency_map(["us","eu","us"]) -> {"us": 2, "eu": 1}
aggregate([("us",10.0), ("us",30.0), ("us",None)])
         -> {"us": {"rows": 3, "distinct": 2, "total": 40.0, "avg": 13.333...}}
```

**Constraints.**

- O(n) time; no pandas; do not mutate the input records.
- Preserve input order for dedupe, and first-seen order for groups.
- Decide and document null behaviour: here a null *key* is dropped, while a null *value* counts as a row but is excluded from `SUM` and `COUNT(DISTINCT)` (SQL semantics).

### 1a. Dedupe preserving order

The whole problem is one pass with a `set` of keys already seen. A set gives average O(1) membership, so the pass is O(n) time and O(k) memory for k distinct keys -- the same shape as a Spark `dropDuplicates`, but with deterministic first-seen order rather than arbitrary order.

The pitfall is null handling: a `None` key is not a real business key (it would merge unrelated rows), so it is dropped rather than treated as one group. Note also that the records themselves are appended by reference and never copied or mutated.

In [ ]:
from collections import Counter, defaultdict
from collections.abc import Iterable
from typing import Any


def dedupe_preserving_order(records: Iterable[dict[str, Any]], key: str = "id") -> list[dict[str, Any]]:
    """Keep the first record seen for each value of `key`, in input order."""
    seen: set[Any] = set()
    out: list[dict[str, Any]] = []
    for record in records:
        value = record.get(key)
        if value is None or value in seen:
            continue
        seen.add(value)
        out.append(record)
    return out

### 1b. `frequency_map` - `COUNT(*) GROUP BY item`

`collections.Counter` is a dict subclass specialised for counting, so the implementation is a one-liner that is still O(n) time and O(k) memory. Because `Counter` iterates in first-seen order, `most_common` and the raw dict both produce deterministic output -- important when a test or a dashboard compares two runs.

Keep the input out of the pipeline after this: `frequency_map` doubles as the reducer for a distributed word count once each partition has emitted its local counts.

In [ ]:
def frequency_map(items: Iterable[Any]) -> dict[Any, int]:
    """COUNT(*) GROUP BY item, without pandas."""
    return dict(Counter(items))

### 1c. `group_by` - bucket rows per key

A `defaultdict(list)` plus one append per record is the in-memory `GROUP BY`. Because dicts preserve insertion order since Python 3.7, the resulting groups come out in first-seen order, not sorted -- a detail worth stating in an interview, since a downstream `ORDER BY` is then unnecessary.

The method retains the original row objects, so the memory cost is O(n), not O(k). Mutating a row inside a group mutates the caller's data.

In [ ]:
def group_by(records: Iterable[dict[str, Any]], key: str) -> dict[Any, list[dict[str, Any]]]:
    """GROUP BY key -> list of rows, preserving first-seen group order."""
    groups: dict[Any, list[dict[str, Any]]] = defaultdict(list)
    for record in records:
        groups[record.get(key)].append(record)
    return dict(groups)

### 1d. `aggregate` - `COUNT`, `COUNT(DISTINCT)`, `SUM`, `AVG`

One accumulator dict per group holds the row count, a set for distinct values, and the running total. A `None` value increments `rows` (so `COUNT(*)` matches SQL) but is skipped for the distinct set and the total, which is exactly `COUNT(col)`/`SUM(col)` semantics.

The average is computed once at the end from the total, never by averaging averages. That is the single most common aggregation bug in pipelines, and it is why `AVG` is not additive across partitions.

In [ ]:
def aggregate(rows: Iterable[tuple[Any, float | None]]) -> dict[Any, dict[str, float]]:
    """COUNT / COUNT(DISTINCT) / SUM / AVG grouped by the first tuple element."""
    buckets: dict[Any, dict[str, Any]] = defaultdict(lambda: {"rows": 0, "seen": set(), "total": 0.0})
    for group, value in rows:
        bucket = buckets[group]
        bucket["rows"] += 1
        if value is not None:
            bucket["seen"].add(value)
            bucket["total"] += value
    return {
        group: {
            "rows": bucket["rows"],
            "distinct": len(bucket["seen"]),
            "total": bucket["total"],
            "avg": bucket["total"] / bucket["rows"],
        }
        for group, bucket in buckets.items()
    }

### 1e. Happy path

This is the documented example, and it also pins the *order* of the result, not just its contents. For dedupe the first `"a"` row wins (`v = 1`), and the returned records are the very same objects from the input list.

In [ ]:
records = [
    {"id": "a", "v": 1},
    {"id": "b", "v": 2},
    {"id": "a", "v": 3},
    {"id": "c", "v": 4},
    {"id": "b", "v": 5},
]
print("dedupe:", dedupe_preserving_order(records))
assert dedupe_preserving_order(records) == [
    {"id": "a", "v": 1},
    {"id": "b", "v": 2},
    {"id": "c", "v": 4},
]
assert dedupe_preserving_order(records, key="v") == records  # every v is unique

### 1f. Happy path - groups and aggregates

`group_by` must return groups in first-seen order (`a`, `b`, `c`) and keep each group's rows in input order. The aggregate check uses the documented null behaviour: three rows for `us`, two distinct non-null values, total 40.0, average `40/3`, while `eu` has two rows but one distinct value and therefore average 5.0.

In [ ]:
grouped = group_by(records, key="id")
print("group order:", list(grouped))
assert list(grouped) == ["a", "b", "c"]  # first-seen order, not sorted
assert [row["v"] for row in grouped["a"]] == [1, 3]

agg = aggregate([("us", 10.0), ("eu", 5.0), ("us", 30.0), ("eu", 5.0), ("us", None)])
print("aggregate:", agg)
assert agg["us"] == {"rows": 3, "distinct": 2, "total": 40.0, "avg": 40.0 / 3}
assert agg["eu"] == {"rows": 2, "distinct": 1, "total": 10.0, "avg": 5.0}

### 1g. Edge cases - empty input, null keys, duplicates

Empty input must return empty containers, never raise. A `None` key is dropped by dedupe, and duplicate `None` keys collapse to nothing rather than to a group. `frequency_map` on an empty iterable returns `{}` because `Counter` over nothing is empty.

In [ ]:
assert dedupe_preserving_order([]) == []
assert dedupe_preserving_order([{"id": None}, {"id": None}, {"id": 1}]) == [{"id": 1}]
assert frequency_map(["us", "eu", "us", "us", "eu"]) == {"us": 3, "eu": 2}
assert frequency_map([]) == {}
assert group_by([], key="id") == {}
assert aggregate([]) == {}

### 1h. Stated output check

The documented example prints exactly these lines. Run the cell and compare the printed values with the docstring above; the asserts are the machine-checkable version of the same claim.

In [ ]:
print("dedupe   :", [row["id"] for row in dedupe_preserving_order(records)])
print("frequency:", frequency_map(["us", "eu", "us", "us", "eu"]))
print("aggregate:", aggregate([("us", 10.0), ("us", 30.0), ("us", None)]))
print("P1 checks passed")

**Complexity & edge cases.** Dedupe and counting are O(n) time; dedupe uses O(k) memory for k distinct keys, while `group_by`/`aggregate` use O(n) because they retain rows or distinct values. Edge cases: empty input returns `[]`/`{}`; unhashable keys must be normalised to tuples; `1 == True` and `1 == 1.0` collide in a Python dict, so normalise types on ingest; summing money in `float` drifts, so accumulate integer cents or `Decimal`; keep order in a list because iterating a `set` is not order-stable across runs.

**Interview follow-ups.**

1. *The data no longer fits in RAM - how does each operation change?* Sort-merge or hash-partition on disk; approximate distinct counts with HyperLogLog; a group-by becomes a shuffle in Spark.
2. *Why is `dict`/`set` lookup called O(1), and when is it not?* Average-case hashing; worst case O(n) with adversarial collisions; keys must be hashable with a stable hash.

## Problem 2 - Multi-key sorting with key functions

**Statement.** Reproduce `ORDER BY status ASC, amount DESC, id ASC`, then generalise to any number of `(field, descending)` pairs. Also handle nulls-last and the case where a key function cannot express the comparison.

**Example.**

```text
input:
  [{"status":"a","amount":3,"id":3}, {"status":"a","amount":1,"id":1},
   {"status":"b","amount":2,"id":2}, {"status":"a","amount":1,"id":4}]
output:
  [{"status":"a","amount":3,"id":3}, {"status":"a","amount":1,"id":1},
   {"status":"a","amount":1,"id":4}, {"status":"b","amount":2,"id":2}]
```

**Constraints.** Use a key function where possible; avoid `cmp_to_key` in the hot path; `None` must sort last and never raise `TypeError`; do not mutate the input list.

### 2a. One tuple key for a fixed `ORDER BY`

A tuple key is compared element by element, so `(status, -amount, id)` reproduces `status ASC, amount DESC, id ASC` in a single pass. Negating a numeric field is the cheapest way to flip its direction, and it works because the key is computed once per row (decorate-sort-undecorate) rather than once per comparison.

The catch is that `-x` only exists for numbers. Strings, dates, and custom objects cannot be negated, which is what forces the general version next.

In [ ]:
from functools import cmp_to_key
from typing import Any


def sort_orders(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """ORDER BY status ASC, amount DESC, id ASC using a single tuple key."""
    return sorted(rows, key=lambda row: (row["status"], -row["amount"], row["id"]))

### 2b. Any number of mixed-direction keys

Python's Timsort is stable, so sorting repeatedly from the least-significant key to the most-significant one gives the same result as a single combined key. Each pass only has to handle one direction, so strings and dates work fine and no value is ever negated.

The cost is one extra pass per key and a copy of the list; for a handful of keys that is irrelevant next to the O(n log n) comparisons. Copy with `list(rows)` first so the caller's list is never reordered.

In [ ]:
def sort_by_keys_stable(rows: list[dict[str, Any]], keys: list[tuple[str, bool]]) -> list[dict[str, Any]]:
    """ORDER BY any number of (field, descending) pairs using stable passes."""
    out = list(rows)
    for field, descending in reversed(keys):
        out.sort(key=lambda row: row[field], reverse=descending)
    return out

### 2c. Nulls last

Comparing `None` with an `int` raises `TypeError`, so a null flag must lead the tuple: `False` sorts before `True`, pushing every null to the end regardless of the value type. This is the standard `NULLS LAST` emulation and it works for any comparable column.

If the direction is descending, you place the flag accordingly (`row[field] is not None`) -- a common off-by-one in interview answers.

In [ ]:
def sort_nulls_last(rows: list[dict[str, Any]], field: str) -> list[dict[str, Any]]:
    """None sorts after every real value because the boolean flag leads the key."""
    return sorted(rows, key=lambda row: (row[field] is None, row[field]))

### 2d. The comparator escape hatch

Sometimes the ordering is genuinely not expressible as a key -- for example custom business logic that compares two records field by field with early exit. `functools.cmp_to_key` wraps a `compare(a, b) -> -1/0/1` function into something `sorted` accepts.

It is the slow path: the comparator runs from Python on every comparison, so O(n log n) comparisons each pay interpreter overhead instead of one cheap key per element. Mention it as a fallback, not as the default.

In [ ]:
def sort_with_comparator(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """The escape hatch when a key function cannot express the comparison."""

    def compare(left: dict[str, Any], right: dict[str, Any]) -> int:
        if left["status"] != right["status"]:
            return -1 if left["status"] < right["status"] else 1
        if left["amount"] != right["amount"]:
            return -1 if left["amount"] > right["amount"] else 1
        return (left["id"] > right["id"]) - (left["id"] < right["id"])

    return sorted(rows, key=cmp_to_key(compare))

### 2e. Happy path

All three strategies must agree on the documented example. That agreement is the real test: the tuple key, the stable multi-pass sort, and the comparator are three implementations of one `ORDER BY`, so any divergence is a bug in the translation.

In [ ]:
rows = [
    {"status": "a", "amount": 3, "id": 3},
    {"status": "a", "amount": 1, "id": 1},
    {"status": "b", "amount": 2, "id": 2},
    {"status": "a", "amount": 1, "id": 4},
]
expected = [
    {"status": "a", "amount": 3, "id": 3},
    {"status": "a", "amount": 1, "id": 1},
    {"status": "a", "amount": 1, "id": 4},
    {"status": "b", "amount": 2, "id": 2},
]
print("sorted:", [(r["status"], r["amount"], r["id"]) for r in sort_orders(rows)])
assert sort_orders(rows) == expected
assert sort_by_keys_stable(rows, [("status", False), ("amount", True), ("id", False)]) == expected
assert sort_with_comparator(rows) == expected

### 2f. Edge cases - stability, nulls, no mutation

`sorted` copies and `sort_by_keys_stable` copies first, so the caller's list must be byte-for-byte unchanged after either call -- a real requirement in tests and in pipelines that reuse a batch. Stability means equal keys keep their input order, which is what makes deterministic tie-breaking possible. Nulls-last has to handle an empty input and a `None` sitting next to a number without raising.

In [ ]:
before = [dict(row) for row in rows]
sort_orders(rows)
assert rows == before, "sorted() must not mutate the input"

# Stability: equal keys keep their original relative order.
letters = [("b", 1), ("a", 1), ("b", 0)]
assert sorted(letters, key=lambda pair: pair[1]) == [("b", 0), ("b", 1), ("a", 1)]

# Nulls-last never raises, even when None sits next to a number.
assert sort_nulls_last([{"x": 2}, {"x": None}, {"x": 1}], "x") == [{"x": 1}, {"x": 2}, {"x": None}]
assert sort_nulls_last([], "x") == []

### 2g. Stated output check

Print the sorted projection from the documented example and confirm the order by eye, then let the asserts confirm it mechanically. The cell also re-checks that the generic multi-key path produces the same sequence.

In [ ]:
print("expected:", [(r["status"], r["amount"], r["id"]) for r in expected])
print("actual  :", [(r["status"], r["amount"], r["id"]) for r in sort_orders(rows)])
assert sort_orders(rows) == sort_by_keys_stable(rows, [("status", False), ("amount", True), ("id", False)])
print("P2 checks passed")

**Complexity & edge cases.** Sorting is O(n log n) comparisons plus O(n) memory for `sorted` (`list.sort` is in place). Python's Timsort is stable, which is what makes multi-pass sorting and deterministic tie-breaking work. Edge cases: `None` inside a tuple key raises `TypeError` unless a null flag leads the tuple; negating fails for strings, dates, and custom objects, so prefer `reverse=` passes; naive and aware datetimes cannot be compared; `cmp_to_key` runs Python for every comparison and is several times slower than a key function.

**Interview follow-ups.**

1. *How would you sort 100 GB on one machine?* Sort each chunk, write it to disk, then k-way merge with `heapq.merge`; keep only one buffer per chunk in memory.
2. *Why does `sorted` need the key function to be cheap?* The key is computed once per element (decorate-sort-undecorate), but an expensive or non-deterministic key breaks that assumption; the comparator variant recomputes logic on every comparison.

## Problem 3 - Top-K and top-N-per-group with `heapq`

**Statement.** Return the K most frequent keys from a stream, and the largest N rows per group, without sorting everything. A full sort is O(n log n); a bounded heap is O(n log k) and stores only k elements.

**Example.**

```text
keys = ["a","b","a","c","a","b","d"], k = 2
top_k -> [("a", 3), ("b", 2)]

rows grouped by dept, top 2 salaries per dept, computed in one pass
eng: [("bob",150), ("ana",120)]
ops: [("ed",110),  ("di",100)]
```

**Constraints.** Return `[]` for `k <= 0`; ties must be deterministic; keep memory bounded per group; do not sort the full input.

### 3a. `top_k` with `Counter.most_common`

`Counter.most_common(k)` counts everything in O(n), then uses a heap to pick the k largest counts in O(m log k) for m distinct keys. It is the shortest correct answer, and for `k == 0` it returns an empty list rather than raising.

Counting still needs O(m) memory, so for a genuinely unbounded key space this is a sketch problem (Count-Min or Space-Saving), not a `Counter` problem.

In [ ]:
import heapq
from collections import Counter, defaultdict
from collections.abc import Iterable
from typing import Any


def top_k(keys: Iterable[Any], k: int) -> list[tuple[Any, int]]:
    """K most common keys; Counter.most_common uses a heap internally."""
    return Counter(keys).most_common(k)

### 3b. Explicit bounded min-heap

Building the heap by hand shows the mechanism an interviewer wants: keep a *min*-heap of size k, push while it is not full, and only replace the root when the incoming count beats the smallest kept count. Memory is O(k) beyond the counter, and the final `sorted` is O(k log k).

The tie-break `(-count, str(key))` makes equal counts deterministic; without it, two runs could return different keys.

In [ ]:
def top_k_bounded(keys: Iterable[Any], k: int) -> list[tuple[Any, int]]:
    """Explicit bounded min-heap: O(n log k) time, O(k) extra memory."""
    if k <= 0:
        return []
    heap: list[tuple[int, Any]] = []
    for key, count in Counter(keys).items():
        if len(heap) < k:
            heapq.heappush(heap, (count, key))
        elif count > heap[0][0]:
            heapq.heapreplace(heap, (count, key))
    return sorted(((key, count) for count, key in heap), key=lambda item: (-item[1], str(item[0])))

### 3c. Top-N per group in one pass

Keep one bounded heap per group key. Each row is pushed with its running index, so the heap orders by `(metric, index)`; popping from a min-heap drops the smallest metric and, on a tie, the earliest index. That yields a deterministic "largest N, ties broken by input order".

Memory is O(g x N) for g groups rather than O(n), which is the whole point: a per-department leaderboard should not materialise every row.

In [ ]:
def top_n_per_group(
    rows: Iterable[dict[str, Any]], group: str, metric: str, n: int
) -> dict[Any, list[dict[str, Any]]]:
    """Largest n rows per group in one pass, with bounded memory per group."""
    if n <= 0:
        return {}
    heaps: dict[Any, list[tuple[float, int, dict[str, Any]]]] = defaultdict(list)
    for index, row in enumerate(rows):
        heap = heaps[row[group]]
        # The running index breaks metric ties deterministically.
        heapq.heappush(heap, (row[metric], index, row))
        if len(heap) > n:
            heapq.heappop(heap)
    return {
        group_key: [
            row for _value, _index, row in sorted(heap, key=lambda item: (-item[0], item[1]))
        ]
        for group_key, heap in heaps.items()
    }

### 3d. Happy path - top K

The counted example gives `a` three times, `b` twice, then `c` and `d` once each, so `k = 2` is `[("a", 3), ("b", 2)]`. The manual heap must return the same pairs, and asking for more keys than exist must return everything.

In [ ]:
keys = ["a", "b", "a", "c", "a", "b", "d"]
print("top_k:", top_k(keys, 2))
assert top_k(keys, 2) == [("a", 3), ("b", 2)]
assert top_k_bounded(keys, 2) == [("a", 3), ("b", 2)]
assert sorted(top_k_bounded(keys, 10)) == sorted(top_k(keys, 10))

### 3e. Happy path - largest N per group

For `eng`, the two largest salaries are Bob at 150 and Ana at 120; for `ops`, Ed at 110 and Di at 100. The output rows are the original dicts, so the caller can read every column, and the order within a group is largest-first.

In [ ]:
rows = [
    {"dept": "eng", "name": "ana", "salary": 120},
    {"dept": "eng", "name": "bob", "salary": 150},
    {"dept": "eng", "name": "cy", "salary": 90},
    {"dept": "ops", "name": "di", "salary": 100},
    {"dept": "ops", "name": "ed", "salary": 110},
]
top2 = top_n_per_group(rows, "dept", "salary", 2)
print("top2 eng:", [(r["name"], r["salary"]) for r in top2["eng"]])
assert [(r["name"], r["salary"]) for r in top2["eng"]] == [("bob", 150), ("ana", 120)]
assert [(r["name"], r["salary"]) for r in top2["ops"]] == [("ed", 110), ("di", 100)]

### 3f. Edge cases - `k <= 0`, empty input, ties

A non-positive `k` must return an empty result instead of raising or looping forever, and an empty input must not invent groups. Ties are the subtle case: two keys with the same count (or two rows with the same metric) need an explicit secondary key or the output is order-dependent, which breaks reproducibility.

In [ ]:
assert top_k(keys, 0) == []
assert top_k_bounded(keys, 0) == []
assert top_n_per_group(rows, "dept", "salary", 0) == {}
assert top_n_per_group([], "dept", "salary", 2) == {}

# Ties fall back to input order (the running index), so the result is stable.
tied = [
    {"dept": "d", "name": "first", "salary": 10},
    {"dept": "d", "name": "second", "salary": 10},
    {"dept": "d", "name": "low", "salary": 1},
]
assert [r["name"] for r in top_n_per_group(tied, "dept", "salary", 2)["d"]] == ["first", "second"]

### 3g. Stated output check

Print the documented `top_k` result and the top two engineers, then assert the exact tuples from the statement. This is the line a reviewer will read first, so it should match the `Example` block character for character.

In [ ]:
print("top_k ->", top_k(keys, 2))
print("eng   ->", [(r["name"], r["salary"]) for r in top2["eng"]])
print("ops   ->", [(r["name"], r["salary"]) for r in top2["ops"]])
print("P3 checks passed")

**Complexity & edge cases.** `heapq.nlargest(k, ...)`/`nsmallest` and `Counter.most_common(k)` are O(n log k) with O(k) memory. `top_n_per_group` is O(n log n) worst case but only O(g x n) memory for g groups. Edge cases: `k` larger than the number of distinct keys returns everything; ties need an explicit secondary key or the output is order-dependent; one heavy hitter dominates a `Counter`, so a Count-Min Sketch is the scalable answer; unhashable keys must be normalised first; `k <= 0` must not crash.

**Interview follow-ups.**

1. *When is `nlargest` worse than a full sort?* When k is close to n: O(n log k) approaches O(n log n) while adding heap overhead, so `sorted(...)[:k]` can win.
2. *How do you get the top K globally across partitions?* Each partition emits its local top K, then a single reducer merges K x partitions candidates with a bounded heap.

## Problem 4 - Rolling windows: max sum, moving average, percentile

**Statement.** Implement the windowed primitives behind dashboards, rate limiters, and SLOs:

1. `moving_average(values, window)` - trailing simple moving average, one value per full window.
2. `max_window_sum(values, window)` - largest sum of any contiguous window of exactly `window` elements.
3. `percentile(values, pct)` and `rolling_percentile(values, window, pct)` - linear-interpolation percentiles.
4. `trailing_event_count(timestamps, window)` - events in `(t - window, t]`, amortised O(1) per event.

**Example.**

```text
moving_average([1,2,3,4,5], 3)          -> [2.0, 3.0, 4.0]
max_window_sum([1,2,3,4,5], 3)          -> 12
percentile([10,12,13,15,20,100], 50)    -> 14.0
trailing_event_count([0,1,2,10,11], 3)  -> [1, 2, 3, 1, 2]
```

**Constraints.** O(1) update per element for the average and the count; document the window boundary; raise on a non-positive window; handle a window larger than the input.

### 4a. Moving average - sliding sum, not a re-sum

Add the incoming value and subtract the one leaving the window, so each step is O(1) instead of O(window). The output has exactly `len(values) - window + 1` entries: one value per *full* window, with the partial prefix discarded.

Because the running total is a float, long windows accumulate rounding error; `math.fsum` on an occasional reset, or Kahan summation, keeps it honest. A window larger than the input returns `[]` rather than a partial average.

In [ ]:
import math
from collections import deque
from collections.abc import Sequence


def moving_average(values: Sequence[float], window: int) -> list[float]:
    """Trailing simple moving average, one output per full window."""
    if window <= 0:
        raise ValueError("window must be positive")
    if window > len(values):
        return []
    out: list[float] = []
    total = 0.0
    for index, value in enumerate(values):
        total += value
        if index >= window:
            total -= values[index - window]
        if index >= window - 1:
            out.append(total / window)
    return out

### 4b. Maximum window sum

The same sliding total, but keep the best value seen instead of emitting every window. This is the O(n) shape used for "busiest 5-minute bucket" style questions; the naive recompute is O(n x window).

Returning `None` (rather than `0` or `[]`) for an impossible window keeps the failure explicit. Negative values matter: initialising `best` to 0 would be wrong on an all-negative array, so seed it with the first window's sum.

In [ ]:
def max_window_sum(values: Sequence[float], window: int) -> float | None:
    """Largest sum of any contiguous window of exactly `window` elements."""
    if window <= 0 or window > len(values):
        return None
    total = sum(values[:window])
    best = total
    for index in range(window, len(values)):
        total += values[index] - values[index - window]
        best = max(best, total)
    return best

### 4c. Percentile with linear interpolation

Sort once, map the percentile to a fractional position `(n - 1) * pct / 100`, then interpolate between the two neighbouring order statistics. This matches the numpy/`pandas` default, so the numbers agree with a notebook the interviewer may compare against.

Two pitfalls: `[]` has no percentile (raise), and `pct` outside `[0, 100]` is a caller bug, not something to clamp silently. On a handful of points, p99 is simply the maximum.

In [ ]:
def percentile(values: Sequence[float], pct: float) -> float:
    """Linear-interpolation percentile (the numpy default); pct in [0, 100]."""
    if not values:
        raise ValueError("empty sample")
    if not 0 <= pct <= 100:
        raise ValueError("pct must be in [0, 100]")
    ordered = sorted(values)
    position = (len(ordered) - 1) * pct / 100
    lower = math.floor(position)
    upper = math.ceil(position)
    if lower == upper:
        return float(ordered[int(position)])
    weight = position - lower
    return ordered[lower] * (1 - weight) + ordered[upper] * weight

### 4d. Rolling percentile - the honest O(n x w log w) version

Re-sorting each window is the correct baseline to write first. It is deliberately not the fancy answer: an exact rolling quantile needs an order-statistic structure (two heaps or a balanced tree) or an approximate sketch, and saying so out loud is worth more than silently shipping a slow loop.

The slice `values[i : i + window]` builds a fresh list per window, so the memory churn is O(n x w) allocations over the run.

In [ ]:
def rolling_percentile(values: Sequence[float], window: int, pct: float) -> list[float]:
    """Percentile of each trailing window (recomputes the sort per window)."""
    if window <= 0:
        raise ValueError("window must be positive")
    return [percentile(values[i : i + window], pct) for i in range(len(values) - window + 1)]

### 4e. Trailing event count - the rate-limiter window

A `deque` of live timestamps gives O(1) amortised work per event: append the new timestamp, then pop from the left while the oldest event is outside the window. Each timestamp is pushed and popped at most once across the whole run.

The boundary is documented as half-open on the left: an event at exactly `t - window` is dropped, so the window is `(t - window, t]`. Choosing that convention explicitly prevents the classic off-by-one in rate limiters.

In [ ]:
def trailing_event_count(timestamps: Sequence[int], window: int) -> list[int]:
    """Deque-based count of events in (t - window, t]; amortised O(1) per event."""
    buffer: deque[int] = deque()
    out: list[int] = []
    for current in timestamps:
        buffer.append(current)
        while buffer and buffer[0] <= current - window:
            buffer.popleft()
        out.append(len(buffer))
    return out

### 4f. Happy path - all four primitives

Every documented example in one cell: the moving average, the best window sum, the interpolated median, and the trailing counts. The median of `[10, 12, 13, 15, 20, 100]` is `(13 + 15) / 2 = 14.0`, which is the interpolation case rather than a raw sample.

In [ ]:
print("moving_average:", moving_average([1, 2, 3, 4, 5], 3))
assert moving_average([1, 2, 3, 4, 5], 3) == [2.0, 3.0, 4.0]
assert moving_average([1], 1) == [1.0]
print("max_window_sum:", max_window_sum([1, 2, 3, 4, 5], 3))
assert max_window_sum([1, 2, 3, 4, 5], 3) == 12
print("percentile p50:", percentile([10.0, 12.0, 13.0, 15.0, 20.0, 100.0], 50))
assert percentile([10.0, 12.0, 13.0, 15.0, 20.0, 100.0], 50) == 14.0
print("trailing_event_count:", trailing_event_count([0, 1, 2, 10, 11], 3))
assert trailing_event_count([0, 1, 2, 10, 11], 3) == [1, 2, 3, 1, 2]

### 4g. Happy path - interpolation details

`percentile([1, 2, 3, 4], 50)` is 2.5 because the fractional position lands halfway between the second and third samples; the endpoints (`pct = 0` and `pct = 100`) must return the min and max exactly. A single-value sample returns that value for any percentile.

In [ ]:
assert percentile([1, 2, 3, 4], 50) == 2.5
assert percentile([5], 99) == 5.0
assert percentile([1.0, 2.0, 3.0], 0) == 1.0
assert percentile([1.0, 2.0, 3.0], 100) == 3.0
assert rolling_percentile([1, 2, 3, 4], 3, 100) == [3.0, 4.0]

### 4h. Edge cases - oversized windows and bad arguments

A window larger than the input is a legitimate no-result case: `[]` for the average and rolling percentile, `None` for the max window sum, and the deque simply counts what it has for the event window. Non-positive windows and empty samples raise, because returning a plausible-looking number would hide the caller's bug.

Note the asymmetry: an empty timestamp list is fine and returns `[]`, while an empty *sample* for a percentile raises -- a count of zero events is meaningful, a median of zero values is not.

In [ ]:
assert moving_average([1, 2], 3) == []
assert max_window_sum([1, 2], 3) is None
assert max_window_sum([], 1) is None
assert max_window_sum([-5, -1, -3], 2) == -4
assert trailing_event_count([], 3) == []

for bad_call in (
    lambda: moving_average([1, 2], 0),
    lambda: percentile([], 50),
    lambda: percentile([1, 2, 3], 101),
    lambda: rolling_percentile([1, 2], 0, 50),
):
    try:
        bad_call()
    except ValueError:
        pass
    else:
        raise AssertionError("invalid window or percentile must raise ValueError")

### 4i. Stated output check

Re-print the four documented outputs so the cell output in the saved notebook matches the `Example` block exactly. The assert on the trailing counts also checks the left-edge convention: the event at 0 is gone by the time the window at 3 arrives.

In [ ]:
print("moving_average            ->", moving_average([1, 2, 3, 4, 5], 3))
print("max_window_sum            ->", max_window_sum([1, 2, 3, 4, 5], 3))
print("percentile(p50)           ->", percentile([10.0, 12.0, 13.0, 15.0, 20.0, 100.0], 50))
print("trailing_event_count      ->", trailing_event_count([0, 1, 2, 10, 11], 3))
assert trailing_event_count([0, 1, 2, 10, 11], 3) == [1, 2, 3, 1, 2]
print("P4 checks passed")

**Complexity & edge cases.** The moving average and event count are O(n) time with O(window) memory (the deque only ever holds live events); `max_window_sum` is O(n) with O(1) extra space; `rolling_percentile` is O(n x w log w) because each window is re-sorted. Edge cases: a window larger than the input yields `[]`/`None`; define whether the window is inclusive at the left edge (here it is exclusive: `t - window` is dropped); empty samples raise in `percentile`; floating-point averages can drift, so use `math.fsum` or Kahan summation for very long windows; p99 on a handful of points is just the maximum and is statistical noise.

**Interview follow-ups.**

1. *How do you compute an exact rolling median efficiently?* Keep a max-heap of the lower half and a min-heap of the upper half, rebalancing on insert/evict: O(log window) per element.
2. *Why are percentiles not additive across buckets?* The p99 of a union is not the max or average of the per-bucket p99s - you must merge the underlying sketches or raw samples.

## Problem 5 - Two pointers and prefix sums

**Statement.** Find two values in a sorted array that sum to a target (two pointers), answer range-sum queries in O(1) (prefix sums), count subarrays summing to a target (prefix sums + hash map), and find the first element at or after a target (`bisect`).

**Example.**

```text
two_sum_sorted([1,3,4,6,8,11], 10)   -> (2, 3)          # values 4 + 6
build_prefix_sums([2,4,6,8])         -> [0, 2, 6, 12, 20]
range_sum(prefix, 1, 3)              -> 10               # 4 + 6
count_subarrays_with_sum([1,1,1], 2) -> 2
first_at_or_after([1,3,3,7], 3)      -> 3
```

**Constraints.** Two pointers is O(n) time and O(1) space on already-sorted input; prefix arrays build in O(n) and answer in O(1); support negative numbers.

### 5a. Two pointers on a sorted array

Start at both ends: if the sum is too small the only way to grow it is to advance the left pointer, and if it is too large the only way to shrink it is to retreat the right pointer. Each index moves at most n times, so the scan is O(n) time and O(1) space.

The precondition is that the data is sorted; on unsorted data the move decision is invalid and you must hash instead. Returning indices (not values) keeps duplicates unambiguous, and any valid pair is acceptable.

In [ ]:
from bisect import bisect_left, bisect_right
from collections.abc import Sequence
from itertools import accumulate


def two_sum_sorted(nums: Sequence[int], target: int) -> tuple[int, int] | None:
    """Two-pointer scan on a sorted array; O(n) time, O(1) space."""
    left, right = 0, len(nums) - 1
    while left < right:
        total = nums[left] + nums[right]
        if total == target:
            return left, right
        if total < target:
            left += 1
        else:
            right -= 1
    return None

### 5b. Prefix sums

Store `prefix[i] = sum(values[:i])` with a leading zero, so the array has length `n + 1` and any range `[lo, hi)` is one subtraction: `prefix[hi] - prefix[lo]`. Building is O(n) with `itertools.accumulate`, and every later query is O(1).

The leading zero is what makes `lo = 0` work without a branch. Skipping it is the classic off-by-one that forces a special case for the first element.

In [ ]:
def build_prefix_sums(values: Sequence[int]) -> list[int]:
    """prefix[i] = sum(values[:i]); length n + 1 so prefix[0] == 0."""
    return [0, *accumulate(values)]

### 5c. O(1) range queries with validation

With the prefix array in hand, `range_sum` is two lookups. Validate `0 <= lo <= hi <= len(prefix) - 1` up front: an invalid range must raise rather than silently return a negative or wrapped number, because a wrong aggregate is far worse than a failed job.

An empty range (`lo == hi`) is valid and returns 0, matching the mathematical convention.

In [ ]:
def range_sum(prefix: Sequence[int], lo: int, hi: int) -> int:
    """sum(values[lo:hi]) in O(1) once the prefix array is built."""
    if not 0 <= lo <= hi <= len(prefix) - 1:
        raise ValueError("invalid range")
    return prefix[hi] - prefix[lo]

### 5d. Counting subarrays that sum to a target

Keep a hash map from each running prefix sum to how many times it has occurred, seeded with `{0: 1}` for the empty prefix. At every step, the number of subarrays ending here is the count of times `running - target` has been seen.

This is O(n) time and O(n) memory, and it is the version that *works with negatives*, where a sliding window does not. The `{0: 1}` seed is what counts subarrays that start at index 0.

In [ ]:
def count_subarrays_with_sum(values: Sequence[int], target: int) -> int:
    """Count contiguous subarrays summing to target (works with negatives)."""
    seen = {0: 1}
    running = 0
    count = 0
    for value in values:
        running += value
        count += seen.get(running - target, 0)
        seen[running] = seen.get(running, 0) + 1
    return count

### 5e. Lower and upper bound with `bisect`

`bisect_left` returns the first index at or after the target and `bisect_right - 1` the last index at or before it; both are O(log n) and run in C. Convert the index to a value and return `None` at the edges so callers do not have to catch `IndexError`.

Interviews care about the pair because it answers "first row at or after this watermark" (catch-up) and "last partition at or before this key" (lookup) without scanning. Any monotonic ordering works, which is why ISO date strings are the standard partition key.

In [ ]:
def first_at_or_after(sorted_values: Sequence[int], target: int) -> int | None:
    """Lower bound via bisect; None when every value is smaller than target."""
    index = bisect_left(sorted_values, target)
    return sorted_values[index] if index < len(sorted_values) else None


def last_at_or_before(sorted_values: Sequence[int], target: int) -> int | None:
    """Upper bound minus one; None when every value is larger than target."""
    index = bisect_right(sorted_values, target) - 1
    return sorted_values[index] if index >= 0 else None

### 5f. Happy path - two pointers

The documented target 10 is met by the values 4 and 6 at indices 2 and 3. A target no pair can reach returns `None`, and a single-element array cannot contain a pair. Negative inputs are fine because the comparison only depends on the ordering.

In [ ]:
assert two_sum_sorted([1, 3, 4, 6, 8, 11], 10) == (2, 3)
assert two_sum_sorted([1, 2, 3], 100) is None
assert two_sum_sorted([1], 1) is None
assert two_sum_sorted([-3, -1, 4, 9], 1) == (0, 2)

### 5g. Happy path - prefix sums and subarray counts

`build_prefix_sums([2, 4, 6, 8])` is `[0, 2, 6, 12, 20]`, so `range_sum(prefix, 1, 3)` is `12 - 2 = 10` (the values 4 and 6) and the full range is 20. The subarray counter finds two subarrays in `[1, 1, 1]` that sum to 2 and four in the alternating `[1, -1, 1, -1]`.

In [ ]:
prefix = build_prefix_sums([2, 4, 6, 8])
print("prefix:", prefix, "range[1:3] =", range_sum(prefix, 1, 3))
assert prefix == [0, 2, 6, 12, 20]
assert range_sum(prefix, 1, 3) == 10
assert range_sum(prefix, 0, 4) == 20
assert range_sum(prefix, 2, 2) == 0
assert count_subarrays_with_sum([1, 1, 1], 2) == 2
assert count_subarrays_with_sum([1, -1, 1, -1], 0) == 4

### 5h. Happy path - bounds

`first_at_or_after([1, 3, 3, 7], 3)` is the first 3, `last_at_or_before` is the last 3 (duplicates are handled by the two different bisect variants), and a target beyond both ends returns `None`. The empty list must also return `None` instead of raising.

In [ ]:
print("first_at_or_after([1,3,3,7], 3) =", first_at_or_after([1, 3, 3, 7], 3))
assert first_at_or_after([1, 3, 3, 7], 3) == 3
assert first_at_or_after([1, 3, 3, 7], 8) is None
assert first_at_or_after([], 1) is None
assert last_at_or_before([1, 3, 3, 7], 3) == 3
assert last_at_or_before([1, 3, 3, 7], 0) is None

### 5i. Edge cases - empty inputs and invalid ranges

Empty inputs must be inert everywhere: `count_subarrays_with_sum([], 0)` is 0 because no subarray exists, and the bound helpers return `None`. An inverted range must raise, and the raise happens *before* any indexing so the caller gets a clear error rather than a wrong number.

In [ ]:
assert count_subarrays_with_sum([], 0) == 0
assert build_prefix_sums([]) == [0]
assert range_sum(build_prefix_sums([]), 0, 0) == 0

try:
    range_sum(prefix, 3, 1)
except ValueError:
    pass
else:
    raise AssertionError("lo > hi must raise")

### 5j. Stated output check

Print the documented example outputs and assert the two that are not plain numbers. The prefix print makes the leading-zero convention visible, which is exactly what an interviewer will check first.

In [ ]:
print("prefix      ->", build_prefix_sums([2, 4, 6, 8]))
print("range[1:3]  ->", range_sum(build_prefix_sums([2, 4, 6, 8]), 1, 3))
print("subarrays   ->", count_subarrays_with_sum([1, 1, 1], 2))
print("first >= 3  ->", first_at_or_after([1, 3, 3, 7], 3))
print("P5 checks passed")

**Complexity & edge cases.** Two pointers is O(n) time / O(1) space but only valid on sorted data; a prefix array builds in O(n) and each query is O(1); `count_subarrays_with_sum` is O(n) time / O(n) memory and handles negatives (sliding windows do not). Edge cases: duplicate values are fine because any valid pair is acceptable; `lo == hi` returns 0; empty input; `bisect_left` gives the first equal element, `bisect_right - 1` gives the last; a monotonic predicate is required for binary search, so date partitions must be zero-padded ISO strings.

**Interview follow-ups.**

1. *Why does a sliding window fail with negative numbers?* Shrinking the window does not monotonically reduce the sum, so you cannot decide when to move the left edge.
2. *How do prefix sums extend to 2-D?* Build a summed-area table where `S[i][j]` is the sum of the rectangle above-left of `(i, j)`; each rectangle query is four lookups.

## Problem 6 - Merge overlapping intervals

**Statement.** Merge every overlapping or touching half-open `[start, end)` interval - how a scheduler coalesces file-availability windows and how compaction groups small files.

**Example.**

```text
merge([[1,3],[2,6],[8,10],[15,18]]) -> [[1,6],[8,10],[15,18]]
merge([[1,4],[4,5]])                -> [[1,5]]     # touching merges
merge([[1,10],[2,3],[4,5]])         -> [[1,10]]    # nested absorbed
```

**Constraints.** O(n log n) time; do not mutate the input; raise on `start > end`; keep zero-length intervals; treat touching endpoints as overlapping (half-open intervals).

### 6a. Sort, then one sweep

Sorting by start time reduces the problem to a single left-to-right pass: each interval either extends the last merged interval (`start <= merged[-1][1]`) or begins a new one. The sort dominates at O(n log n); the sweep is O(n) time and O(n) output.

Treating touching endpoints as overlapping (`<=`) is the half-open convention: `[1, 4)` and `[4, 5)` cover a continuous span with no gap. Switching to `<` gives closed intervals instead -- pick one and document it.

In [ ]:
from collections.abc import Iterable


def merge_intervals(intervals: Iterable[tuple[int, int]]) -> list[tuple[int, int]]:
    """Merge overlapping or touching half-open [start, end) intervals."""
    merged: list[tuple[int, int]] = []
    for start, end in sorted(intervals):
        if start > end:
            raise ValueError(f"invalid interval: {(start, end)}")
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    return merged

### 6b. Happy path

The four documented cases: disjoint intervals pass through, an overlap fuses to `(1, 6)`, touching intervals fuse to `(1, 5)`, and a nested interval is absorbed by `(1, 10)`. The `max` in the extension step is what handles nesting -- without it, `[1, 10)` followed by `[2, 3)` would shrink the end to 3.

In [ ]:
print(merge_intervals([(1, 3), (2, 6), (8, 10), (15, 18)]))
assert merge_intervals([(1, 3), (2, 6), (8, 10), (15, 18)]) == [(1, 6), (8, 10), (15, 18)]
assert merge_intervals([(1, 4), (4, 5)]) == [(1, 5)]
assert merge_intervals([(1, 10), (2, 3), (4, 5)]) == [(1, 10)]

### 6c. Edge cases - empty, zero-length, unordered, no mutation

`merge_intervals([])` must be `[]`, and a zero-length interval `(5, 5)` is valid and must survive. Because the function sorts a copy, the caller's list order is irrelevant and the caller's list is never reordered -- both properties matter when intervals arrive as a batch from an unpredictable source.

`start > end` is rejected loudly; a silently malformed interval corrupts every later merge.

In [ ]:
assert merge_intervals([(5, 5)]) == [(5, 5)]
assert merge_intervals([]) == []

# Input order must not matter, and the input must not be mutated.
data = [(8, 10), (1, 3), (2, 6), (15, 18)]
snapshot = list(data)
assert merge_intervals(data) == [(1, 6), (8, 10), (15, 18)]
assert data == snapshot

try:
    merge_intervals([(3, 1)])
except ValueError:
    pass
else:
    raise AssertionError("start > end must raise")

### 6d. Stated output check

Print the merged list for the documented input and compare with `[[1,6],[8,10],[15,18]]`. `sorted()` returns a list of tuples, so the printed form uses tuple syntax; the asserts are the authoritative check.

In [ ]:
print("merged ->", merge_intervals([(1, 3), (2, 6), (8, 10), (15, 18)]))
print("touching ->", merge_intervals([(1, 4), (4, 5)]))
print("nested ->", merge_intervals([(1, 10), (2, 3), (4, 5)]))
print("P6 checks passed")

**Complexity & edge cases.** O(n log n) for the sort plus O(n) for the sweep and the output; `sorted` copies, so the input is untouched. Edge cases: empty input returns `[]`; zero-length intervals survive; closed intervals need `start < merged[-1][1]` so touching endpoints stay separate; nested intervals are absorbed by `max`; huge coordinates favour an interval list over a bitmap; streaming intervals that arrive unordered require a buffer and can only be partially merged.

**Interview follow-ups.**

1. *How would you find the peak number of concurrent intervals?* Sweep the endpoints as `(+1, start)` and `(-1, end)` events, sort, and track a running maximum - O(n log n), or O(n) counting sort on small integer ranges.
2. *How does this relate to small-file compaction?* Group adjacent files whose size range overlaps or is below a threshold, then rewrite each group as one file - the same merge-with-tolerance idea.

## Problem 7 - Binary search over sorted partitions

**Statement.** A table is stored as sorted ISO-date partitions. (a) Find the partition that can contain a key, or the first partition at or after a watermark. (b) Generalise the partitioning idea: find the k-th smallest element of two sorted arrays by binary-searching the split point, which is the "median of two sorted arrays" interview classic.

**Example.**

```text
partitions = ["2026-01-01","2026-02-01","2026-03-01","2026-04-01"]
partition_for_key("2026-03-15")   -> "2026-03-01"
first_at_or_after("2026-03-15")   -> "2026-04-01"

kth_smallest_two_sorted([1,3,8], [2,4,5,6,7], k=4) -> 4
```

**Constraints.** O(log n) per lookup; O(log(min(n, m))) for the two-array selection; handle empty inputs and out-of-range `k`; the sorted arrays must be ascending (raise if not).

### 7a. Partition lookup for a key

`bisect_right(partitions, key) - 1` is the last partition start that is `<=` the key: the file that could contain it. This is the prune step in partition elimination, and it is O(log n) with no scanning.

`bisect_right` (not `bisect_left`) is what makes an exact partition-boundary key resolve to its own partition rather than the previous one. A key before every partition returns `None`.

In [ ]:
from bisect import bisect_left, bisect_right
from collections.abc import Sequence


def partition_for_key(partitions: Sequence[str], key: str) -> str | None:
    """Last partition whose start is <= key (the file that can contain key)."""
    index = bisect_right(partitions, key) - 1
    return partitions[index] if index >= 0 else None

### 7b. Watermark resume point

`bisect_left(partitions, watermark)` is the first partition at or after the watermark, i.e. where incremental processing resumes after the last committed watermark. The two helpers are deliberately mirrored: one looks back for a lookup, one looks forward for catch-up.

The partition strings must be zero-padded ISO dates, because `bisect` compares lexicographically and `"2026-1-01"` would sort after `"2026-12-01"`.

In [ ]:
def first_at_or_after(partitions: Sequence[str], watermark: str) -> str | None:
    """First partition at or after a watermark (where incremental processing resumes)."""
    index = bisect_left(partitions, watermark)
    return partitions[index] if index < len(partitions) else None

### 7c. k-th smallest of two sorted arrays

Binary-search the *split point* instead of the value: take `i` elements from `a` and `j = k - i` from `b`, and check that the largest left element is no bigger than the smallest right element on both sides. Each step halves the search range, so the selection is O(log(min(n, m))).

Always search the shorter array: it bounds the range and keeps both partition indices inside their arrays. The sentinels `-inf`/`inf` remove the boundary special cases. Unsorted input must be rejected explicitly, because the invariant fails silently otherwise.

In [ ]:
def kth_smallest_two_sorted(a: Sequence[int], b: Sequence[int], k: int) -> int:
    """k-th smallest (1-indexed) of two sorted arrays via partition binary search."""
    if not 1 <= k <= len(a) + len(b):
        raise ValueError("k out of range")
    if any(a[i] > a[i + 1] for i in range(len(a) - 1)) or any(b[i] > b[i + 1] for i in range(len(b) - 1)):
        raise ValueError("inputs must be sorted")
    if len(a) > len(b):
        a, b = b, a  # binary-search the shorter array
    n, m = len(a), len(b)
    low, high = max(0, k - m), min(k, n)
    while low <= high:
        i = (low + high) // 2          # take i elements from a
        j = k - i                      # ... and j elements from b
        a_left = a[i - 1] if i > 0 else float("-inf")
        a_right = a[i] if i < n else float("inf")
        b_left = b[j - 1] if j > 0 else float("-inf")
        b_right = b[j] if j < m else float("inf")
        if a_left <= b_right and b_left <= a_right:
            return max(a_left, b_left)
        if a_left > b_right:
            high = i - 1
        else:
            low = i + 1
    raise RuntimeError("unreachable: inputs were not sorted")

### 7d. Median of two sorted arrays

An even total length needs the average of the two middle order statistics, an odd total just the middle one. Building `median_two_sorted` on top of the k-th selection keeps one tested primitive and makes the even/odd branch obvious.

Two empty arrays have no median, so raise; one empty array is fine and reduces to a normal index.

In [ ]:
def median_two_sorted(a: Sequence[int], b: Sequence[int]) -> float:
    """Median of two sorted arrays built on the k-th selection."""
    total = len(a) + len(b)
    if total == 0:
        raise ValueError("both arrays are empty")
    if total % 2 == 1:
        return float(kth_smallest_two_sorted(a, b, total // 2 + 1))
    lower = kth_smallest_two_sorted(a, b, total // 2)
    upper = kth_smallest_two_sorted(a, b, total // 2 + 1)
    return (lower + upper) / 2

### 7e. Happy path - partition lookups

For the documented partitions, the key `2026-03-15` falls inside the March partition, an exact boundary key resolves to its own partition, and a key before the first partition returns `None`. The watermark helpers mirror that: a mid-month watermark resumes at the next partition, an exact boundary resumes at itself, and a watermark past the end returns `None`.

In [ ]:
partitions = ["2026-01-01", "2026-02-01", "2026-03-01", "2026-04-01"]
print("partition_for_key:", partition_for_key(partitions, "2026-03-15"))
assert partition_for_key(partitions, "2026-03-15") == "2026-03-01"
assert partition_for_key(partitions, "2026-01-01") == "2026-01-01"
assert partition_for_key(partitions, "2025-12-31") is None
assert partition_for_key([], "2026-01-01") is None
assert first_at_or_after(partitions, "2026-03-15") == "2026-04-01"
assert first_at_or_after(partitions, "2026-03-01") == "2026-03-01"
assert first_at_or_after(partitions, "2027-01-01") is None

### 7f. Happy path - every order statistic

Cross-check the selection against the obvious (slow) reference: `sorted(a + b)` and index `k - 1`. Looping over every valid `k` is the strongest cheap test of a partition algorithm, because an off-by-one in `i` or `j` usually shows up for only one value of `k`.

One empty array and the two median cases (odd and even total length) round out the happy path.

In [ ]:
a, b = [1, 3, 8], [2, 4, 5, 6, 7]
merged = sorted(a + b)
print("merged:", merged)
for k in range(1, len(merged) + 1):
    assert kth_smallest_two_sorted(a, b, k) == merged[k - 1]
assert kth_smallest_two_sorted([], [1, 2, 3], 2) == 2
assert kth_smallest_two_sorted([1, 2], [], 1) == 1
assert median_two_sorted([1, 3], [2]) == 2.0
assert median_two_sorted([1, 2], [3, 4]) == 2.5

### 7g. Edge cases - invalid input must raise

Three invalid calls: `k` larger than the combined length, an unsorted array, and two empty arrays for the median. All three raise `ValueError` rather than returning a plausible number, because a silently wrong order statistic is much harder to debug than an exception at the boundary.

In [ ]:
for bad_call in (
    lambda: kth_smallest_two_sorted([1], [2], 3),
    lambda: kth_smallest_two_sorted([2, 1], [3], 2),
    lambda: median_two_sorted([], []),
):
    try:
        bad_call()
    except ValueError:
        pass
    else:
        raise AssertionError("invalid input must raise ValueError")

### 7h. Stated output check

Print the two documented lookups and the merged reference array. The assert re-checks the k-th value for the documented `k = 4`, which should be 4.

In [ ]:
print("partition_for_key(2026-03-15) ->", partition_for_key(partitions, "2026-03-15"))
print("first_at_or_after(2026-03-15) ->", first_at_or_after(partitions, "2026-03-15"))
print("kth_smallest_two_sorted(k=4)  ->", kth_smallest_two_sorted([1, 3, 8], [2, 4, 5, 6, 7], 4))
assert kth_smallest_two_sorted([1, 3, 8], [2, 4, 5, 6, 7], 4) == 4
print("P7 checks passed")

**Complexity & edge cases.** `bisect` gives O(log n) lookups in C; the two-array selection is O(log(min(n, m))) time and O(1) space. Edge cases: empty partition lists; duplicate partition starts (`bisect_left` -> first, `bisect_right` -> last); lexicographic order only equals chronological order for zero-padded ISO dates; `k` out of range; one empty array; even total length needs the average of two order statistics; unsorted input must be rejected, because the partition invariant silently breaks.

**Interview follow-ups.**

1. *Why binary-search the shorter array?* It bounds the search range at min(n, m) and keeps the partition indices valid, giving the tighter O(log(min(n,m))) bound.
2. *How do you binary-search data in a data lake?* Store per-file min/max statistics (Parquet footers, Iceberg/Delta manifests) and binary-search the sorted manifest instead of scanning every file.

## Problem 8 - Generator-based streaming of a large file

**Statement.** Process a file far larger than RAM on one machine. Read it lazily, transform in bounded batches, and never hold the whole file. Generators keep memory flat.

**Example.**

```text
file   : events.csv.gz with lines "user_id,amount" (millions of rows)
target : total amount per user, O(distinct users) memory
search : first N matching lines, short-circuiting the read
```

**Constraints.** O(1) memory for the reader, independent of file size; handle plain and gzip files; skip blank lines; dead-letter malformed rows instead of aborting; no third-party libraries.

### 8a. A lazy line reader

A generator function holds one line at a time and suspends between yields, so the open file handle is the only state. Iterating a file object already streams with a bounded buffer -- the generator just layers CRLF stripping and transparent gzip opening on top.

The `with` block closes the handle when the generator is exhausted *or* garbage-collected early. Never call `read()`/`readlines()` here; that is the single decision that makes the difference between working and OOM on a file bigger than RAM.

In [ ]:
import gzip
from collections import defaultdict
from collections.abc import Iterator
from pathlib import Path

try:  # Python 3.12+
    from itertools import batched
except ImportError:  # Python 3.11 and older, including some Colab runtimes
    from itertools import islice

    def batched(iterable, n):
        "Batch data into tuples of length n; the last batch may be shorter."
        it = iter(iterable)
        while chunk := tuple(islice(it, n)):
            yield chunk


def read_lines(path: Path) -> Iterator[str]:
    """Lazily yield stripped lines, transparently handling .gz files."""
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as handle:
        for line in handle:
            yield line.rstrip("\r\n")

### 8b. One streaming pass to per-user totals

Consume the lazily-read lines in fixed batches and accumulate into a dict keyed by user. Memory is O(batch size + number of distinct users), not O(file size) -- this is the standard single-pass aggregation that a map-reduce reducer performs per partition.

Malformed rows go to the dead-letter path: skip and continue rather than aborting the whole job. Note the order of operations: cast first, *then* add, because `totals[user_id] += value` on a `defaultdict` would insert `0.0` before the cast raises and leave a ghost user behind.

In [ ]:
def totals_by_user(path: Path) -> dict[str, float]:
    """One streaming pass: O(batch + distinct users) memory, not O(file)."""
    totals: dict[str, float] = defaultdict(float)
    for batch in batched(read_lines(path), 1_000):
        for line in batch:
            if not line:
                continue
            user_id, _, amount = line.partition(",")
            try:
                value = float(amount)
            except ValueError:
                continue  # dead-letter malformed rows instead of aborting
            # Add only after the cast succeeds: `totals[user_id] += ...` would
            # create a 0.0 entry via defaultdict before the cast raises.
            totals[user_id] += value
    return dict(totals)

### 8c. Short-circuiting search

`first_n_matching` stops reading as soon as it has `n` hits, which is the whole benefit of a generator: an early `break` closes the underlying file instead of scanning the remaining gigabytes. This is how a "find the first bad record in today's dump" tool stays fast.

The alternative -- filtering a materialised list -- pays for the full scan before returning anything.

In [ ]:
def first_n_matching(path: Path, needle: str, n: int) -> list[str]:
    """Short-circuiting search over a huge file: stop as soon as n matches exist."""
    matches: list[str] = []
    for line in read_lines(path):
        if needle in line:
            matches.append(line)
            if len(matches) == n:
                break
    return matches

### 8d. Happy path - plain and gzip files

Write a small fixture with a quoted-free CSV, a blank line, and one malformed row, then read it back. The malformed row is skipped, the blank line is skipped, and `u1` accumulates 10 + 2.5. The same function reads the `.gz` fixture without a code change because `read_lines` picks the opener from the suffix.

In [ ]:
import gzip
import itertools
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)
    raw = tmp_path / "events.csv"
    raw.write_text(
        "u1,10\n\n" + "u2,5\n" + "u1,2.5\n" + "bad,not-a-number\n" + "u3,7",
        encoding="utf-8",
    )
    print("totals:", totals_by_user(raw))
    assert totals_by_user(raw) == {"u1": 12.5, "u2": 5.0, "u3": 7.0}
    assert first_n_matching(raw, "u1", 1) == ["u1,10"]
    assert first_n_matching(raw, "missing", 5) == []

    gz = tmp_path / "events.csv.gz"
    with gzip.open(gz, "wt", encoding="utf-8") as handle:
        handle.write("a,1\nb,2\n")
    assert totals_by_user(gz) == {"a": 1.0, "b": 2.0}

### 8e. Edge cases - laziness and bounded state

A generator's body does not run until the first `next()`, so an infinite source is harmless as long as the consumer stops early. The `explode` probe proves it: the only way to observe the `AssertionError` is to advance the generator, and `close()` disposes of it without finishing.

This is also the test that a reader is truly lazy: if `read_lines` eagerly read the file, the infinite example could not be constructed at all.

In [ ]:
def naturals():
    value = 0
    while True:
        yield value
        value += 1


# Generators are lazy: an infinite source is fine as long as you stop early.
assert list(itertools.islice(naturals(), 3)) == [0, 1, 2]
gen = naturals()
assert next(gen) == 0 and next(gen) == 1  # state advances only on demand


def explode():
    yield 1
    raise AssertionError("generator body must not run eagerly")


probe = explode()
assert next(probe) == 1  # nothing raises until the generator is advanced
probe.close()

### 8f. Stated output check

Re-create a tiny fixture here and print the per-user totals: `u1` should accumulate `10 + 2.5 = 12.5` while the malformed `bad` row is dead-lettered. The generator-size assertion then shows that a source of a billion rows still holds constant state.

In [ ]:
import sys
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    sample = Path(tmp) / "events.csv"
    sample.write_text("u1,10\nu1,2.5\nbad,not-a-number\n", encoding="utf-8")
    print("totals ->", totals_by_user(sample))
    assert totals_by_user(sample) == {"u1": 12.5}

large_generator = (line for line in range(10**9))
assert sys.getsizeof(large_generator) < sys.getsizeof([0] * 10_000)
next(large_generator)               # state advances only on demand
large_generator.close()
print("P8 checks passed")

**Complexity & edge cases.** O(n) time, O(batch size + retained state) memory; the reader holds one line at a time. Edge cases: a final line without a newline; CRLF endings; malformed records should be dead-lettered with a reason rather than killing the job; sniff gzip magic bytes instead of trusting the suffix in production; parallel workers need byte ranges that resync to the next newline; closing a generator early still runs the `with` block via `GeneratorExit`.

**Interview follow-ups.**

1. *How do you parallelise a single huge file?* Give each worker a byte range, seek, discard the partial first line, and stop at the last newline; gzip blocks can be decompressed in parallel with `zlib`-level offsets.
2. *Generator vs `itertools.batched` list chunks - what changes?* Batching gives amortised per-item overhead and lets you hand whole chunks to a vectorised or bulk-insert path, at the cost of holding one batch in memory.

## Problem 9 - Parse CSV, JSON, and NDJSON

**Statement.** Read the common landing formats without pandas: header CSV (with quoted fields and a BOM), a JSON array, NDJSON (one object per line), and a memory-flat streaming JSON array via `JSONDecoder.raw_decode`.

**Example.**

```text
orders.csv   : order_id,amount,note  /  1,10.5,"a, b"
orders.json  : [{"order_id": 1, "amount": 10.5}]
orders.ndjson: {"order_id": 1}\n{"order_id": 2}
```

**Constraints.** O(1) memory for CSV/NDJSON; report bad NDJSON with a line number; reject a non-array JSON document in `parse_json_array`; strip a UTF-8 BOM; no pandas.

### 9a. Header CSV with `csv.DictReader`

`csv.DictReader` maps each row onto the header names and understands quoting, so `"a, b"` stays one field. Wrapping the text in `io.StringIO` keeps the function testable on a string while the same code works on an open file object.

The BOM is stripped explicitly: Excel and many exporters prepend `\ufeff`, which otherwise glues itself to the first header name and breaks every lookup on that column. In production you would open with `encoding="utf-8-sig"` rather than slicing.

In [ ]:
import csv
import io
import json
from collections.abc import Iterator
from typing import Any


def parse_csv(text: str) -> list[dict[str, str]]:
    """Header-based CSV; csv handles quoted fields that contain commas/newlines."""
    return list(csv.DictReader(io.StringIO(text.lstrip("\ufeff"))))

### 9b. A JSON array

`json.loads` returns a Python object, but the caller expects a list of records, so a top-level object or scalar is a contract violation and must raise. `parse_json_array` therefore materialises the whole document -- acceptable for a small config payload, not for a multi-GB export.

Deeply nested documents are where `json` recursion limits and memory both bite; that is the cue to reach for streaming.

In [ ]:
def parse_json_array(text: str) -> list[dict[str, Any]]:
    """A JSON array must be materialised unless you stream it with raw_decode."""
    payload = json.loads(text)
    if not isinstance(payload, list):
        raise ValueError("expected a JSON array")
    return payload

### 9c. NDJSON with line numbers

One JSON document per line is the natural landing format for streams: each line is independent, so a writer can append and a reader can parallelise or resume. Blank lines are skipped, and a malformed line is re-raised as a `ValueError` carrying its 1-based line number.

The line number is the whole point -- "bad NDJSON somewhere in 4 GB" is not actionable, "bad NDJSON at line 91,233" is.

In [ ]:
def parse_ndjson(text: str) -> list[dict[str, Any]]:
    """One JSON object per line; blank lines skipped, bad lines reported with a line number."""
    records: list[dict[str, Any]] = []
    for number, line in enumerate(text.splitlines(), start=1):
        if not line.strip():
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as exc:
            raise ValueError(f"bad NDJSON at line {number}") from exc
    return records

### 9d. Streaming a JSON array with `raw_decode`

`JSONDecoder.raw_decode(text, index)` parses exactly one value and returns the index where it stopped, so you can walk a large buffer one element at a time instead of building the whole list. The loop skips the structural characters that sit between values (`[`, `]`, `,`, whitespace) and yields each decoded value.

Real code feeds the decoder from a chunked file reader and keeps a tail buffer; the point of the exercise is that the parser hands you a resumable position, which is what makes streaming possible at all.

In [ ]:
def iter_json_array(text: str) -> Iterator[Any]:
    """Stream a JSON array value-by-value with raw_decode, so it need not fit in memory."""
    decoder = json.JSONDecoder()
    position = 0
    length = len(text)
    while position < length:
        while position < length and text[position] in " \t\r\n,[]":
            position += 1
        if position >= length:
            break
        value, position = decoder.raw_decode(text, position)
        yield value

### 9e. Happy path - CSV and JSON

The CSV fixture has a quoted field containing a comma (`"a, b"`) and an empty trailing field, both of which `csv` handles correctly. The BOM case is a separate assertion because it is invisible in the output until a column lookup fails. The JSON array parses back to the documented records.

In [ ]:
csv_text = 'order_id,amount,note\n1,10.5,"a, b"\n2,3.0,\n'
print("csv:", parse_csv(csv_text))
assert parse_csv(csv_text) == [
    {"order_id": "1", "amount": "10.5", "note": "a, b"},
    {"order_id": "2", "amount": "3.0", "note": ""},
]
assert parse_csv("\ufeffid,v\n1,2\n") == [{"id": "1", "v": "2"}]  # BOM stripped

json_text = '[{"order_id": 1, "amount": 10.5}, {"order_id": 2, "amount": 3.0}]'
assert parse_json_array(json_text) == [
    {"order_id": 1, "amount": 10.5},
    {"order_id": 2, "amount": 3.0},
]

### 9f. Happy path - NDJSON and the streaming decoder

NDJSON skips blank lines and returns the objects in order. The streaming decoder walks a deliberately ugly buffer (`"  [ {...}, {...}, 3 ] "`) to prove it skips commas, brackets, and whitespace, not just newlines; `"[]"` yields nothing.

In [ ]:
ndjson_text = '{"order_id": 1}\n\n{"order_id": 2}\n'
assert parse_ndjson(ndjson_text) == [{"order_id": 1}, {"order_id": 2}]
assert parse_ndjson("") == []

streamed = list(iter_json_array('  [ {"a": 1}, {"b": [1, 2]}, 3 ] '))
print("streamed JSON:", streamed)
assert streamed == [{"a": 1}, {"b": [1, 2]}, 3]
assert list(iter_json_array("[]")) == []

### 9g. Edge cases - non-array JSON and bad NDJSON

A top-level JSON object is a schema-contract failure, and a malformed NDJSON line must name its line. Both raise `ValueError`, which the caller can route to a dead-letter path with the raw payload attached.

Empty NDJSON is valid and returns `[]` -- an empty file is not an error, it is a zero-row batch.

In [ ]:
try:
    parse_json_array('{"order_id": 1}')
except ValueError:
    pass
else:
    raise AssertionError("non-array JSON must raise")

try:
    parse_ndjson('{"ok": 1}\nnot json\n')
except ValueError as exc:
    assert "line 2" in str(exc)
else:
    raise AssertionError("bad NDJSON must raise")

### 9h. Stated output check

Print the parsed CSV rows and the streamed JSON values. The CSV output shows the quoted comma preserved as a single field and the missing note as an empty string, which is exactly how a downstream loader should see it.

In [ ]:
print("csv      ->", parse_csv(csv_text))
print("ndjson   ->", parse_ndjson(ndjson_text))
print("streamed ->", list(iter_json_array('  [ {"a": 1}, {"b": [1, 2]}, 3 ] ')))
print("P9 checks passed")

**Complexity & edge cases.** CSV and NDJSON are O(n) time / O(1) memory; a JSON array is O(n) memory because the document must exist. `raw_decode` makes the array streamable but requires you to skip separators yourself. Edge cases: a UTF-8 BOM (`utf-8-sig`); embedded newlines inside quoted CSV fields; very long NDJSON lines; empty strings that break `float`; a JSON file that is an object rather than an array; deeply nested JSON needs `ijson` for true streaming.

**Interview follow-ups.**

1. *Why is NDJSON the preferred landing format for streams?* Each line is an independent, appendable, partially readable document, so a crash mid-file loses at most one line and readers can parallelise by line ranges.
2. *How do you make parsing robust to schema drift?* Parse into `dict`, coerce types with an explicit per-column cast function, and route cast failures to a dead-letter path with the raw line and error.

## Problem 10 - Flatten nested JSON into rows

**Statement.** Turn nested documents into tabular rows: (a) `flatten_record` collapses nested objects into dotted paths and indexes arrays; (b) `explode_rows` emits one output row per array element with the parent fields repeated - the in-memory equivalent of `UNNEST` / `LATERAL VIEW explode`.

**Example.**

```text
input : {"order_id":1,"customer":{"country":"us"},
         "items":[{"sku":"a","qty":2},{"sku":"b","qty":1}]}
flat  : {"order_id":1,"customer.country":"us",
         "items[0].sku":"a","items[0].qty":2,"items[1].sku":"b","items[1].qty":1}
rows  : [{"order_id":1,"customer":{...},"sku":"a","qty":2},
         {"order_id":1,"customer":{...},"sku":"b","qty":1}]
```

**Constraints.** Preserve key order; do not mutate the input; an empty array emits one parent row (left-join semantics); a missing array field behaves like an empty array.

### 10a. Recursive flatten with dotted paths

Walk the document depth-first and build a path string as you descend; dict keys extend the path with `sep`, list elements extend it with `[index]`, and scalars are written to the output. Insertion order is preserved because `out` is filled in traversal order, which is what lets a column-selecting reader stay stable.

A literal key that already contains the separator collides with a nested path -- production code escapes it. Deeply nested input can also hit the recursion limit, at which point an explicit stack is the safer rewrite.

In [ ]:
from collections.abc import Iterable
from typing import Any


def flatten_record(record: dict[str, Any], prefix: str = "", sep: str = ".") -> dict[str, Any]:
    """Collapse nested objects into dotted paths; arrays are indexed."""
    out: dict[str, Any] = {}
    for key, value in record.items():
        path = f"{prefix}{sep}{key}" if prefix else key
        if isinstance(value, dict):
            out.update(flatten_record(value, path, sep))
        elif isinstance(value, list):
            for index, element in enumerate(value):
                if isinstance(element, dict):
                    out.update(flatten_record(element, f"{path}[{index}]", sep))
                else:
                    out[f"{path}[{index}]"] = element
        else:
            out[path] = value
    return out

### 10b. Explode - one row per array element

For each parent record, copy every field except the array, then emit one row per element; when the element is a dict its keys are merged into the row, so `{"sku": ..., "qty": ...}` becomes columns. The parent fields are *repeated* on every generated row, which is exactly the row-multiplication that `UNNEST` produces.

An empty or missing array emits a single parent row, matching a left join. That choice is what keeps the parent count conserved; dropping the parent would silently lose orders with no items.

In [ ]:
def explode_rows(records: Iterable[dict[str, Any]], array_field: str) -> list[dict[str, Any]]:
    """One output row per array element, repeating the parent fields (UNNEST)."""
    rows: list[dict[str, Any]] = []
    for record in records:
        values = record.get(array_field) or []
        base = {key: value for key, value in record.items() if key != array_field}
        if not values:
            rows.append(dict(base))  # left-join semantics: keep one row
            continue
        for value in values:
            row = dict(base)
            if isinstance(value, dict):
                row.update(value)
            else:
                row[array_field] = value
            rows.append(row)
    return rows

### 10c. Happy path - flatten

The documented order flattens to nine scalar columns: the top-level `order_id`, two dotted `customer.*` paths, four indexed `items[*].*` paths, and two indexed `tags[*]` values. The exact dict comparison checks both the values and the traversal order.

In [ ]:
order = {
    "order_id": 1,
    "customer": {"country": "us", "city": "austin"},
    "items": [{"sku": "a", "qty": 2}, {"sku": "b", "qty": 1}],
    "tags": ["gift", "rush"],
}
flat = flatten_record(order)
print("flat:", flat)
assert flat == {
    "order_id": 1,
    "customer.country": "us",
    "customer.city": "austin",
    "items[0].sku": "a",
    "items[0].qty": 2,
    "items[1].sku": "b",
    "items[1].qty": 1,
    "tags[0]": "gift",
    "tags[1]": "rush",
}

### 10d. Happy path - explode

The two-item order produces two rows, each carrying `order_id`, the nested `customer`, and the full `tags` list, plus the item's own `sku` and `qty`. The input document is asserted unchanged afterwards, because the function builds new dicts rather than editing in place.

In [ ]:
rows = explode_rows([order], "items")
print("exploded rows:", len(rows))
assert rows == [
    {
        "order_id": 1,
        "customer": {"country": "us", "city": "austin"},
        "tags": ["gift", "rush"],
        "sku": "a",
        "qty": 2,
    },
    {
        "order_id": 1,
        "customer": {"country": "us", "city": "austin"},
        "tags": ["gift", "rush"],
        "sku": "b",
        "qty": 1,
    },
]
assert order["items"] == [{"sku": "a", "qty": 2}, {"sku": "b", "qty": 1}]  # not mutated

### 10e. Edge cases - empty arrays, missing arrays, scalars, empty input

An empty array and a missing array both emit the parent row minus the array field, which is the left-join contract. An array of scalars keeps the original field name and repeats it once per value. `flatten_record({})` is `{}`, and `explode_rows([], ...)` is `[]`.

In [ ]:
assert flatten_record({}) == {}
assert flatten_record({"a": {"b": {"c": 1}}}) == {"a.b.c": 1}
assert flatten_record({"a": 1}, sep="/") == {"a": 1}
assert explode_rows([{"id": 1, "items": []}], "items") == [{"id": 1}]
assert explode_rows([{"id": 1}], "items") == [{"id": 1}]
assert explode_rows([{"id": 1, "items": ["x", "y"]}], "items") == [
    {"id": 1, "items": "x"},
    {"id": 1, "items": "y"},
]
assert explode_rows([], "items") == []

### 10f. Stated output check

Print the flattened keys in order and the exploded rows so the saved output matches the `Example` block. The number of rows equals the total number of item elements across all parents, which is the count a downstream aggregate must expect.

In [ ]:
print("flat keys ->", list(flat))
print("rows      ->", [(r["sku"], r["qty"]) for r in rows])
print("P10 checks passed")

**Complexity & edge cases.** Both functions are O(total nodes) time; `flatten_record` uses O(depth) recursion and `explode_rows` materialises one row per array element. Edge cases: a missing parent key; an empty array should still emit one parent row to match a left join; arrays of arrays and null array elements; a literal key containing the separator collides with a nested path, so production code should escape it; deeply nested documents can exceed the recursion limit, so an explicit stack is safer.

**Interview follow-ups.**

1. *Flatten or explode - how do you choose?* Flatten when you need a wide, one-row-per-document table; explode when downstream aggregation needs one row per child event, accepting the row multiplication.
2. *How does row multiplication affect downstream counts?* A naive `COUNT(*)` after exploding counts child records, not orders - count distinct parent ids or pre-aggregate before the explode.

## Problem 11 - In-memory hash join of two datasets

**Statement.** Join a large fact stream to a small dimension without a database. Build a hash map on the smaller side and stream the larger side through it - the same shape as a broadcast join. Support inner and left joins, duplicate build keys, and SQL null semantics.

**Example.**

```text
facts      : [("c1","o1"), ("c2","o2"), ("c9","o3")]
dimensions : [("c1","Alice"), ("c2","Bob")]
inner      : [("c1","o1","Alice"), ("c2","o2","Bob")]
left       : [..., ("c9","o3",None)]
```

**Constraints.** O(n + m) time, O(m) memory for the build side; always build on the smaller side; duplicate build keys multiply rows; NULL keys never match; reject unknown join types.

### 11a. Materialised inner/left hash join

Index the dimension rows by key first (a `defaultdict(list)` so duplicate keys are kept), then probe with each fact row. Hits are emitted once per matching dimension row; misses are emitted with a `None` payload only for a left join.

Two SQL semantics are encoded deliberately: a `None` build key is skipped, and a `None` probe key never matches, because `NULL = NULL` is unknown in SQL. Getting either wrong changes the row count silently.

In [ ]:
from collections import defaultdict
from collections.abc import Iterable, Iterator
from typing import Any, TypeVar

K = TypeVar("K")
V = TypeVar("V")


def hash_join(
    facts: Iterable[tuple[K, V]],
    dimensions: Iterable[tuple[K, Any]],
    how: str = "inner",
    null: Any = None,
) -> list[tuple[K, V, Any]]:
    """Broadcast-style join: build a hash index on the right (small) side."""
    if how not in {"inner", "left"}:
        raise ValueError("how must be 'inner' or 'left'")
    index: dict[K, list[Any]] = defaultdict(list)
    for key, value in dimensions:
        if key is not None:  # SQL semantics: NULL keys never match
            index[key].append(value)
    out: list[tuple[K, V, Any]] = []
    for key, fact_value in facts:
        matches = index.get(key) if key is not None else None
        if matches:
            for dim_value in matches:
                out.append((key, fact_value, dim_value))
        elif how == "left":
            out.append((key, fact_value, null))
    return out

### 11b. The same join as a generator

Yielding instead of appending keeps the fact side un-materialised: memory is the build index only, which is the actual broadcast-join property. The output is consumed once, so the caller can write it straight to a sink or feed another stage.

If the consumer needs to iterate twice, it must call the function again or wrap the result in a list -- the classic generator trap.

In [ ]:
def hash_join_streaming(
    facts: Iterable[tuple[K, V]],
    dimensions: Iterable[tuple[K, Any]],
    how: str = "inner",
) -> Iterator[tuple[K, V, Any]]:
    """Same join as a generator, so the fact side is never materialised."""
    index: dict[K, list[Any]] = defaultdict(list)
    for key, value in dimensions:
        if key is not None:
            index[key].append(value)
    for key, fact_value in facts:
        matches = index.get(key) if key is not None else None
        if matches:
            for dim_value in matches:
                yield key, fact_value, dim_value
        elif how == "left":
            yield key, fact_value, None

### 11c. Happy path - inner and left

The documented join produces two inner rows and, for the left variant, the unmatched fact `("c9", "o3")` with a `None` payload. The streaming generator must produce the identical inner sequence.

In [ ]:
facts = [("c1", "o1"), ("c2", "o2"), ("c9", "o3")]
dims = [("c1", "Alice"), ("c2", "Bob")]
print("inner:", hash_join(facts, dims))
assert hash_join(facts, dims) == [("c1", "o1", "Alice"), ("c2", "o2", "Bob")]
assert hash_join(facts, dims, how="left") == [
    ("c1", "o1", "Alice"),
    ("c2", "o2", "Bob"),
    ("c9", "o3", None),
]
assert list(hash_join_streaming(facts, dims)) == [("c1", "o1", "Alice"), ("c2", "o2", "Bob")]

### 11d. Happy path - duplicate build keys multiply rows

One fact row matching two dimension rows produces two output rows, exactly like a SQL join on a non-unique key. This is correct semantics but a common source of exploded output counts, so it deserves an explicit test rather than a comment.

In [ ]:
assert hash_join([("c1", "o1")], [("c1", "A"), ("c1", "B")]) == [
    ("c1", "o1", "A"),
    ("c1", "o1", "B"),
]

### 11e. Edge cases - NULL keys, empty sides, unknown join type

A `None` key never matches, so the inner join is empty; the left join still emits the fact row with a null payload so no fact is lost. Empty facts give no rows, empty dimensions make every inner join empty, and an unsupported join type is rejected up front rather than silently behaving like an inner join.

In [ ]:
assert hash_join([(None, "o1")], [(None, "A")]) == []
assert hash_join([(None, "o1")], [(None, "A")], how="left") == [(None, "o1", None)]
assert hash_join([], dims) == []
assert hash_join(facts, []) == []

try:
    hash_join([], [], how="outer")
except ValueError:
    pass
else:
    raise AssertionError("unsupported join type must raise")

### 11f. Stated output check

Print the inner and left results for the documented inputs. Compare the left result with the example: the third tuple must be `('c9', 'o3', None)`, which is the anti-join row a left join preserves.

In [ ]:
print("inner ->", hash_join(facts, dims))
print("left  ->", hash_join(facts, dims, how="left"))
print("P11 checks passed")

**Complexity & edge cases.** O(n + m) time and O(m) memory for the m-row build side; building on the large side silently materialises the fact table. Duplicate build keys multiply rows, which is correct SQL semantics but can explode output. Edge cases: NULL keys never match in SQL (unlike `==` in Python); `1` vs `"1"` key types silently drop rows, so normalise on ingest; a single skewed key defeats the in-memory approach and needs a sort-merge or distributed shuffle.

**Interview follow-ups.**

1. *Why does Spark broadcast the small side?* It ships one copy of the small table to every executor, turning a shuffle join into a map-side join with no network shuffle of the fact data.
2. *When is a sort-merge join better than a hash join?* When neither side fits in memory, when both inputs are already sorted, or when the data is skewed and a range partition balances better than a hash.

## Problem 12 - Schema drift and late-arriving dedupe

**Statement.** Two batch-ingestion safety checks. (a) Infer the schema per batch and diff it against a baseline, classifying columns as added, removed, or retyped. (b) Deduplicate replays and out-of-order backfills by keeping the record with the greatest `(updated_at, tiebreak)` per business key.

**Example.**

```text
baseline: {"id":"int", "amount":"float"}
current : {"id":"int", "amount":"str", "coupon":"str"}
drift   : {"added":["coupon"], "removed":[], "retyped":["amount"]}

input : [("u1",t1), ("u2",t1), ("u1",t3), ("u1",t2)]
output: u1 -> the t3 record, u2 -> the t1 record
```

**Constraints.** O(rows x columns) to infer, O(columns) to diff; O(n) time / O(k) memory for dedupe where k is the distinct key count; the tiebreak makes replays deterministic; do not mutate inputs.

### 12a. Infer the observed schema

For every row record the Python type name per column, ignoring `None` (a null carries no type information). The result is `column -> set of observed type names`, so a column that mixes `int` and `float` is visible rather than silently coerced.

This is *observed* schema, not declared schema: it can only find drift in columns that actually appear in the sample. Diffing Parquet/Avro metadata is O(columns) and catches changes in columns with no rows.

In [ ]:
from collections.abc import Iterable, Mapping
from typing import Any


def infer_schema(rows: Iterable[Mapping[str, Any]]) -> dict[str, set[str]]:
    """Column -> set of observed Python type names (None contributes nothing)."""
    schema: dict[str, set[str]] = {}
    for row in rows:
        for column, value in row.items():
            if value is not None:
                schema.setdefault(column, set()).add(type(value).__name__)
    return schema

### 12b. Collapse a type set into a label

A column with no non-null value has no type and is labelled `null`; a single type keeps its name; anything else is `mixed(a|b)` in sorted order. Sorting makes the label deterministic, which matters because the drift diff compares labels as strings.

Reporting `mixed(int|float)` is usually benign widening, while a type that becomes `str` is the one that breaks readers.

In [ ]:
def describe_type(types: set[str]) -> str:
    """Collapse a set of observed types into one readable label."""
    if not types:
        return "null"
    if len(types) == 1:
        return next(iter(types))
    return "mixed(" + "|".join(sorted(types)) + ")"

### 12c. Diff baseline against current

Set algebra on the column names gives added (`now - base`) and removed (`base - now`); the intersection is checked for a changed type label and reported as retyped. All three lists are sorted so the output is stable and diffable in CI.

The diff is O(columns) once the schemas exist, which is why comparing declared catalog metadata is far cheaper than rescanning every row.

In [ ]:
def detect_schema_drift(
    baseline: Mapping[str, set[str]], current: Mapping[str, set[str]]
) -> dict[str, Any]:
    """Classify every column as added, removed, or retyped."""
    base, now = set(baseline), set(current)
    return {
        "added": sorted(now - base),
        "removed": sorted(base - now),
        "retyped": sorted(
            column
            for column in base & now
            if describe_type(baseline[column]) != describe_type(current[column])
        ),
    }

### 12d. Keep the latest record per key

One pass with a dict keyed by the business key. A record replaces the stored winner only when its `(updated_at, tiebreak)` rank is strictly greater, so the first record wins ties -- which is what makes replays deterministic.

The tiebreak (`offset`, version, or ingestion id) is not optional: if two replays carry the same timestamp, a map with no secondary key picks an arbitrary winner and the sink stops converging. Tombstones and deletes must also participate here.

In [ ]:
def dedupe_latest(
    records: Iterable[dict[str, Any]],
    key: str = "id",
    ts: str = "updated_at",
    tiebreak: str = "offset",
) -> list[dict[str, Any]]:
    """Keep the record with the greatest (timestamp, tiebreak) per key."""
    winners: dict[Any, dict[str, Any]] = {}
    for record in records:
        current = winners.get(record[key])
        rank = (record[ts], record.get(tiebreak, 0))
        if current is None or rank > (current[ts], current.get(tiebreak, 0)):
            winners[record[key]] = record
    return list(winners.values())

### 12e. Happy path - drift

The documented baseline declares `amount` as a float and `note` as optional; the current batch retypes `amount` to a string and adds `coupon`. The diff must report exactly `{"added": ["coupon"], "removed": [], "retyped": ["amount"]}` -- a string-to-number change is the breaking kind, so it must not be silently widened.

In [ ]:
baseline = [{"id": 1, "amount": 10.0, "note": "x"}, {"id": 2, "amount": 3.0, "note": None}]
current = [{"id": 3, "amount": "10", "note": "y", "coupon": "SAVE"}]
drift = detect_schema_drift(infer_schema(baseline), infer_schema(current))
print("drift:", drift)
assert drift == {"added": ["coupon"], "removed": [], "retyped": ["amount"]}
assert detect_schema_drift({}, {}) == {"added": [], "removed": [], "retyped": []}
assert detect_schema_drift({"a": {"int"}}, {"a": {"int"}, "b": {"str"}})["added"] == ["b"]

### 12f. Happy path - type labels

The label helper is the vocabulary of the drift report, so it gets its own checks: an empty type set is `null`, one type keeps its name, and a mixture is sorted and joined. This is also where an all-null column is distinguishable from a missing column.

In [ ]:
assert describe_type(set()) == "null"
assert describe_type({"int"}) == "int"
assert describe_type({"float", "int"}) == "mixed(float|int)"

### 12g. Happy path - dedupe latest wins

Five records for two keys: `u1` has an old row, a stale row with a large offset but an older timestamp, a newest row at `t = 3`, and a `t = 3` tiebreak loser. The winner is the `t = 3` record with offset 9, because the timestamp dominates and the offset only breaks ties.

In [ ]:
records = [
    {"id": "u1", "updated_at": 1, "offset": 1, "v": "old"},
    {"id": "u2", "updated_at": 1, "offset": 1, "v": "only"},
    {"id": "u1", "updated_at": 3, "offset": 9, "v": "new"},
    {"id": "u1", "updated_at": 3, "offset": 4, "v": "older-tie"},
    {"id": "u1", "updated_at": 2, "offset": 50, "v": "stale"},
]
result = dedupe_latest(records)
by_id = {row["id"]: row for row in result}
print("winners:", {k: v["v"] for k, v in by_id.items()})
assert by_id["u1"]["v"] == "new"      # greatest timestamp wins
assert by_id["u2"]["v"] == "only"     # single record retained
assert len(result) == 2

### 12h. Edge cases - empty input and no mutation

`dedupe_latest([])` returns `[]`, and choosing a winner must not edit the losing or winning records in place -- the function stores references, so the caller's objects are shared but never rewritten. That property is what makes it safe to run over a cached batch twice.

In [ ]:
assert dedupe_latest([]) == []
assert records[0]["v"] == "old"       # input not mutated

### 12i. Stated output check

Print the drift report and the winner for each key. The drift dict must match the example verbatim, and the winners map should read `{"u1": "new", "u2": "only"}`.

In [ ]:
print("drift   ->", drift)
print("winners ->", {k: v["v"] for k, v in by_id.items()})
print("P12 checks passed")

**Complexity & edge cases.** Inference is O(rows x columns); the diff is O(columns), which is why diffing declared Parquet/Avro metadata beats sampling data. Dedupe is O(n) time / O(k) memory; a warehouse `MERGE` or sort-merge spills instead. Edge cases: all-null columns have no type; int-to-float widening is usually benign while string-to-number is breaking; column-order changes are invisible to a set diff but break positional CSV readers; equal timestamps need a version or Kafka-offset tiebreak; naive and aware datetimes cannot be compared, so normalise to UTC; tombstone/delete rows must win and be applied by the writer.

**Interview follow-ups.**

1. *Where should drift detection run - in the producer or the consumer?* Both: producers validate against a contract, consumers compare the landed batch against the catalog because producers lie or get bypassed.
2. *Why is the tiebreak important for idempotency?* Without it, two replays of the same timestamp can pick different winners, so the sink is non-deterministic and reruns do not converge.

## Problem 13 - Sessionize clickstream events

**Statement.** Group ordered user events into sessions, starting a new session when the idle gap exceeds a threshold - the exercise behind every product-analytics model. Handle out-of-order input and per-user interleaving.

**Example.**

```text
events: u1 at [0, 300, 2400, 2500], u2 at [0], gap = 1800 seconds
output: [u1 0-300 (2 events), u1 2400-2500 (2 events), u2 0-0 (1 event)]
```

**Constraints.** Sort by `(user, timestamp)` first; O(n log n) time, O(sessions) output; a gap of exactly the threshold stays in the session; reject a negative gap; a single event forms a one-event session.

### 13a. Sort, then extend the open session

Sorting by `(user, ts, event_id)` makes the problem sequential: each event either extends the last open session for the same user when `when - last_end <= gap`, or opens a new one. The event id is the final tiebreak so equal timestamps produce a deterministic order.

The boundary is inclusive on purpose -- a gap of exactly `gap_seconds` is still the same session, which is how most product-analytics definitions are written. State the choice explicitly, because `<` versus `<=` changes session counts at the edge.

In [ ]:
from collections.abc import Iterable
from typing import Any


def sessionize(
    events: Iterable[dict[str, Any]],
    gap_seconds: int,
    user_key: str = "user",
    ts_key: str = "ts",
    id_key: str = "event_id",
) -> list[dict[str, Any]]:
    """Group events into sessions; a new session starts when the idle gap exceeds
    `gap_seconds`. The boundary is inclusive: a gap of exactly `gap_seconds`
    stays in the same session.
    """
    if gap_seconds < 0:
        raise ValueError("gap_seconds must be non-negative")
    ordered = sorted(events, key=lambda event: (event[user_key], event[ts_key], event[id_key]))
    sessions: list[dict[str, Any]] = []
    for event in ordered:
        user, when = event[user_key], event[ts_key]
        if (
            sessions
            and sessions[-1]["user"] == user
            and when - sessions[-1]["end"] <= gap_seconds
        ):
            session = sessions[-1]
            session["end"] = when
            session["count"] += 1
            session["duration"] = session["end"] - session["start"]
            session["event_ids"].append(event[id_key])
        else:
            sessions.append(
                {
                    "user": user,
                    "start": when,
                    "end": when,
                    "count": 1,
                    "duration": 0,
                    "event_ids": [event[id_key]],
                }
            )
    return sessions

### 13b. Happy path

For `gap = 1800`, the documented events produce three sessions: `u1` at 0-300 (two events), `u1` at 2400-2500 (two events, after the 2100-second idle gap), and `u2` at 0-0 (one event). Each session carries its start, end, count, duration, and the ordered event ids.

In [ ]:
events = [
    {"user": "u1", "ts": 0, "event_id": "e1"},
    {"user": "u1", "ts": 300, "event_id": "e2"},
    {"user": "u1", "ts": 2400, "event_id": "e3"},  # 2100s gap -> new session
    {"user": "u1", "ts": 2500, "event_id": "e4"},
    {"user": "u2", "ts": 0, "event_id": "e5"},
]
sessions = sessionize(events, gap_seconds=1800)
print("sessions:", [(s["user"], s["start"], s["end"], s["count"]) for s in sessions])
assert [(s["user"], s["start"], s["end"], s["count"]) for s in sessions] == [
    ("u1", 0, 300, 2),
    ("u1", 2400, 2500, 2),
    ("u2", 0, 0, 1),
]
assert sessions[0]["event_ids"] == ["e1", "e2"]
assert sessions[0]["duration"] == 300
assert sessions[2]["duration"] == 0

### 13c. Edge cases - inclusive boundary, out-of-order input, empty, negative gap

Exactly the gap keeps the events in one session, and a shuffled input must yield byte-identical output because the function sorts internally before doing anything else. An empty input is a valid empty result, while a negative gap is a caller bug.

Out-of-order input is handled by sorting, but note the limitation: a truly unbounded stream needs a lateness buffer, because events cannot be re-sorted once a session has been emitted.

In [ ]:
same = sessionize(
    [{"user": "u", "ts": 0, "event_id": "a"}, {"user": "u", "ts": 1800, "event_id": "b"}], 1800
)
assert len(same) == 1
assert same[0]["count"] == 2

shuffled = [events[3], events[0], events[1], events[2], events[4]]
assert sessionize(shuffled, 1800) == sessions
assert sessionize([], 1800) == []

try:
    sessionize([], -1)
except ValueError:
    pass
else:
    raise AssertionError("negative gap must raise")

### 13d. Stated output check

Print the compact `(user, start, end, count)` view and compare it with the example. The saved output is the fastest way for a reviewer to confirm the session boundaries without reading the assert.

In [ ]:
print("sessions ->", [(s["user"], s["start"], s["end"], s["count"]) for s in sessions])
print("ids      ->", [s["event_ids"] for s in sessions])
print("P13 checks passed")

**Complexity & edge cases.** O(n log n) if the input must be sorted, O(n) if the source already delivers ordered partitions per user; memory is O(sessions) for the output (or O(1) state per active user when streaming). Edge cases: out-of-order events need a lateness buffer before emitting; one event forms a one-event session; define `<=` vs `<` at the boundary (here exactly the gap is inactive); users with no events are absent, so downstream joins must be left joins; clock skew can make a user's timestamps non-monotonic.

**Interview follow-ups.**

1. *How would you sessionize an unbounded stream?* Keep per-user last-event state in a keyed store (Redis/Flink state) and emit a session when a timer for `last_event + gap` fires, handling late events with allowed lateness.
2. *How does this map to a Spark window function?* `LAG(ts)` over `PARTITION BY user ORDER BY ts`, then mark a new session when `ts - lag > gap`, and finally `SUM(new_session)` as a session id.

## Problem 14 - Production utilities: LRU cache and retry with backoff

**Statement.** Two small classes you will be asked to write (or reason about) in a data-platform interview: a fixed-capacity **LRU cache** with O(1) `get`/`put`, and a **retry** helper with capped exponential backoff plus optional jitter.

**Example.**

```text
capacity 2: put("a",1) put("b",2) get("a") -> 1; put("c",3) evicts "b"
backoff  : attempts=4, base=0.1, factor=2 -> sleeps [0.1, 0.2, 0.4]
```

**Constraints.** O(1) amortised `get`/`put`; validate capacity; injectable `sleep`/`rand` so the retry logic is testable and deterministic; cap both the delay and the attempt count; re-raise the final error.

### 14a. `LRUCache` on top of `OrderedDict`

`OrderedDict` is a hash map plus a doubly linked list, which is exactly the two data structures an LRU cache needs. A hit calls `move_to_end` to mark the key most-recently-used; a write past capacity evicts with `popitem(last=False)`, the least-recently-used end. Both operations are O(1).

The hit/miss counters make the cache observable in tests. The documented pitfall: `get` returns `None` for a miss, so `None` cannot also be a stored value without a sentinel.

In [ ]:
import random
import time
from collections import OrderedDict
from collections.abc import Callable, Hashable
from typing import Generic, TypeVar

K = TypeVar("K", bound=Hashable)
V = TypeVar("V")


class LRUCache(Generic[K, V]):
    """O(1) get/put: an OrderedDict is a hash map plus a doubly linked list."""

    def __init__(self, capacity: int) -> None:
        if capacity <= 0:
            raise ValueError("capacity must be positive")
        self.capacity = capacity
        self._data: OrderedDict[K, V] = OrderedDict()
        self.hits = 0
        self.misses = 0

    def get(self, key: K) -> V | None:
        if key not in self._data:
            self.misses += 1
            return None
        self.hits += 1
        self._data.move_to_end(key)
        return self._data[key]

    def put(self, key: K, value: V) -> None:
        if key in self._data:
            self._data.move_to_end(key)
        self._data[key] = value
        if len(self._data) > self.capacity:
            self._data.popitem(last=False)

    def keys_oldest_first(self) -> list[K]:
        return list(self._data)

### 14b. One delay: `min(cap, base * factor**attempt)`

`compute_delay` is the pure core of the backoff policy: the attempt number goes in, the sleep before the next try comes out, capped by `max_delay` so a long outage does not create a 30-minute sleep. The random source is injected so a test can pin jitter to a known value.

Jitter interpolates between the deterministic delay (`jitter = 0`) and a random one (`jitter = 1`, "full jitter"), which is what desynchronises clients that failed together.

In [ ]:
def compute_delay(
    attempt: int,
    base: float,
    factor: float,
    max_delay: float,
    jitter: float,
    rand: Callable[[], float],
) -> float:
    """Delay before the retry after a zero-indexed failed `attempt`."""
    delay = min(max_delay, base * factor**attempt)
    if jitter:
        # Full jitter (jitter=1) spreads retries uniformly; partial jitter dampens.
        delay = delay * (1 - jitter) + delay * jitter * rand()
    return delay

### 14c. The whole schedule

`backoff_delays(attempts)` returns the sleep before each retry for a plan of `attempts` total tries, so there are `attempts - 1` sleeps: a plan of 4 tries sleeps 3 times. Expressing the schedule as a list makes it trivial to assert and to log as a plan before running it.

`attempts = 1` means "try once, never retry" and therefore an empty schedule.

In [ ]:
def backoff_delays(
    attempts: int,
    base: float = 0.1,
    factor: float = 2.0,
    max_delay: float = 10.0,
    jitter: float = 0.0,
    rand: Callable[[], float] = random.random,
) -> list[float]:
    """The sleep before each retry for a plan of `attempts` total tries."""
    return [compute_delay(n, base, factor, max_delay, jitter, rand) for n in range(attempts - 1)]

### 14d. The retry loop

Run the callable, catch only the declared exception types, and sleep before the next try; on the last attempt re-raise the original error instead of a wrapper, so the caller sees the real failure. `sleep` and `rand` are injected, which turns a timing-dependent routine into a deterministic one under test.

Retry only transient, idempotent operations: retrying a non-idempotent write double-writes. `attempts < 1` is rejected rather than treated as "no tries".

In [ ]:
def retry(
    func: Callable[[], V],
    attempts: int = 4,
    base: float = 0.1,
    factor: float = 2.0,
    max_delay: float = 10.0,
    jitter: float = 0.0,
    retry_on: tuple[type[BaseException], ...] = (Exception,),
    sleep: Callable[[float], None] = time.sleep,
    rand: Callable[[], float] = random.random,
) -> V:
    """Run `func`, retrying transient failures with capped exponential backoff."""
    if attempts < 1:
        raise ValueError("attempts must be >= 1")
    for attempt in range(attempts):
        try:
            return func()
        except retry_on:
            if attempt == attempts - 1:
                raise
            sleep(compute_delay(attempt, base, factor, max_delay, jitter, rand))
    raise RuntimeError("unreachable")

### 14e. Happy path - cache

With capacity 2: put `a`, put `b`, get `a` (which makes `a` the most recent), then put `c` evicts `b`. Updating an existing key keeps it freshest, and the counters report exactly the hits and misses the sequence implies.

In [ ]:
cache: LRUCache[str, int] = LRUCache(2)
cache.put("a", 1)
cache.put("b", 2)
assert cache.get("a") == 1          # "a" is now most recently used
cache.put("c", 3)                   # evicts "b"
assert cache.get("b") is None
assert cache.get("c") == 3
assert cache.keys_oldest_first() == ["a", "c"]
cache.put("c", 30)                  # update keeps it freshest
assert cache.get("c") == 30
cache.put("d", 4)                   # evicts "a"
assert cache.get("a") is None
print("cache hits/misses:", cache.hits, cache.misses)
assert cache.hits == 3 and cache.misses == 2

### 14f. Happy path - backoff schedule and a flaky callable

The documented plan `attempts=4, base=0.1, factor=2` sleeps `[0.1, 0.2, 0.4]`, and a cap of 5.0 flattens a base-3 schedule to `[1.0, 3.0, 5.0, 5.0]`. A function that fails twice then succeeds must be called exactly three times and must sleep exactly `[0.1, 0.2]`.

In [ ]:
calls = {"n": 0}
slept: list[float] = []


def flaky() -> str:
    calls["n"] += 1
    if calls["n"] < 3:
        raise TimeoutError("temporary")
    return "ok"


print("delays:", backoff_delays(4, base=0.1, factor=2.0, max_delay=10.0))
assert backoff_delays(4, base=0.1, factor=2.0, max_delay=10.0) == [0.1, 0.2, 0.4]
assert backoff_delays(5, base=1.0, factor=3.0, max_delay=5.0) == [1.0, 3.0, 5.0, 5.0]
assert backoff_delays(1) == []
assert retry(flaky, attempts=5, base=0.1, factor=2.0, sleep=slept.append) == "ok"
assert calls["n"] == 3
print("slept between retries:", slept)
assert slept == [0.1, 0.2]          # two failures -> two sleeps

### 14g. Edge cases - invalid capacity, exhausted retries, pinned jitter

Capacity zero or negative must be rejected, not silently no-op. A callable that always fails must surface the original exception after the final attempt rather than looping forever. Jitter is checked with a pinned random draw: `rand = 0` gives the deterministic lower bound, `rand = 1` the full delay.

In [ ]:
try:
    LRUCache(0)
except ValueError:
    pass
else:
    raise AssertionError("capacity 0 must raise")


def always_fails() -> None:
    raise TimeoutError("down")


try:
    retry(always_fails, attempts=2, sleep=slept.append)
except TimeoutError:
    pass
else:
    raise AssertionError("exhausted retries must raise")

assert compute_delay(0, base=1.0, factor=2.0, max_delay=10.0, jitter=0.5, rand=lambda: 0.0) == 0.5
assert compute_delay(0, base=1.0, factor=2.0, max_delay=10.0, jitter=0.5, rand=lambda: 1.0) == 1.0
assert retry(lambda: "ok", attempts=1) == "ok"

### 14h. Stated output check

Print the cache counters, the documented delay schedule, and the observed sleeps. `hits = 3, misses = 2` is exactly the access pattern above: hits on `a`, `c`, and the second `c`; misses on `b` and the evicted `a`.

In [ ]:
print("hits/misses ->", cache.hits, cache.misses)
print("schedule    ->", backoff_delays(4, base=0.1, factor=2.0, max_delay=10.0))
print("slept       ->", slept)
print("P14 checks passed")

**Complexity & edge cases.** `OrderedDict` gives O(1) amortised `get`/`put` because it is a hash map plus a doubly linked list; `move_to_end` makes a hit the most recent and `popitem(last=False)` evicts the least recent. Edge cases: capacity zero/negative must be rejected, not silently no-op; `None` as a legal cached value is ambiguous with a miss, so use a sentinel; a hand-rolled version must update both the map and the list on every hit and eviction; retries must target idempotent, transient errors only; cap attempts and `max_delay` so a scheduler slot is not held forever; jitter is mandatory to avoid retry storms.

**Interview follow-ups.**

1. *Why does `functools.lru_cache` not solve every caching need?* It keys on hashable positional/keyword arguments, cannot expire by time, cannot be invalidated selectively, and is per-process rather than shared.
2. *What is full jitter and why does it work?* Sleep `uniform(0, min(cap, base * 2**n))`; randomising waits desynchronises clients that failed together, flattening the thundering herd when the service recovers.

## Interview questions

Twelve questions that come up when interviewers probe Python for data engineering, each with a concise model answer. Where a claim is checkable, a code cell in this section proves it.

### Q1. When do you reach for a generator instead of a list?

**Model answer.** A list materialises every result: O(n) memory and O(n) time up front. A generator produces one item per `next()` and holds O(1) state, so it can represent an unbounded or larger-than-RAM source. Use generators for file/stream ingestion and one-pass transform pipelines; use lists when you need random access, `len()`, reuse, or multiple passes.

### Q2. Why is `dict`/`set` lookup considered O(1), and what breaks it?

**Model answer.** They are hash tables: `hash(key)` picks a bucket, then equality confirms the match. Average-case lookup/insert/delete is O(1); iteration is O(n). Worst case is O(n) per operation under heavy collisions (a bad `__hash__` or adversarial keys). Keys must be hashable with value-based equality; a mutable key would break bucket invariants. Dicts preserve insertion order since Python 3.7, which is what makes an ordered dedupe a one-liner.

### Q3. `sorted()`, `list.sort()`, or `heapq.nlargest()` - which and why?

**Model answer.** `list.sort()` sorts in place, O(n log n) time, O(1) extra space (Timsort may use O(n/2) worst case), returns `None`. `sorted()` copies first, adding O(n) memory. `heapq.nlargest(k, it)` is O(n log k) time and O(k) memory, so it wins when k << n and preserves input order for ties. For top-N per group, keep one bounded heap per group.

### Verification A - generators, heap selection, ordered dicts

Three checkable claims from Q1-Q3. The heap comparison proves `nlargest` agrees with a full sort while bounding memory; the generator size comparison proves a generator's state does not grow with its potential output; and the dict checks show first-seen key order with last-write-wins values.

In [ ]:
import heapq
import sys

# nlargest equals a full-sort slice, but keeps memory bounded by k.
data = [5, 1, 9, 3, 7, 2]
assert heapq.nlargest(3, data) == sorted(data, reverse=True)[:3]
assert heapq.nsmallest(2, data) == sorted(data)[:2]


# A generator holds constant state regardless of how many items it could produce.
def count_up(limit):
    value = 0
    while value < limit:
        yield value
        value += 1


large_generator = count_up(10**9)
assert sys.getsizeof(large_generator) < sys.getsizeof([0] * 10_000)
assert [next(large_generator) for _ in range(3)] == [0, 1, 2]
large_generator.close()


# dict preserves insertion order, which is why ordered dedupe is easy.
pairs = [("a", 1), ("b", 2), ("a", 3)]
assert list(dict(pairs)) == ["a", "b"]     # first-seen order
assert dict(pairs)["a"] == 3               # but last write wins the value
assert list(dict.fromkeys(["x", "y", "x"])) == ["x", "y"]
print("Interview verification A passed")

### Q4. How do you deduplicate records while preserving order, and how does it scale?

**Model answer.** Stream the data and keep a `set` of seen keys, appending the first occurrence of each: O(n) time, O(k) memory for k distinct keys. At scale, partition by key so each partition dedupes independently; or sort by key and drop adjacent duplicates (spills to disk, O(n log n)); or use a Bloom filter as a cheap pre-filter, accepting false positives. In a warehouse: `ROW_NUMBER() OVER (PARTITION BY key ORDER BY ts DESC) = 1`.

### Q5. How do you process a file larger than RAM with only the standard library?

**Model answer.** Never call `read()`/`readlines()`. Iterate the file object line by line (bounded buffer), wrap it in a generator, process in fixed batches (`itertools.batched`), and keep only aggregates or the required state. Use `gzip.open(..., "rt")` for compressed files. Parallelise by byte ranges only when the format allows seeking to a record boundary; otherwise split by file. Write aggregates out incrementally.

### Q6. `json.load` vs `json.loads` vs streaming with `raw_decode`.

**Model answer.** `json.loads(str)` parses a string; `json.load(file)` reads a file object and parses it, but both materialise the whole document. For a JSON array too large for memory, drive `json.JSONDecoder().raw_decode(buffer, index)` to pull one value at a time, or use NDJSON where every line is an independent document. Third-party options are `ijson` and `jsonlines`.

### Q7. How does an in-memory hash join work, and when does it fail?

**Model answer.** Build a hash map on the smaller (dimension) side, then stream the larger (fact) side through it: O(n + m) time, O(m) memory for the build side. It fails when the build side does not fit in memory, when one join key is badly skewed, or when null/type semantics differ (`1` vs `"1"`, NULL never equals NULL). Remedies: broadcast the small side, sort-merge join, or hash-partition by key and join partition-wise.

### Q8. How do you handle late-arriving records and duplicates?

**Model answer.** Make the sink idempotent. Keep a map keyed by the business key and retain the record with the greatest `(event_timestamp, tiebreak)`, where the tiebreak is a Kafka offset or version so replays are deterministic. If producer clocks are unreliable, prefer the ingestion timestamp for lateness. Tombstones/deletes must also win, and the downstream write must merge/upsert rather than blindly append.

### Q9. What is schema drift and how do you detect it?

**Model answer.** Schema drift is any change in columns or types between batches. Infer the observed schema (column -> set of types) per batch, compare against the catalog or the previous batch, and classify added, removed, and retyped columns. Silent type changes such as int -> string are the dangerous ones. Diff declared Parquet/Avro metadata in O(columns); added required or removed fields break readers, while int -> float is usually benign widening.

### Verification B - recency order, hash-join fan-out, sorted invariants

Three checkable claims from Q7-Q9. `OrderedDict.move_to_end` plus `popitem(last=False)` is the LRU primitive; a hash join with duplicate build keys multiplies rows exactly like SQL; and unsorted input breaks the binary-search partition invariant, which is why the check is explicit in Problem 7.

In [ ]:
from collections import OrderedDict, defaultdict

# OrderedDict.move_to_end is the primitive behind LRU recency ordering.
od = OrderedDict()
for key in "abc":
    od[key] = key.upper()
od.move_to_end("a")
assert list(od) == ["b", "c", "a"]         # "a" touched most recently
od.popitem(last=False)
assert list(od) == ["c", "a"]              # oldest evicted

# A hash join with duplicate build keys multiplies rows, exactly like SQL.
build = defaultdict(list)
for key, value in [("k", 1), ("k", 2), ("x", 9)]:
    build[key].append(value)
probe = [("k", "A"), ("x", "B"), ("missing", "C")]
joined = [(key, payload, dim) for key, payload in probe for dim in build.get(key, [])]
assert joined == [("k", "A", 1), ("k", "A", 2), ("x", "B", 9)]


# Rejecting unsorted input is what keeps a binary-search partition valid.
def kth_partition_is_valid(sorted_values):
    return all(sorted_values[i] <= sorted_values[i + 1] for i in range(len(sorted_values) - 1))


assert kth_partition_is_valid([1, 2, 2, 5])
assert not kth_partition_is_valid([2, 1])
print("Interview verification B passed")

### Q10. How do you compute percentiles and rolling windows efficiently?

**Model answer.** An exact percentile needs the sorted sample: sort once (O(n log n)) and each percentile is O(1). For rolling windows, means/sums are O(1) per element with a deque, but exact quantiles are not. Use an order-statistic tree (O(log n) insert) or an approximate sketch (t-digest, HDR histogram, Greenwald-Khanna) for large windows. Percentiles are not additive across buckets, and on a small sample p99 is just the maximum.

### Q11. How do you implement an LRU cache, and what does `functools.lru_cache` give you?

**Model answer.** Combine a hash map (O(1) lookup) with a doubly linked list or `OrderedDict` (recency order): `get` moves the key to the most-recent end, `put` inserts and evicts the least-recent end when over capacity. Both are O(1). `functools.lru_cache` is the C implementation for pure functions keyed on hashable arguments; it cannot key unhashable objects, has no time-based expiry, and caches per process. A sentinel is needed when `None` is a legal value.

### Q12. Why retry with exponential backoff and jitter?

**Model answer.** Delay before retry n is `min(cap, base * factor**n)`, plus jitter. Backoff gives a failing dependency time to recover; jitter (`uniform(0, delay)` for full jitter) desynchronises clients so they do not stampede the service the moment it returns. Retry only transient, idempotent operations (timeouts, 429, 503), cap attempts and max delay so no scheduler slot is held forever, and re-raise the final error after recording it.

### Verification C - memoization, streaming JSON, backoff arithmetic

Three checkable claims from Q10-Q12. `functools.lru_cache` exposes its hit/miss counters and evicts by capacity; `raw_decode` returns the index where one document stopped, which is the primitive behind streaming JSON; and the backoff delay is a pure function of the attempt number, so it can be asserted exactly.

In [ ]:
import functools
import json

# functools.lru_cache memoizes by hashable args; cache_info exposes the counters.
calls = {"n": 0}


@functools.lru_cache(maxsize=2)
def slow_square(value):
    calls["n"] += 1
    return value * value


assert [slow_square(v) for v in (1, 2, 1, 3, 1)] == [1, 4, 1, 9, 1]
assert calls["n"] == 3                    # only 1, 2, 3 were computed
info = slow_square.cache_info()
assert (info.hits, info.misses) == (2, 3)

# raw_decode parses one value at a time out of a buffer: the streaming JSON core.
decoder = json.JSONDecoder()
text = '{"a": 1} {"b": 2}'
first, index = decoder.raw_decode(text)
second, _ = decoder.raw_decode(text, index + 1)
assert (first, second) == ({"a": 1}, {"b": 2})

# Exponential backoff is a pure function of the attempt number.
delays = [min(0.5 * 2**n, 4.0) for n in range(5)]
assert delays == [0.5, 1.0, 2.0, 4.0, 4.0]
print("Interview verification C passed")

### Closing checklist

Before you call a coding answer finished in an interview, walk this list out loud:

- **State the preconditions.** Sorted input, non-null keys, hashable types, ascending partitions.
- **State the complexity** for time and memory, including the build side of a join or the retained state of a stream.
- **Name the edge cases** you handle and the ones you deliberately reject with an error.
- **Say what changes at scale**, and which single-machine assumption breaks first.

## Python exercises

Eight short exercises cover the mechanics that the problem set does not: defaults, closures, identity, copying, containers, errors, dataclasses, and string building. Write your answer, then run the cell to check it.


### Try it 1. Mutable default arguments


**Task.** Write `add_tag(tag, tags=None)` that returns a list with `tag` appended. Show that a mutable default of `[]` is shared across calls, and that `None` gives each call a fresh list.

**Hint.** Default arguments are evaluated once, when the `def` statement runs, so the same list object is reused on every call. Use `None` as the sentinel and build the list inside the body.

**Expected.** The buggy version returns `['a']` then `['a', 'b']`; the safe version returns `['a']` then `['b']`.


In [ ]:
def buggy(tag, tags=[]):
    tags.append(tag)
    return tags


def safe(tag, tags=None):
    if tags is None:
        tags = []
    tags.append(tag)
    return tags


assert buggy("a") == ["a"]
assert buggy("b") == ["a", "b"]   # the default list is shared
assert safe("a") == ["a"]
assert safe("b") == ["b"]         # a fresh list on each call
print("buggy:", buggy("c"), "| safe:", safe("c"))


### Try it 2. Closure late binding


**Task.** Build a list of lambdas in a loop that each return their own loop value, not the final value. Compare a late-binding version with a default-argument fix and with `functools.partial`.

**Hint.** A closure captures the variable, not its value at definition time, so all the lambdas see the loop variable after the loop ends. Bind the value with a default argument or `partial`.

**Expected.** The naive lambdas all return `2`; the fixed ones return `0, 1, 2`.


In [ ]:
from functools import partial

late = [lambda: i for i in range(3)]
assert [f() for f in late] == [2, 2, 2]

bound = [lambda i=i: i for i in range(3)]
assert [f() for f in bound] == [0, 1, 2]

partials = [partial(lambda i: i, i) for i in range(3)]
assert [f() for f in partials] == [0, 1, 2]
print("late:", [f() for f in late], "| bound:", [f() for f in bound])


### Try it 3. `is` versus `==`


**Task.** Show the difference between identity and equality for a list alias, two equal dicts, and `None`. Use `is` only for singletons.

**Hint.** `==` asks whether two objects compare equal; `is` asks whether they are the same object in memory. CPython also caches small integers, so `256 is 256` is true by implementation detail.

**Expected.** The alias is identical, the equal dicts are not, and `None` is only safely tested with `is`.


In [ ]:
row = {"id": 1}
alias = row
assert alias is row                 # same object
assert {"id": 1} == row             # equal value
assert {"id": 1} is not row         # a new object

value = None
assert value is None                # identity is the right test for None
assert value == None                # works, but style tools flag it

small = 256
cached = 256
assert small is cached              # CPython caches -5..256
print("alias is row:", alias is row, "| equal dicts:", {"id": 1} == row)


### Try it 4. Shallow versus deep copy


**Task.** Copy a nested list with `copy.copy` and `copy.deepcopy`. Mutate an inner list through the shallow copy and show that the original changes; show that the deep copy stays independent.

**Hint.** `copy.copy` builds a new outer container but reuses the inner objects. `copy.deepcopy` walks the whole object graph and copies every level.

**Expected.** After `shallow[0].append(99)`, the original first row contains `99`; the deep copy does not.


In [ ]:
import copy

matrix = [[1, 2], [3, 4]]
shallow = copy.copy(matrix)
deep = copy.deepcopy(matrix)

shallow[0].append(99)
assert matrix[0] == [1, 2, 99]      # the inner list is shared
assert deep[0] == [1, 2]            # the deep copy is independent

deep[1].append(7)
assert matrix[1] == [3, 4]
print("matrix:", matrix, "| deep:", deep)


### Try it 5. Dict ordering and `collections`


**Task.** Count words into a plain `dict` and confirm insertion order, then use `Counter`, `defaultdict`, and `OrderedDict.move_to_end`.

**Hint.** Since Python 3.7 `dict` keeps insertion order as a language guarantee. `Counter` counts, `defaultdict` creates missing values, and `OrderedDict` has order-aware methods.

**Expected.** Keys come back in first-seen order; `Counter.most_common(2)` returns the two most frequent words; `move_to_end` reorders the `OrderedDict`.


In [ ]:
from collections import Counter, OrderedDict, defaultdict

words = "b a b c a b".split()
counts = {}
for word in words:
    counts[word] = counts.get(word, 0) + 1
assert list(counts) == ["b", "a", "c"]        # insertion order is preserved
assert dict(sorted(counts.items())) == {"a": 2, "b": 3, "c": 1}

counter = Counter(words)
assert counter.most_common(2) == [("b", 3), ("a", 2)]

groups = defaultdict(list)
for key, value in [("us", 1), ("eu", 2), ("us", 3)]:
    groups[key].append(value)
assert dict(groups) == {"us": [1, 3], "eu": [2]}

ordered = OrderedDict([("first", 1), ("second", 2)])
ordered.move_to_end("first")
assert list(ordered) == ["second", "first"]
print("order:", list(counts), "| top:", counter.most_common(2))


### Try it 6. Exception chaining


**Task.** Catch a low-level `FileNotFoundError` and raise a `ConfigError` with `raise ... from`. Inspect `__cause__`. Then raise `from None` and confirm the chain is suppressed.

**Hint.** `raise New from original` sets `__cause__` and keeps the traceback honest. `raise ... from None` sets `__suppress_context__` and hides the original context.

**Expected.** The first error reports a `FileNotFoundError` cause; the second has `__cause__ is None`.


In [ ]:
class ConfigError(Exception):
    pass


def load(path):
    try:
        raise FileNotFoundError(path)
    except FileNotFoundError as exc:
        raise ConfigError(f"cannot load {path}") from exc


try:
    load("settings.yaml")
except ConfigError as exc:
    assert isinstance(exc.__cause__, FileNotFoundError)
    assert "settings.yaml" in str(exc)
    print("chained:", type(exc).__name__, "<-", type(exc.__cause__).__name__)

try:
    try:
        raise ValueError("inner")
    except ValueError:
        raise RuntimeError("clean") from None
except RuntimeError as exc:
    assert exc.__cause__ is None and exc.__suppress_context__ is True
print("exception chaining checks passed")


### Try it 7. Dataclass versus a hand-written `__init__`


**Task.** Compare a class with a hand-written `__init__` and `__eq__` against an equivalent `@dataclass`. Cover equality, `repr`, `default_factory`, and `frozen=True`.

**Hint.** `@dataclass` generates `__init__`, `__repr__`, and `__eq__` from the annotations. `field(default_factory=list)` gives each instance its own list, and `frozen=True` blocks attribute assignment.

**Expected.** The dataclass prints its fields, equal instances compare equal, each instance owns a separate list, and assignment to a frozen instance raises `FrozenInstanceError`.


In [ ]:
from dataclasses import FrozenInstanceError, dataclass, field


class Manual:
    def __init__(self, order_id, amount):
        self.order_id = order_id
        self.amount = amount

    def __eq__(self, other):
        return (self.order_id, self.amount) == (other.order_id, other.amount)


@dataclass
class Order:
    order_id: str
    amount: float
    tags: list = field(default_factory=list)


assert Manual("o1", 10.0) == Manual("o1", 10.0)
assert "Manual object" in repr(Manual("o1", 10.0))    # no auto repr

order_a = Order("o1", 10.0)
order_b = Order("o1", 10.0)
assert order_a == order_b
assert repr(order_a) == "Order(order_id='o1', amount=10.0, tags=[])"
order_a.tags.append("priority")
assert order_b.tags == []                             # one list per instance


@dataclass(frozen=True)
class Point:
    x: int
    y: int


try:
    Point(1, 2).x = 5
except FrozenInstanceError as exc:
    print("frozen:", type(exc).__name__)
print("dataclass repr:", order_a)


### Try it 8. String joining versus concatenation


**Task.** Build one long string with a `+=` loop and with `"".join(...)`. Assert that both give the same text, and print the timing of each as a guide.

**Hint.** Strings are immutable, so `+=` builds a new string and copies everything so far, which is quadratic in the number of parts. `join` walks the parts once and allocates the result once.

**Expected.** The two results are equal, and the printed timings show `join` doing the same work without repeated copies.


In [ ]:
import time


def concat(parts):
    result = ""
    for part in parts:
        result += part
    return result


def join(parts):
    return "".join(parts)


words = [f"row-{i}," for i in range(20_000)]
assert concat(words[:100]) == join(words[:100])

started = time.perf_counter()
joined = join(words)
join_ms = (time.perf_counter() - started) * 1000

started = time.perf_counter()
built = concat(words)
concat_ms = (time.perf_counter() - started) * 1000

assert joined == built
print(f"join {join_ms:.2f} ms | concat {concat_ms:.2f} ms")
print("join builds one buffer; += copies the accumulated string each time")
